# NB3 — Scenario Evaluation

> **Goal:** Evaluate 23 routing scenarios (parameter sweeps + structural separation)
> using the calibrated simulator. Reproduces Table 1 from the paper.

> **Paper:** *Diagnosing ML-Driven Queue Systems in Production* — ICDM 2026

> **Note:** Data loading cells use stubs. Replace with your own loader matching the schema below.


## Table of Contents

### Part I — Setup & Data Loading
1. [Imports & Environment Setup](#imports)
2. [Load Operational Results](#load-data)
3. [Filter QPlanner Calls](#filter-qp)
4. [Load Brains Scores](#brains-scores)
5. [Data Preparation — Simulation Inputs](#data-preparation)

### Part II — Simulation Engine
6. [Simulator Core Classes](#simulator-core)
7. [Objective Function (FOConfig & compute_score)](#fo-scoring)
8. [Assignment Solver (Hungarian)](#assignment-solver)
9. [Two-Tier Solver](#two-tier-solver)
10. [Main Simulation Loop](#simulation-loop)
11. [MLflow Re-scoring & Full Score Coverage](#full-score-coverage)

### Part III — Calibration & Analysis
12. [Calibrated Parameters & Day Setup](#calibrated-parameters)
13. [Two-Tier Feasibility — Slate Segmentation](#two-tier-feasibility)
14. [Valuable Operators for Two-Tier](#valuable-operators)

### Part IV — Scenario Framework
15. [Scenario Synthesis — Experimental Roadmap](#scenario-synthesis)
16. [Experiment Runner](#experiment-runner)

### Part V — Layer Execution & Results
17. [Layer 0 — Baselines (S0, S0')](#layer-0)
18. [Diagnostic: S0 vs S0' G-value Mechanism](#s0-diagnostic)
19. [Layer 1 — Imminent Inflation Test](#layer-1)
20. [Layer 1 — Findings & Decision](#layer-1-findings)
21. [Layer 2 — FO Parameter Sweep](#layer-2)
22. [Layer 3 — Two-Tier Structural Variants (S3a–S3c)](#layer-3)
23. [Scenario Comparison Table](#comparison-table)

### Part VI — Robustness Checks
24. [Multi-Seed Statistical Inference](#multi-seed)
25. [Bias Sensitivity Bootstrap](#bias-sensitivity)
26. [Calibration Curve & Brier Decomposition](#calibration-curve)
27. [Temporal Holdout Validation](#temporal-holdout)

### Part VII — Appendix
28. [Solver Pair Coverage Diagnostic](#pair-coverage-diagnostic)


---
# Part I — Setup & Data Loading


<a id="imports"></a>
## Imports & Environment Setup

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & ENVIRONMENT SETUP
# ============================================================
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import time
from scipy.optimize import linear_sum_assignment  # Hungarian algorithm


<a id="load-data"></a>
## Load Operational Results from BigQuery

## Data Loading — Replace with your own loader

The original code loads production telemetry from BigQuery via an internal library.
Replace the cell below with your own data loader.

### Expected schema for `df_ops` (operational results)

| Column | Type | Description |
|--------|------|-------------|
| `call_id` | str | Unique call identifier |
| `call_start_ts` | datetime | Call arrival timestamp |
| `operator_id` | str | Assigned operator identifier |
| `wait_time_s` | float | Wait time in seconds at assignment |
| `outcome` | str | `RETAINED` or `CHURNED` |
| `qp_result` | str | `QPlanner-AgentAvailable`, `QPlanner-NoAgentReturned`, `QPlanner-AgentNotAvailable` |
| `assigned_ts` | datetime | Assignment timestamp (NaT if unserved) |

### Expected schema for `df_scores` (call-operator pairings / brains scores)

| Column | Type | Description |
|--------|------|-------------|
| `call_id` | str | Call identifier |
| `operator_id` | str | Operator identifier |
| `churn_ij` | float | Predicted churn probability P(churn | call_i, operator_j) |
| `churn_i` | float | Baseline churn probability P(churn | call_i, null operator) |
| `tick_ts` | datetime | Solver tick timestamp |


In [ ]:
# ============================================================
# DATA LOADING STUB — replace with your own loader
# ============================================================
# Load from CSV/Parquet files. Columns must match the schema
# described in the markdown cell above.
#
# Example:
#   df_ops    = pd.read_parquet('data/operational_results.parquet')
#   df_scores = pd.read_parquet('data/brains_scores.parquet')
#
# The production loader (removed for confidentiality) used:
#   from crm_qp_lib.data_reader import read_op_results, read_brains_data
#   df_ops    = read_op_results(start_date, end_date)
#   df_scores = read_brains_data(start_date, end_date)

DATA_PATH = 'data/'  # set to your data directory
df_ops    = pd.read_parquet(os.path.join(DATA_PATH, 'operational_results.parquet'))
df_scores = pd.read_parquet(os.path.join(DATA_PATH, 'brains_scores.parquet'))


<a id="filter-qp"></a>
## Filter to QPlanner-Handled Calls

In [ ]:
# =============================================================================
# CELL 3 — FILTER TO QPLANNER-HANDLED CALLS
# =============================================================================
# INPUT:  df_calls (all calls — see data loading output for row count)
# DOES:   - Filters to rows where QUEUE_PLANNER_STATUS_DESC == 'On' (QP was active)
#         - Keeps only QPlanner outcome types: AgentAvailable, NoAgentReturned, AgentNotAvailable
#         - Excludes BAU (non-QPlanner) calls — those were handled by the default telephony system
# OUTPUT: df_calls_qplanner — only calls that went through QPlanner (see filter output for count)
# =============================================================================

df_calls_qplanner = df_calls[
    (df_calls['QUEUE_PLANNER_STATUS_DESC'] == 'On') 
    & (df_calls['QUEUE_PLANNER_RESULT_DESC'].isin([
        'QPlanner-AgentAvailable',
        'QPlanner-NoAgentReturned',
        'QPlanner-AgentNotAvailable'
    ]))
]
print(f"QPlanner calls: {len(df_calls_qplanner):,} ({len(df_calls_qplanner)/len(df_calls)*100:.1f}% of total)")
df_calls_qplanner.shape

<a id="brains-scores"></a>
## Load Brains Scores (Call-Operator Pairings)

In [ ]:
# =============================================================================
# CONSTANTS — Outcome definitions (from Cell 3b in original notebook)
# =============================================================================
VALID_OUTCOMES = ["N RECUPERADO", "RECUPERADO COM ALTERACAO", "RECUPERADO SEM ALTERACAO"]

QP_ACCOUNTABLE_RESULTS = [
    'QPlanner-AgentAvailable',
    'QPlanner-NoAgentReturned',
    'QPlanner-AgentNotAvailable',
    'BAU',
]
print(f"✅ Constants loaded: VALID_OUTCOMES ({len(VALID_OUTCOMES)}), QP_ACCOUNTABLE_RESULTS ({len(QP_ACCOUNTABLE_RESULTS)})")

<a id="data-preparation"></a>
## Data Preparation — Build Simulation Inputs

In [ ]:
# =============================================================================
# CELL 7 — DATA PREPARATION: Build all simulation inputs
# =============================================================================
# INPUT:  df_calls_qplanner, df_brains (see data loading outputs for counts)
#
# DOES (9 steps):
#   1. Parse timestamps & sort calls chronologically
#   2. Extract operator pool (unique operators + group)
#   3. Build call events table (one row per unique CALL_ID with arrival, churn_i, duration)
#   4. Build score_lookup dict: (call_id, operator) → churn_ij from FULL brains data
#      Also builds churn_i_lookup: call_id → churn_i
#   5. Compute simulation time window (start → end)
#   6. Define get_day_operators() helper — extracts day-specific operator pool + shifts
#   7. Compute real QPlanner tick cadence from brains request timestamps
#      (time between consecutive request_ids → DATA_TICK_MEDIAN derived from data)
#   8. Compute real pipeline latency from data:
#      overhead = WAIT_TIME_IN_QUEUE (operational) - waiting_time (brains solver)
#      → DATA_PIPELINE_MEAN and DATA_PIPELINE_STD derived from data
#   9. Total wait-time distribution of unserved calls (INFORMATIONAL ONLY)
#      End-to-end wait (QP queue + FIFO) — NOT used as simulation input
#
# OUTPUT:
#   - df_operators: DataFrame with all unique operators (see output for count)
#   - df_call_events: DataFrame with one row per call (see output for count)
#   - score_lookup: dict of (call_id, operator) → churn_ij (~100k+ pairs)
#   - churn_i_lookup: dict of call_id → churn_i
#   - global_median_churn_ij: fallback churn_ij for missing pairs
#   - get_day_operators(): function to extract day-specific ops + shifts
#   - DATA_TICK_MEDIAN, DATA_TICK_MEAN: data-derived tick cadence
#   - DATA_PIPELINE_MEAN, DATA_PIPELINE_STD: data-derived latency params
#   - DATA_MAX_WAIT_SAMPLES: total wait times of unserved calls (informational)
#   - df_calls_qplanner_ts: copy with parsed timestamps (used by later cells)
# =============================================================================

# --- 1. Parse timestamps and sort by arrival time ---
df_sim = df_calls_qplanner.copy()
df_sim['arrival_time'] = pd.to_datetime(df_sim['START_DATE_TIME'])
df_sim = df_sim.sort_values('arrival_time').reset_index(drop=True)

# --- 2. Extract the operator pool (unique operators with their group) ---
operator_cols = ['agent_username', 'agent_groupname']
df_operators = (
    df_sim[df_sim['agent_username'].notna()]
    .drop_duplicates(subset='agent_username')[operator_cols]
    .reset_index(drop=True)
)
print(f"📋 Operator pool: {len(df_operators)} unique operators")

# --- 3. Build call events (one row per unique call arriving at the queue) ---
# Each call needs: call_id, arrival_time, customer_id, churn_i, call_duration
# We pick the first occurrence of each call (some calls may have multiple operator pairings logged)
call_cols = [
    'CALL_ID', 'arrival_time', 'SA_COD', 'churn_i',
    'CALL_DURATION_SEC_QTY', 'WAIT_TIME_IN_QUEUE_SEC_QTY',
    'QUEUE_PLANNER_RESULT_DESC', 'agent_username'
]
df_call_events = (
    df_sim[call_cols]
    .drop_duplicates(subset='CALL_ID', keep='first')
    .reset_index(drop=True)
)
# Fill missing durations with median
median_duration = df_call_events['CALL_DURATION_SEC_QTY'].median()
df_call_events['call_duration'] = df_call_events['CALL_DURATION_SEC_QTY'].fillna(median_duration)
print(f"📞 Total unique calls to simulate: {len(df_call_events)}")
print(f"⏱️  Median call duration: {median_duration:.0f}s")

# --- 4. Build pairing score lookup from BRAINS DATA ---
# df_brains has (call_id, operator) pairings from t_raw_api_brains.
# Each row reflects an operator that was AVAILABLE or IMMINENTLY AVAILABLE
# at a specific historical tick (~8-10 operators per tick).
# Accumulated across all ticks for a call, this gives the "sparse" universe
# of operators the solver actually evaluated for each call.
# We map call_id from brains → CALL_ID used in df_call_events via matching.

# First, build a mapping from brains call_id to df_sim CALL_ID
# (brains uses call_id, df_sim uses CALL_ID — these should match)
brains_call_ids = set(df_brains['call_id'].unique())
sim_call_ids = set(df_call_events['CALL_ID'].unique())
matched_ids = brains_call_ids & sim_call_ids
print(f"🔗 Call ID matching: brains={len(brains_call_ids)}, sim={len(sim_call_ids)}, matched={len(matched_ids)}")
# Build score_lookup from df_brains (ALL operators ever scored across ALL ticks)
# NOTE: This sparse lookup is later replaced by full_score_lookup (Cell 12d)
df_brains_valid = df_brains[
    df_brains['call_id'].isin(matched_ids) & df_brains['churn_ij'].notna()
].copy()
score_lookup = dict(
    zip(
        zip(df_brains_valid['call_id'], df_brains_valid['agent_username']),
        df_brains_valid['churn_ij']
    )
)
churn_i_lookup = dict(
    zip(df_brains_valid['call_id'], df_brains_valid['churn_i'])
)
calls_with_scores = len(set(c for c, o in score_lookup.keys()))
ops_per_call = len(score_lookup) / calls_with_scores if calls_with_scores else 0
print(f"🎯 Pairing scores: {len(score_lookup):,} (call, operator) pairs")
print(f"   Calls with scores: {calls_with_scores:,} / {len(sim_call_ids)}")
print(f"   Avg operators per call: {ops_per_call:.1f}")

# For calls without brains scores (not in brains data), we'll rely on KAPPA gate
global_median_churn_ij = float(np.median(list(score_lookup.values())))
print(f"📊 Global median churn_ij: {global_median_churn_ij:.4f}")

# --- 5. Simulation time window ---
sim_start = df_call_events['arrival_time'].min()
sim_end = df_call_events['arrival_time'].max()
print(f"🕐 Simulation window: {sim_start} → {sim_end}")
print(f"   Duration: {(sim_end - sim_start).total_seconds() / 3600:.1f} hours")

# --- 6. Day-specific operator extraction & shift estimation ---
def get_day_operators(df_sim_data: pd.DataFrame, sim_day: str):
    """
    Extract operators who actually worked on a specific day + their shift windows.
    
    INPUT:  df_sim_data (QPlanner calls with timestamps), sim_day (YYYY-MM-DD string)
    OUTPUT: (df_day_operators, operator_shifts)
      - df_day_operators: DataFrame with agent_username, agent_groupname for that day
      - operator_shifts: dict mapping username → (shift_start, shift_end)
        Shifts estimated from first/last call ± 10min buffer
    """
    day_data = df_sim_data[
        pd.to_datetime(df_sim_data['START_DATE_TIME']).dt.date == pd.to_datetime(sim_day).date()
    ].copy()
    day_data['ts'] = pd.to_datetime(day_data['START_DATE_TIME'])

    day_ops = day_data[day_data['agent_username'].notna()]
    if day_ops.empty:
        return pd.DataFrame(columns=['agent_username', 'agent_groupname']), {}

    # Build shift windows: (first_call - buffer, last_call + avg_duration + buffer)
    shift_buffer = pd.Timedelta(minutes=10)
    median_dur = day_data['CALL_DURATION_SEC_QTY'].median()
    median_dur = median_dur if pd.notna(median_dur) else 1800

    operator_shifts = {}
    for username in day_ops['agent_username'].unique():
        op_calls = day_ops[day_ops['agent_username'] == username]
        shift_start = op_calls['ts'].min() - shift_buffer
        shift_end = op_calls['ts'].max() + pd.Timedelta(seconds=median_dur) + shift_buffer
        operator_shifts[username] = (shift_start, shift_end)

    df_day_operators = (
        day_ops.drop_duplicates(subset='agent_username')[['agent_username', 'agent_groupname']]
        .reset_index(drop=True)
    )
    return df_day_operators, operator_shifts

print(f"\n✅ Helper: get_day_operators() defined")

# --- 7. Compute real QPlanner tick cadence from brains request timestamps ---
# Each unique request_id in df_brains represents one solver invocation.
# The time between consecutive requests IS the real decision cadence.
df_brains_requests = (
    df_brains[['request_id', 'timestamp']]
    .drop_duplicates(subset='request_id')
    .sort_values('timestamp')
    .reset_index(drop=True)
)
df_brains_requests['timestamp'] = pd.to_datetime(df_brains_requests['timestamp'])
df_brains_requests['delta_s'] = df_brains_requests['timestamp'].diff().dt.total_seconds()

# Filter to reasonable cadence (< 5 min = within the same active window, not across gaps)
cadence = df_brains_requests['delta_s'].dropna()
cadence_within_window = cadence[cadence <= 300]

DATA_TICK_MEDIAN = float(cadence_within_window.median())
DATA_TICK_MEAN = float(cadence_within_window.mean())
DATA_TICK_P25 = float(cadence_within_window.quantile(0.25))
DATA_TICK_P75 = float(cadence_within_window.quantile(0.75))

print(f"⏱️  QPlanner request cadence (from {len(cadence_within_window):,} consecutive request pairs):")
print(f"   Median: {DATA_TICK_MEDIAN:.1f}s")
print(f"   Mean:   {DATA_TICK_MEAN:.1f}s")
print(f"   IQR:    [{DATA_TICK_P25:.1f}s, {DATA_TICK_P75:.1f}s]")
print(f"   → Using median ({DATA_TICK_MEDIAN:.0f}s) as tick_interval for simulation")

# --- 8. Compute real operational overhead (pipeline latency) from data ---
# For recommended calls in brains:
#   brains.waiting_time = wait at moment of solver decision (seconds)
#   WAIT_TIME_IN_QUEUE_SEC_QTY = total end-to-end wait in operational results (seconds)
#   overhead = total_wait - solver_wait = real pipeline latency (solver → actual assignment)
df_brains_rec = df_brains[df_brains['recommended'] == True].drop_duplicates(subset='call_id', keep='first')
df_overhead = df_brains_rec[['call_id', 'waiting_time']].rename(
    columns={'waiting_time': 'brains_wait_s'}
).merge(
    df_calls_qplanner[['CALL_ID', 'WAIT_TIME_IN_QUEUE_SEC_QTY']].drop_duplicates(subset='CALL_ID'),
    left_on='call_id', right_on='CALL_ID', how='inner'
)
df_overhead['overhead_s'] = df_overhead['WAIT_TIME_IN_QUEUE_SEC_QTY'] - df_overhead['brains_wait_s']

# Filter to reasonable range (0 to 600s — negative means data timing mismatch)
oh = df_overhead['overhead_s'].dropna()
oh_valid = oh[(oh >= 0) & (oh <= 600)]

DATA_PIPELINE_MEAN = float(oh_valid.mean())
DATA_PIPELINE_STD = float(oh_valid.std())
DATA_PIPELINE_MEDIAN = float(oh_valid.median())
DATA_PIPELINE_P25 = float(oh_valid.quantile(0.25))
DATA_PIPELINE_P75 = float(oh_valid.quantile(0.75))

print(f"\n⚙️  Operational overhead — solver decision → actual assignment (from {len(oh_valid):,} recommended calls):")
print(f"   Median: {DATA_PIPELINE_MEDIAN:.1f}s")
print(f"   Mean:   {DATA_PIPELINE_MEAN:.1f}s")
print(f"   Std:    {DATA_PIPELINE_STD:.1f}s")
print(f"   IQR:    [{DATA_PIPELINE_P25:.1f}s, {DATA_PIPELINE_P75:.1f}s]")
print(f"   Negative overhead count: {(oh < 0).sum()} (data timing mismatches, excluded)")
print(f"   → Using as data-derived pipeline_latency: μ={DATA_PIPELINE_MEAN:.0f}s, σ={DATA_PIPELINE_STD:.0f}s")

# --- 9. Total wait-time distribution of unserved calls (INFORMATIONAL) ---
# QPlanner holds calls for up to 180s (fixed window, agreed with operations).
# After 180s, the telephony system takes over via FIFO routing.
# The wait times below are the TOTAL end-to-end wait (QP queue + FIFO queue),
# NOT the QPlanner-specific timeout. They show how long the telephony system
# kept calls after QPlanner released them.
# NOTE: This distribution is NOT used as a simulation input.
_unserved_mask = df_call_events['QUEUE_PLANNER_RESULT_DESC'].isin([
    'QPlanner-NoAgentReturned', 'QPlanner-AgentNotAvailable'
])
DATA_MAX_WAIT_SAMPLES = df_call_events.loc[_unserved_mask, 'WAIT_TIME_IN_QUEUE_SEC_QTY'].dropna().values.astype(float)
DATA_MAX_WAIT_MEAN = float(np.mean(DATA_MAX_WAIT_SAMPLES))
DATA_MAX_WAIT_MEDIAN = float(np.median(DATA_MAX_WAIT_SAMPLES))
DATA_MAX_WAIT_P25 = float(np.percentile(DATA_MAX_WAIT_SAMPLES, 25))
DATA_MAX_WAIT_P75 = float(np.percentile(DATA_MAX_WAIT_SAMPLES, 75))
DATA_MAX_WAIT_P95 = float(np.percentile(DATA_MAX_WAIT_SAMPLES, 95))
DATA_MAX_WAIT_N = len(DATA_MAX_WAIT_SAMPLES)
print(f"\n⏳ Total wait-time of unserved calls (from {DATA_MAX_WAIT_N:,} calls — QP queue + FIFO, informational):")
print(f"   P25={DATA_MAX_WAIT_P25:.0f}s  Median={DATA_MAX_WAIT_MEDIAN:.0f}s  "
      f"P75={DATA_MAX_WAIT_P75:.0f}s  P95={DATA_MAX_WAIT_P95:.0f}s")
print(f"   Mean={DATA_MAX_WAIT_MEAN:.1f}s  Min={DATA_MAX_WAIT_SAMPLES.min():.0f}s  Max={DATA_MAX_WAIT_SAMPLES.max():.0f}s")
print(f"   → Stored DATA_MAX_WAIT_SAMPLES ({DATA_MAX_WAIT_N:,} values) — informational only, NOT used as simulation input")

# Keep a timestamped copy for later cells
df_calls_qplanner_ts = df_sim.copy()

---
# Part II — Simulation Engine


<a id="simulator-core"></a>
## Simulator Core — State Management Classes

In [ ]:
# =============================================================================
# CELL 9 — SIMULATOR CORE: State management classes
# =============================================================================
# INPUT:  None (defines data structures used by later cells)
#
# DOES:   Defines the core domain objects for the simulation:
#   - QueuedCall:       A call waiting in the queue (call_id, churn_i, arrival_time, duration)
#   - Operator:         An operator (username, group, status=available|busy, busy_until)
#   - Assignment:       Record of a completed assignment (call→operator with scores and wait)
#   - SimulationState:  The full state tracker — manages queue, operators, assignments.
#       Key methods:
#         get_available_operators()   → list of operators with status="available"
#         release_finished_calls()    → frees operators whose busy_until has passed
#         enqueue_calls(new_calls)    → adds calls to queue
#         remove_expired(to_fifo)     → removes calls exceeding max_wait_time (180s)
#                                       to_fifo=True → move to FIFO queue (telephony)
#         remove_fifo_abandoned()     → removes FIFO calls exceeding fifo_max_wait
#         execute_assignment(call, op, churn_ij, score, pipeline_latency)
#           → marks operator busy, removes call from queue, records Assignment
#           → waiting_time = time_in_queue + pipeline_latency
#           → operator busy for: pipeline_latency + call_duration
#         route_fifo(available_ops, score_lookup, pipeline_latency)
#           → assigns FIFO queue calls (oldest first) to available operators
#           → models telephony BAU routing for calls QPlanner couldn't serve
#           → operators get busy → realistic occupancy feedback loop
#   - detect_active_windows(): scans call timestamps, finds gaps > 5min
#       → returns list of (start, end) tuples when QPlanner was active
#       → used by the simulation to pause during QPlanner-inactive periods
#
# OUTPUT: Classes and functions available for cells 10-13
# =============================================================================
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from datetime import datetime, timedelta


@dataclass
class QueuedCall:
    """A call waiting in the queue."""
    call_id: str
    customer_id: str
    arrival_time: datetime
    churn_i: float              # baseline churn prob (no agent)
    call_duration: float = 0    # expected call duration (seconds)


@dataclass
class Operator:
    """An operator that can handle calls."""
    username: str
    group: str
    status: str = "available"   # available | busy
    busy_until: Optional[datetime] = None
    current_call_id: Optional[str] = None


@dataclass
class Assignment:
    """Record of a call-operator assignment decision."""
    tick: int
    timestamp: datetime
    call_id: str
    customer_id: str
    operator: str
    waiting_time: float         # seconds the call waited in queue
    churn_i: float              # baseline churn
    churn_ij: float             # churn with this operator
    churn_reduction: float      # churn_i - churn_ij
    optimization_score: float   # the FO score for this pair


@dataclass
class SimulationState:
    """Tracks the full state of the simulation."""
    current_time: datetime = None
    tick: int = 0
    queue: List[QueuedCall] = field(default_factory=list)
    operators: Dict[str, Operator] = field(default_factory=dict)
    assignments: List[Assignment] = field(default_factory=list)
    unserved_calls: List[dict] = field(default_factory=list)
    fifo_queue: List[QueuedCall] = field(default_factory=list)      # calls awaiting FIFO telephony routing
    fifo_assignments: List[Assignment] = field(default_factory=list) # FIFO-routed assignments
    failed_deliveries: List[dict] = field(default_factory=list)  # recommended but not delivered (AgentNotAvailable)
    max_wait_time: float = 180.0  # max seconds before call expires from QP queue

    def get_available_operators(self) -> List[Operator]:
        return [op for op in self.operators.values() if op.status == "available"]

    def get_imminent_operators(self, window_s: float) -> List[Operator]:
        """Return operators who are busy but will become available within window_s seconds.

        In production, the operatorsavailability_api classifies operators finishing
        their current call soon as "imminent". The brains solver includes them in the
        candidate pool. This method identifies such operators in the simulation.
        """
        if window_s <= 0 or self.current_time is None:
            return []
        cutoff = self.current_time + timedelta(seconds=window_s)
        return [
            op for op in self.operators.values()
            if op.status == "busy" and op.busy_until
            and self.current_time < op.busy_until <= cutoff
        ]

    def release_finished_calls(self):
        """Release operators whose calls have finished."""
        released = 0
        for op in self.operators.values():
            if op.status == "busy" and op.busy_until and self.current_time >= op.busy_until:
                op.status = "available"
                op.busy_until = None
                op.current_call_id = None
                released += 1
        return released

    def enqueue_calls(self, new_calls: List[QueuedCall]):
        """Add new calls to the queue."""
        self.queue.extend(new_calls)

    def remove_expired(self, to_fifo=True):
        """Remove calls that exceeded max wait time (expired from QP queue).

        After max_wait_time (180s, agreed with operations), QPlanner releases
        the call and the telephony system takes over via FIFO routing.

        Args:
            to_fifo: If True, move expired calls to FIFO queue for telephony
                     routing instead of marking them as permanently unserved.
        """
        expired = []
        remaining = []
        for call in self.queue:
            wait = (self.current_time - call.arrival_time).total_seconds()
            if wait > self.max_wait_time:
                expired.append(call)
            else:
                remaining.append(call)
        self.queue = remaining
        if to_fifo:
            self.fifo_queue.extend(expired)
        else:
            for c in expired:
                self.unserved_calls.append({
                    'call_id': c.call_id,
                    'reason': 'expired',
                    'wait_time': (self.current_time - c.arrival_time).total_seconds()
                })
        return len(expired)

    def remove_fifo_abandoned(self, fifo_max_wait: float = 600.0) -> int:
        """Remove calls from FIFO queue that exceed max total wait time.

        Models customer abandonment: customers who waited > fifo_max_wait
        (measured from original arrival_time) hang up and become truly unserved.

        Args:
            fifo_max_wait: Max total wait from original arrival (QP queue + FIFO queue).
                Default 600s (10 min) — customers unlikely to wait longer.

        Returns:
            Number of calls abandoned.
        """
        abandoned = []
        remaining = []
        for call in self.fifo_queue:
            total_wait = (self.current_time - call.arrival_time).total_seconds()
            if total_wait > fifo_max_wait:
                abandoned.append(call)
            else:
                remaining.append(call)
        self.fifo_queue = remaining
        for c in abandoned:
            self.unserved_calls.append({
                'call_id': c.call_id,
                'reason': 'fifo_abandoned',
                'wait_time': (self.current_time - c.arrival_time).total_seconds()
            })
        return len(abandoned)

    def route_fifo(self, available_ops: List[Operator],
                   score_lookup: dict = None,
                   pipeline_latency: float = 2.0,
                   block_operators: bool = False) -> int:
        """
        Route FIFO queue calls to available operators (oldest call first).
        Models telephony BAU routing for calls that QPlanner couldn't serve.

        When block_operators=False (default), FIFO assignments are recorded for
        churn outcome estimation but operators are NOT marked busy. The occupancy
        model already captures telephony BAU utilization implicitly; blocking
        operators here would double-count and starve QPlanner.

        When block_operators=True, operators are marked busy for the full call
        duration (pipeline_latency + call_duration). Use only if the occupancy
        model (occ_scale) has been reduced to compensate.

        Args:
            available_ops: List of currently available Operator objects.
            score_lookup: Optional dict (call_id, operator) → churn_ij.
                If available, uses actual churn_ij; otherwise falls back to churn_i.
            pipeline_latency: Seconds of telephony routing overhead (much
                smaller than QPlanner pipeline — no ML solver, just ACD routing).
            block_operators: If True, mark operators as busy after FIFO assignment.

        Returns:
            Number of FIFO assignments made this tick.
        """
        if not self.fifo_queue or not available_ops:
            return 0

        # Sort FIFO queue by arrival time (oldest first — true FIFO)
        self.fifo_queue.sort(key=lambda c: c.arrival_time)

        assigned = 0
        remaining_ops = list(available_ops)
        remaining_calls = []

        for call in self.fifo_queue:
            if not remaining_ops:
                remaining_calls.append(call)
                continue

            # Assign to first available operator (no optimization)
            op = remaining_ops.pop(0)

            # Look up churn_ij if available; else use churn_i (no optimization benefit)
            churn_ij = score_lookup.get((call.call_id, op.username)) if score_lookup else None
            if churn_ij is None:
                churn_ij = call.churn_i

            wait_time = (self.current_time - call.arrival_time).total_seconds() + pipeline_latency

            if block_operators:
                # Full blocking: operator busy for routing + call duration
                # WARNING: this double-counts with the occupancy model, starving QP
                op.status = "busy"
                op.busy_until = self.current_time + timedelta(
                    seconds=pipeline_latency + call.call_duration
                )
                op.current_call_id = call.call_id
            # else: non-blocking — operator stays available
            # The occupancy model already captures telephony BAU utilization.
            # FIFO records the assignment for churn outcome estimation only.

            # Record FIFO assignment (optimization_score=0 — no optimization)
            self.fifo_assignments.append(Assignment(
                tick=self.tick,
                timestamp=self.current_time,
                call_id=call.call_id,
                customer_id=call.customer_id,
                operator=op.username,
                waiting_time=wait_time,
                churn_i=call.churn_i,
                churn_ij=churn_ij,
                churn_reduction=call.churn_i - churn_ij,
                optimization_score=0.0,
            ))
            assigned += 1

        self.fifo_queue = remaining_calls
        return assigned

    def execute_assignment(self, call: QueuedCall, operator: Operator,
                           churn_ij: float, score: float,
                           pipeline_latency: float = 0.0,
                           acw_time: float = 0.0):
        """Assign a call to an operator and update state.

        pipeline_latency: seconds of overhead between solver decision and operator pickup.
            Models the production pipeline chain (API orchestration → ACD routing → pickup).
            Added to both recorded waiting_time and operator busy duration.
        acw_time: after-call work time in seconds. Operators stay busy after the call
            ends for wrap-up work (CRM notes, system updates). Added to busy_until
            but NOT to customer waiting_time.
        """
        wait_time = (self.current_time - call.arrival_time).total_seconds() + pipeline_latency
        # Update operator state (busy for routing overhead + call duration + ACW)
        operator.status = "busy"
        operator.busy_until = self.current_time + timedelta(seconds=pipeline_latency + call.call_duration + acw_time)
        operator.current_call_id = call.call_id
        # Remove call from queue
        self.queue = [c for c in self.queue if c.call_id != call.call_id]
        # Record assignment
        self.assignments.append(Assignment(
            tick=self.tick,
            timestamp=self.current_time,
            call_id=call.call_id,
            customer_id=call.customer_id,
            operator=operator.username,
            waiting_time=wait_time,
            churn_i=call.churn_i,
            churn_ij=churn_ij,
            churn_reduction=call.churn_i - churn_ij,
            optimization_score=score,
        ))


def detect_active_windows(df_qp_calls: pd.DataFrame, gap_threshold_min: float = 5.0) -> List[Tuple[datetime, datetime]]:
    """
    Detect QPlanner active windows from call timestamps.
    A gap > gap_threshold_min minutes means QPlanner was inactive.
    Returns list of (window_start, window_end) tuples.
    """
    if df_qp_calls.empty:
        return []

    ts = pd.to_datetime(df_qp_calls['START_DATE_TIME']).sort_values().reset_index(drop=True)
    threshold = pd.Timedelta(minutes=gap_threshold_min)

    windows = []
    window_start = ts.iloc[0]
    prev = ts.iloc[0]

    for t in ts.iloc[1:]:
        if t - prev > threshold:
            windows.append((window_start, prev))
            window_start = t
        prev = t

    windows.append((window_start, prev))

    return windows


def analyze_simulation(state: SimulationState, title: str = "Simulation") -> pd.DataFrame:
    """Convert simulation results to a DataFrame and show summary stats."""
    if not state.assignments:
        print("⚠️ No assignments to analyze")
        return pd.DataFrame()

    df_results = pd.DataFrame([
        {
            'tick': a.tick,
            'timestamp': a.timestamp,
            'call_id': a.call_id,
            'operator': a.operator,
            'waiting_time': a.waiting_time,
            'churn_i': a.churn_i,
            'churn_ij': a.churn_ij,
            'churn_reduction': a.churn_reduction,
            'optimization_score': a.optimization_score,
        }
        for a in state.assignments
    ])

    print(f"{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(f"  QP Assignments: {len(df_results)}")
    print(f"  Unserved:       {len(state.unserved_calls)}")
    print(f"  Service rate:   {len(df_results) / (len(df_results) + len(state.unserved_calls)) * 100:.1f}%")
    print()
    print(f"  Waiting Time (seconds):")
    print(f"    Mean:   {df_results['waiting_time'].mean():.1f}")
    print(f"    Median: {df_results['waiting_time'].median():.1f}")
    print(f"    P95:    {df_results['waiting_time'].quantile(0.95):.1f}")
    print(f"    Max:    {df_results['waiting_time'].max():.1f}")
    print()
    print(f"  Churn Reduction:")
    print(f"    Mean:   {df_results['churn_reduction'].mean():.4f}")
    print(f"    Median: {df_results['churn_reduction'].median():.4f}")
    print(f"    Total:  {df_results['churn_reduction'].sum():.2f}")

    if state.fifo_assignments:
        fifo_waits = [a.waiting_time for a in state.fifo_assignments]
        total_served = len(df_results) + len(state.fifo_assignments)
        print()
        print(f"  FIFO Telephony Routing:")
        print(f"    FIFO assigned:  {len(state.fifo_assignments)}")
        print(f"    Total served:   {total_served} (QP: {len(df_results)} + FIFO: {len(state.fifo_assignments)})")
        print(f"    Overall svc rate: {total_served / (total_served + len(state.unserved_calls)) * 100:.1f}%")
        print(f"    Avg wait:       {np.mean(fifo_waits):.1f}s (median: {np.median(fifo_waits):.1f}s)")

    print(f"{'='*60}")

    return df_results


print("✅ Simulator classes defined: QueuedCall, Operator, Assignment, SimulationState, detect_active_windows, analyze_simulation")
print("   SimulationState now includes: fifo_queue, fifo_assignments, failed_deliveries, route_fifo(), remove_fifo_abandoned()")
print("   + get_imminent_operators(window_s): busy operators freeing up within window_s seconds")
print("   QPlanner max_wait_time = 180s (fixed, agreed with operations)")

<a id="fo-scoring"></a>
## Scoring Function (FO)

In [ ]:
# =============================================================================
# CELL 9 — OBJECTIVE FUNCTION (FO) CONFIG + SCORING
# =============================================================================
# INPUT:
#   - churn_i:    P(churn | call)  — baseline query (model-predicted)
#   - churn_ij:   P(churn | call, operator)  — pair-specific (from smartpairing)
#   - waiting_time: seconds since call entered the queue
#   - FOConfig:   all FO parameters in one dataclass
#
# DOES:
#   Computes a single score for a (call, operator) pair.
#   The Hungarian algorithm then finds the 1-to-1 matching that maximises
#   the sum of scores across all assignments.
#
#   Production formula:
#     G = (0.5 + g_value) · c
#       where g_value = (churn_i − churn_ij) / (|churn_i| + |churn_ij|)
#             c = sigmoid(K · churn_i − τ)
#     P = exponential(waiting_time)    normalized to [0, 1] by TE_MIN/TE_MAX
#     FO = β_G · G + β_P · P + κ
#
#   The KAPPA gate (κ < 0) means FO can go negative for short waits with no
#   churn reduction → call is NOT assigned (waits for better operator or FIFO).
#
# OUTPUT: float score — higher = better assignment
# =============================================================================

from dataclasses import dataclass


def sigmoid(x: float) -> float:
    """Standard logistic sigmoid: 1 / (1 + exp(-x))."""
    return 1.0 / (1.0 + np.exp(-x))


@dataclass
class FOConfig:
    """Configuration for the Objective Function (FO) used in the simulation.

    All parameters default to production values so FOConfig() reproduces
    the current QPlanner behavior exactly.
    """
    # --- Production FO parameters ---
    TE_MIN: float = 50.0          # seconds — below this, P goes negative (penalty)
    TE_MAX: float = 180.0         # seconds — P saturates at 1.0
    ALFA: float = 3.0             # exponential curvature for P (higher = more convex)
    BETA_G: float = 0.35          # weight on G component (churn reduction)
    BETA_P: float = 0.65          # weight on P component (waiting time urgency)
    KAPPA: float = -0.30414       # intercept (gate) — FO < 0 → don't assign
    CHURN_K: float = 20.0         # steepness of sigmoid gate for c
    CHURN_TAU: float = 5.0        # shift parameter for sigmoid gate

    # --- Experimental toggles ---
    USE_DISCOUNT_WEIGHTING: bool = False  # When True, c = sigmoid(churn_i) × discount_i
                                          # focuses on calls that are both high-risk AND actionable
    # Direct churn oracle mode: bypass FO entirely, score = 1 - churn_ij
    USE_DIRECT_CHURN: bool = False  # When True, Hungarian minimises raw churn_ij (theoretical ceiling)

    # ── Two-tier routing mode ──────────────────────────────────────
    # Structural change to break the zero-sum constraint.
    # Tier 1 (salvageable): high churn_i + high operator spread → wait for RIGHT operator
    # Tier 2 (routine): everything else → assign immediately to non-valuable operators
    USE_TWO_TIER: bool = False
    TIER1_CALL_IDS: set = None       # pre-computed set of Tier 1 call IDs (churn_i > thresh AND best_reduction > thresh)
    TIER1_VALUABLE_OPS: set = None   # operators whose mean cij on Tier 1 calls is below median (good at saving)
    TIER1_MIN_REDUCTION: float = 0.05  # minimum (churn_i - churn_ij) for a Tier 1 assignment to proceed
    TIER1_STRICT_RESERVE: bool = False  # When True, valuable ops NEVER serve T2 (always held for future T1)
                                        # When False, valuable ops serve T2 when no T1 calls are waiting
    # ── Slate-based lift mode ─────────────────────────────────────
    # Instead of reduction = churn_i - churn_ij (biased artifact baseline),
    # use reduction = max(churn_ij in slate) - churn_ij (worst-op-in-slate baseline).
    # Eliminates negative operator contributions. Always ≥ 0.
    # Gate: "operator must capture ≥ TIER1_LIFT_FRAC of available spread".
    # e.g., TIER1_LIFT_FRAC=0.50 → operator must be in the better half of the slate.
    USE_SLATE_LIFT: bool = False
    TIER1_LIFT_FRAC: float = 0.50   # fraction of (max_cij - min_cij) that lift must reach
                                     # 0.50 = "at least 50% of available quality range"

    # ── Slate-based G component ───────────────────────────────────
    # USE_SLATE_G: replace churn_i with max(churn_ij) across the tick's
    # available operator slate in the G component of the FO formula.
    # Production uses churn_i = P(churn | call) computed by nulling operator
    # features — asking the model a question it was never designed to answer.
    # Slate G uses max_cij = worst operator for this call in current slate,
    # which is always a real model output (conditioned on both call AND operator).
    # Same FO formula, same gate c=sigmoid(K·churn_i−τ), same P — only the
    # g_value numerator changes: (max_cij − churn_ij) instead of (churn_i − churn_ij).
    USE_SLATE_G: bool = False

    # ── Layer 1: Imminent operator modelling ─────────────────────
    # In production, operatorsavailability_api reports operators finishing
    # their current call soon as "imminent". The brains solver includes
    # them in the candidate pool alongside truly-available operators.
    # IMMINENT_WINDOW_S controls how far ahead (seconds) we look for busy
    # operators about to free up. At each tick, operators with
    # current_time < busy_until ≤ current_time + IMMINENT_WINDOW_S
    # are added to the solver pool (default behaviour, matching production).
    #
    # S1 (EXCLUDE_IMMINENT_FROM_SLATE=True): solver sees ONLY truly-available
    # operators — tests what happens without imminent inflation.
    EXCLUDE_IMMINENT_FROM_SLATE: bool = False
    IMMINENT_WINDOW_S: float = 120.0  # seconds — operators finishing within this window are "imminent"
                                       # 120s reproduces ~40-45% imminent fraction observed in production


def compute_score(
    churn_i: float,
    churn_ij: float,
    waiting_time: float,
    config: FOConfig,
    discount_i: float = 0.0,  # normalized operator spread for this call [0,1]
    g_baseline: float = None,  # when set, g_value uses this instead of churn_i (slate-G)
) -> float:
    """
    Compute the optimization score for a (call, operator) pair.

    This is the function you swap out to test different FO formulations.
    Returns a score where HIGHER = better assignment.

    Parameters:
      churn_i:    original P(churn|call) — always used for the sigmoid gate c.
      g_baseline: if provided, replaces churn_i ONLY in the g_value numerator
                  (used by USE_SLATE_G to anchor reduction to max_cij).

    Modes:
      - Default: FO = β_G·G + β_P·P + κ  (production formula)
      - USE_DISCOUNT_WEIGHTING: G weights c = sigmoid(churn_i) × discount_i
      - USE_DIRECT_CHURN: bypasses FO entirely → score = 1 - churn_ij
        (theoretical ceiling — Hungarian directly minimises churn_ij)
    """
    # --- Oracle mode: bypass FO, directly optimise churn_ij ---
    if config.USE_DIRECT_CHURN:
        return 1.0 - churn_ij  # always > 0 (no gate), lower churn_ij = higher score

    # --- G component: churn reduction gain ---
    # g_value uses g_baseline (e.g. max_cij from slate) when provided;
    # c gate ALWAYS uses churn_i for risk-based prioritization.
    _g_base = g_baseline if g_baseline is not None else churn_i
    denom = abs(_g_base) + abs(churn_ij)
    g_value = (_g_base - churn_ij) / denom if denom > 1e-9 else 0.0
    c_sig = sigmoid(config.CHURN_K * churn_i - config.CHURN_TAU)  # always churn_i
    if config.USE_DISCOUNT_WEIGHTING:
        c = c_sig * discount_i  # risk × actionability
    else:
        c = c_sig
    G = (0.5 + g_value) * c

    # Production only clips to TE_MAX (no lower bound).
    # When wait < TE_MIN, P goes negative → penalizes very early assignments.
    clamped_wait = min(waiting_time, config.TE_MAX)
    te_range = config.TE_MAX - config.TE_MIN
    norm_wait = (clamped_wait - config.TE_MIN) / te_range if te_range > 0 else 0.0
    exp_alfa = np.exp(config.ALFA)
    P = (np.exp(config.ALFA * norm_wait) - 1.0) / (exp_alfa - 1.0) if exp_alfa > 1 else norm_wait

    # --- Combined FO ---
    score = config.BETA_G * G + config.BETA_P * P + config.KAPPA
    return score


# Quick sanity check with production defaults
cfg = FOConfig()
test_score = compute_score(churn_i=0.7, churn_ij=0.3, waiting_time=60.0, config=cfg)
print(f"✅ FOConfig (production defaults):")
print(f"   TE_MIN={cfg.TE_MIN}, TE_MAX={cfg.TE_MAX}, ALFA={cfg.ALFA}")
print(f"   BETA_G={cfg.BETA_G}, BETA_P={cfg.BETA_P}, KAPPA={cfg.KAPPA}")
print(f"   CHURN_K={cfg.CHURN_K}, CHURN_TAU={cfg.CHURN_TAU}")
print(f"   USE_DISCOUNT_WEIGHTING={cfg.USE_DISCOUNT_WEIGHTING}")
print(f"   USE_DIRECT_CHURN={cfg.USE_DIRECT_CHURN}")
print(f"   USE_TWO_TIER={cfg.USE_TWO_TIER}")
print(f"   TIER1_STRICT_RESERVE={cfg.TIER1_STRICT_RESERVE}")
print(f"   USE_SLATE_LIFT={cfg.USE_SLATE_LIFT}")
print(f"   TIER1_LIFT_FRAC={cfg.TIER1_LIFT_FRAC}")
print(f"   USE_SLATE_G={cfg.USE_SLATE_G}")
print(f"   EXCLUDE_IMMINENT_FROM_SLATE={cfg.EXCLUDE_IMMINENT_FROM_SLATE}")
print(f"   IMMINENT_WINDOW_S={cfg.IMMINENT_WINDOW_S}")
print(f"   Test score (churn_i=0.7, churn_ij=0.3, wait=60s): {test_score:.4f}")

# Oracle mode sanity check
cfg_oracle = FOConfig(USE_DIRECT_CHURN=True)
test_oracle = compute_score(churn_i=0.7, churn_ij=0.3, waiting_time=60.0, config=cfg_oracle)
print(f"   Oracle score (same inputs): {test_oracle:.4f}  (= 1.0 - 0.3)")

print(f"\n   Gate threshold analysis (how much G is needed at various wait times):")
for w in [30, 50, 100, 120, 140, 160, 180]:
    cw = min(w, cfg.TE_MAX)  # production only clips upper bound
    nw = (cw - cfg.TE_MIN) / (cfg.TE_MAX - cfg.TE_MIN)
    p = (np.exp(cfg.ALFA * nw) - 1) / (np.exp(cfg.ALFA) - 1)
    p_weighted = cfg.BETA_P * p
    g_needed = (abs(cfg.KAPPA) - p_weighted) / cfg.BETA_G if cfg.BETA_G > 0 else 0
    print(f"   wait={w:>3}s → P={p:.3f}, β_P·P={p_weighted:.3f}, G needed for FO>0: {max(0, g_needed):.3f}")

<a id="assignment-solver"></a>
## Assignment Solver (Hungarian Algorithm)

In [ ]:
# =============================================================================
# CELL 11 — ASSIGNMENT SOLVER: Optimal matching via Hungarian algorithm
# =============================================================================
# INPUT (per tick):
#   - queue: list of QueuedCall objects currently waiting
#   - available_ops: list of Operator objects currently available
#   - score_lookup: dict (call_id, operator) → churn_ij (from cell 7)
#   - churn_i_lookup: dict call_id → churn_i (from cell 7)
#   - current_time: datetime (for computing waiting_time)
#   - fo_config: FOConfig with production parameters
#   - fallback_churn_ij: optional fallback for missing pairs
#   - call_discounts: pre-computed {call_id: normalized_spread} for discount-priority
#
# DOES:
#   1. Builds an (n_calls × n_ops) cost matrix
#   2. For each (call, operator) pair:
#      - Looks up churn_ij from score_lookup (or uses fallback)
#      - Computes FO score via compute_score()
#      - Sets cost = -score (Hungarian minimizes, we want to maximize)
#      - Infeasible pairs (no churn_ij) get cost = 1e6 (effectively infinite)
#   3. Solves with scipy.optimize.linear_sum_assignment (Hungarian algorithm)
#      → Guarantees 1-to-1 matching: each call ≤ 1 operator, each op ≤ 1 call
#   4. Filters results: only keeps pairs where score > 0 (KAPPA gate)
#
# OUTPUT: List of (QueuedCall, Operator, churn_ij, score) tuples to assign
# =============================================================================

from scipy.optimize import linear_sum_assignment


def solve_assignments(
    queue: List[QueuedCall],
    available_ops: List[Operator],
    score_lookup: dict,
    churn_i_lookup: dict,
    current_time: datetime,
    fo_config: FOConfig,
    min_score_threshold: float = 0.0,
    fallback_churn_ij: float = None,  # fallback churn_ij for missing (call, operator) pairs
    call_discounts: dict = None,  # pre-computed {call_id: normalized_spread} for discount-priority
) -> List[Tuple[QueuedCall, Operator, float, float]]:
    """
    Solve the optimal call-operator assignment.

    Returns list of (call, operator, churn_ij, score) tuples.
    Only returns pairs where score > min_score_threshold.

    If fallback_churn_ij is set, pairs without a pre-computed score
    will use this value instead of being marked infeasible. This
    dramatically increases score coverage (matching real smartpairing
    behavior where ALL available operators get scores).

    If call_discounts is set and fo_config.USE_DISCOUNT_WEIGHTING is True,
    passes per-call discount_i to compute_score for operator-spread weighting.
    """
    if not queue or not available_ops:
        return []

    n_calls = len(queue)
    n_ops = len(available_ops)

    # Build cost matrix (we negate scores because linear_sum_assignment minimizes)
    LARGE_COST = 1e6  # penalty for infeasible pairs (no score available)
    cost_matrix = np.full((n_calls, n_ops), LARGE_COST)
    score_matrix = np.zeros((n_calls, n_ops))
    churn_ij_matrix = np.full((n_calls, n_ops), np.nan)

    # ── Slate-based G: compute max_cij per call from this tick's slate ──
    _slate_max_cij = {}  # call_id → max churn_ij across available ops
    if fo_config.USE_SLATE_G:
        for i, call in enumerate(queue):
            max_cij = None
            for j, op in enumerate(available_ops):
                cij = score_lookup.get((call.call_id, op.username))
                if cij is None and fallback_churn_ij is not None:
                    cij = fallback_churn_ij
                if cij is not None:
                    if max_cij is None or cij > max_cij:
                        max_cij = cij
            if max_cij is not None:
                _slate_max_cij[call.call_id] = max_cij

    for i, call in enumerate(queue):
        wait = (current_time - call.arrival_time).total_seconds()
        _discount = call_discounts.get(call.call_id, 0.0) if call_discounts else 0.0
        # G baseline: use max_cij from slate (USE_SLATE_G) or None (production uses churn_i)
        _g_base = _slate_max_cij.get(call.call_id) if fo_config.USE_SLATE_G else None
        for j, op in enumerate(available_ops):
            # Look up pre-computed churn_ij for this (call, operator) pair
            churn_ij = score_lookup.get((call.call_id, op.username))
            if churn_ij is None and fallback_churn_ij is not None:
                # Use fallback: mimic smartpairing computing a "neutral" score
                churn_ij = fallback_churn_ij
            if churn_ij is not None:
                score = compute_score(call.churn_i, churn_ij, wait, fo_config, discount_i=_discount, g_baseline=_g_base)
                score_matrix[i, j] = score
                churn_ij_matrix[i, j] = churn_ij
                cost_matrix[i, j] = -score  # negate for minimization

    # Solve assignment (handles rectangular matrices)
    row_idx, col_idx = linear_sum_assignment(cost_matrix)

    # Filter: only keep assignments with valid scores above threshold
    results = []
    for i, j in zip(row_idx, col_idx):
        if cost_matrix[i, j] < LARGE_COST and score_matrix[i, j] > min_score_threshold:
            results.append((
                queue[i],
                available_ops[j],
                churn_ij_matrix[i, j],
                score_matrix[i, j],
            ))

    return results


print("✅ Assignment solver defined (Hungarian algorithm via scipy, with fallback + discount-priority + slate-G)")

<a id="two-tier-solver"></a>
## Two-Tier Solver (`solve_tiered`)

In [ ]:
# =============================================================================
# CELL 12 — TWO-TIER SOLVER: Structural change to break zero-sum
# =============================================================================
# INPUT:  queue, available_ops, score_lookup, churn_i_lookup, fo_config
#
# DOES:   Implements two-tier routing:
#   Tier 1 (salvageable) = pre-computed high-risk + high operator-spread calls
#     → Solved FIRST with valuable operators using oracle scoring (1-churn_ij)
#     → Min-reduction gate: only assign if reduction ≥ threshold
#       - Default: reduction = churn_i - churn_ij (baseline is churn_i artifact)
#       - USE_SLATE_LIFT: reduction = max_cij_in_slate - churn_ij (baseline is worst op)
#         with threshold = TIER1_LIFT_FRAC × (max_cij - min_cij)
#     → If no good enough operator available → call WAITS (up to 180s naturally)
#
#   Tier 2 (routine) = everything else
#     → Solved SECOND with remaining operators
#     → Uses same oracle scoring but NO min-reduction gate
#     → Operator pool depends on strict-reserve mode:
#       (a) Default: if T1 calls still waiting → reserve valuable ops, T2 gets only non-valuable
#                    if no T1 waiting → T2 gets ALL remaining ops (including unused valuable)
#       (b) Strict (TIER1_STRICT_RESERVE=True): T2 NEVER gets valuable ops, even if idle
#
# OUTPUT: list of (call, operator, churn_ij, score) pairs
# =============================================================================

def solve_tiered(
    queue: list,
    available_ops: list,
    score_lookup: dict,
    churn_i_lookup: dict,
    current_time: float,
    fo_config: FOConfig,
    fallback_churn_ij: float = 0.5,
    call_discounts: dict = None,
) -> list:
    """Two-tier solver: split queue by call value class, solve T1 first with valuable ops."""
    t1_ids = fo_config.TIER1_CALL_IDS or set()
    val_ops_set = fo_config.TIER1_VALUABLE_OPS or set()
    min_red = fo_config.TIER1_MIN_REDUCTION
    _use_slate = fo_config.USE_SLATE_LIFT
    _lift_frac = fo_config.TIER1_LIFT_FRAC

    # --- Split queue by tier ---
    t1_queue = [c for c in queue if c.call_id in t1_ids]
    t2_queue = [c for c in queue if c.call_id not in t1_ids]

    # --- Split operators by value class ---
    valuable_ops = [op for op in available_ops if op.username in val_ops_set]
    other_ops = [op for op in available_ops if op.username not in val_ops_set]

    all_pairs = []
    used_ops = set()

    # Oracle config for both tiers (bypass FO, minimize churn_ij)
    _oracle_cfg = FOConfig(USE_DIRECT_CHURN=True)

    # ── Pre-compute slate stats for slate-based lift ─────────────
    # For each T1 call, find max_cij and min_cij across available valuable operators
    _slate_max = {}
    _slate_min = {}
    if _use_slate and t1_queue and valuable_ops:
        _val_usernames = [op.username for op in valuable_ops]
        for call in t1_queue:
            cij_vals = []
            for op_name in _val_usernames:
                key = (call.call_id, op_name)
                cij = score_lookup.get(key)
                if cij is not None:
                    cij_vals.append(cij)
            if cij_vals:
                _slate_max[call.call_id] = max(cij_vals)
                _slate_min[call.call_id] = min(cij_vals)
            else:
                _slate_max[call.call_id] = fallback_churn_ij
                _slate_min[call.call_id] = fallback_churn_ij

    # ── Step 1: Tier 1 × valuable operators ──────────────────────
    if t1_queue and valuable_ops:
        t1_pairs = solve_assignments(
            queue=t1_queue,
            available_ops=valuable_ops,
            score_lookup=score_lookup,
            churn_i_lookup=churn_i_lookup,
            current_time=current_time,
            fo_config=_oracle_cfg,
            fallback_churn_ij=fallback_churn_ij,
            call_discounts=call_discounts,
        )
        # Apply min-reduction gate: only assign if reduction is meaningful
        for call, op, cij, score in t1_pairs:
            if _use_slate:
                # Slate-based lift: reduction relative to worst op in slate
                max_cij = _slate_max.get(call.call_id, fallback_churn_ij)
                min_cij = _slate_min.get(call.call_id, fallback_churn_ij)
                spread = max_cij - min_cij
                reduction = max_cij - cij
                # Gate: operator must capture ≥ LIFT_FRAC of the available spread
                threshold = _lift_frac * spread if spread > 1e-6 else 0.0
                passes_gate = reduction >= threshold
            else:
                # Original: reduction relative to churn_i artifact
                reduction = call.churn_i - cij
                passes_gate = reduction >= min_red

            if passes_gate:
                all_pairs.append((call, op, cij, score))
                used_ops.add(op.username)
            # else: call stays in queue → will wait for a better operator next tick

    # ── Step 2: Tier 2 × remaining operators ─────────────────────
    # Determine which operators are available for Tier 2
    t1_assigned_ids = {c.call_id for c, _, _, _ in all_pairs if c.call_id in t1_ids}
    t1_still_waiting = len(t1_queue) - len(t1_assigned_ids)

    remaining_ops = [op for op in available_ops if op.username not in used_ops]

    if t1_still_waiting > 0 or fo_config.TIER1_STRICT_RESERVE:
        # Valuable ops reserved for Tier 1:
        #   - Default mode: only when T1 calls are still waiting
        #   - Strict mode: ALWAYS (valuable ops NEVER serve T2)
        t2_ops = [op for op in remaining_ops if op.username not in val_ops_set]
    else:
        # No Tier 1 calls waiting AND not strict → Tier 2 can use ALL remaining operators
        t2_ops = remaining_ops

    if t2_queue and t2_ops:
        t2_pairs = solve_assignments(
            queue=t2_queue,
            available_ops=t2_ops,
            score_lookup=score_lookup,
            churn_i_lookup=churn_i_lookup,
            current_time=current_time,
            fo_config=_oracle_cfg,
            fallback_churn_ij=fallback_churn_ij,
            call_discounts=call_discounts,
        )
        all_pairs.extend(t2_pairs)

    return all_pairs


<a id="simulation-loop"></a>
## Main Simulation Loop

In [ ]:
# =============================================================================
# CELL 12 — MAIN SIMULATION LOOP (gap-aware, with FIFO telephony routing)
# =============================================================================
# INPUT:
#   - df_call_events: one row per call with arrival_time, churn_i, duration (from cell 7)
#   - df_operators: operator pool for the day (from get_day_operators)
#   - score_lookup: (call_id, operator) → churn_ij (from cell 7)
#   - churn_i_lookup: call_id → churn_i (from cell 7)
#   - fo_config: FOConfig with production parameters (from cell 10)
#   - active_windows: QPlanner on/off periods (from detect_active_windows)
#   - operator_shifts: per-operator shift windows (from get_day_operators)
#   - hourly_occupancy: hour → occupancy fraction (computed per-day in cell 12)
#   - Calibrated parameters: tick_interval, pipeline_latency, nqp_mean_task_s
#
# DOES (main tick loop):
#   Each tick (every 3s of simulated time):
#     0. CHECK GAP: if not in active window, block 70% of operators and skip
#        On gap→active transition, move queued calls to FIFO (telephony takes over)
#     1. RELEASE: free operators whose (pipeline_latency + call_duration + ACW) elapsed
#     2. ENQUEUE: add new calls that arrived since last tick
#     3. CALLBACK DEFLECTION: calls waiting ≥ 55s get one-time callback offer
#     4. EXPIRE: remove calls waiting > max_wait_time (180s) → move to FIFO queue
#     4b. FIFO ABANDON: remove FIFO calls where total wait > fifo_max_wait
#
#   ┌─────────────────────────────────────────────────────────────────────────┐
#   │ OPERATOR AVAILABILITY PIPELINE (6-stage filter, each tick)             │
#   │                                                                       │
#   │ An operator must pass ALL stages to reach the solver:                 │
#   │                                                                       │
#   │  Stage 1: release_finished_calls()                                    │
#   │    → Free operators whose call + ACW ended (busy_until ≤ now).        │
#   │    → Operator goes status="available".                                │
#   │                                                                       │
#   │  Stage 2: get_available_operators()                                   │
#   │    → Filter out status="busy" operators (handling QP or FIFO calls).  │
#   │    → These never reach any occupancy check.                           │
#   │                                                                       │
#   │  Stage 3: Shift filter                                                │
#   │    → Remove operators outside their shift window (start ≤ now ≤ end). │
#   │                                                                       │
#   │  Stage 4: Occupancy check — ongoing non-QP task?                      │
#   │    → Check nqp_busy_until[op] > current_time.                         │
#   │    → If YES: operator still doing admin/breaks/cross-team work from   │
#   │      a previous tick → skip (no dice roll, deterministic).            │
#   │                                                                       │
#   │  Stage 5: Occupancy check — start NEW non-QP task?                    │
#   │    → Roll random() < p_start (≈0.15% per tick).                       │
#   │    → If YES: assign exponential-duration task (~20min mean),          │
#   │      record nqp_busy_until[op] = now + duration → skip.              │
#   │    → If NO: operator is truly available.                              │
#   │                                                                       │
#   │  Stage 6: Solver (Hungarian algorithm)                                │
#   │    → Only operators surviving all 5 filters enter the cost matrix.    │
#   │    → KAPPA gate (FO > 0) further restricts actual assignments.        │
#   │                                                                       │
#   │ KEY: Stages 4-5 use a persistent nqp_busy_until dict that carries     │
#   │ state across ticks. Non-QP tasks last ~20min (hundreds of ticks),     │
#   │ producing bursty correlated unavailability, not memoryless coin flips. │
#   │ CALIBRATED_OCC_SCALE controls the effective occupancy rate that       │
#   │ drives p_start, calibrated so sim QP served ≈ historical served.     │
#   └─────────────────────────────────────────────────────────────────────────┘
#
#     7. EXECUTE QP assignments (mark operators busy for latency + duration + ACW)
#     8. FIFO ROUTE (PRIORITY 2 — Telephony): remaining operators serve FIFO queue
#        → oldest call first, no ML optimization, ~2s routing latency
#        → only fifo_routing_fraction of leftover operators offered (ACD pacing)
#        → non-blocking by default (churn estimation only, no double-count with occ model)
#     9. ADVANCE CLOCK by tick_interval seconds
#
# OUTPUT: SimulationState containing:
#   - state.assignments: QPlanner-optimized assignments
#   - state.fifo_assignments: telephony FIFO-routed assignments
#   - state.unserved_calls: abandoned/end-of-day calls (truly lost)
#   - state.operators: final operator states
#   - state.score_coverage: dict with solver-level coverage stats
# =============================================================================

def _is_in_active_window(t: datetime, windows: List[Tuple[datetime, datetime]]) -> bool:
    """Check if timestamp t falls within any QPlanner active window."""
    for ws, we in windows:
        if ws <= t <= we:
            return True
    return False


def run_simulation(
    df_call_events: pd.DataFrame,
    df_operators: pd.DataFrame,
    score_lookup: dict,
    churn_i_lookup: dict,
    fo_config: FOConfig = FOConfig(),
    tick_interval: int = 5,         # seconds between decisions
    max_wait_time: float = 180.0,   # seconds before call expires from QP queue (fixed, agreed with operations)
    sim_day: str = None,            # simulate a single day (YYYY-MM-DD), or None for all
    active_windows: List[Tuple[datetime, datetime]] = None,  # QPlanner active windows
    gap_busy_fraction: float = 0.7, # fraction of operators busy during gaps (other strategy)
    base_occupancy: float = 0.0,    # fraction of operators unavailable per tick (non-QP work)
    hourly_occupancy: Dict[int, float] = None,  # hour → occupancy override (replaces base_occupancy)
    operator_shifts: Dict[str, Tuple[datetime, datetime]] = None,  # per-operator shift windows
    fallback_churn_ij: float = None,  # fallback churn_ij for missing scores (enables full coverage)
    pipeline_latency: float = 0.0,    # seconds of overhead: API chain + ACD routing + operator pickup
    pipeline_latency_std: float = 0.0,  # std of pipeline latency (lognormal). 0 = fixed latency.
    nqp_mean_task_s: float = 1200.0,   # mean duration of non-QP tasks (seconds). Controls occupancy persistence. (~20 min; matches CALIBRATED_NQP_TASK_S)
    fifo_latency: float = 2.0,        # telephony FIFO routing latency (much faster than QP pipeline)
    fifo_max_wait: float = 600.0,     # max total wait (arrival→abandon) for FIFO calls (seconds)
    fifo_routing_fraction: float = 1.0, # fraction of leftover operators offered to FIFO each tick (0-1). <1 = ACD pacing buffer
    fifo_block_operators: bool = False,  # if True, FIFO marks operators busy for call_duration (double-counts with occ model!)
    enable_fifo: bool = True,          # enable FIFO telephony routing for expired/gap-cleared calls
    delivery_failure_rate: float = 0.0,  # probability of delivery failure per assignment (production ~14.2%)
    callback_deflection_time: float = 0.0,  # seconds of wait before telephony offers callback (0 = disabled)
    callback_deflection_rate: float = 0.0,  # probability [0,1] of accepting callback offer at deflection_time
    acw_mean_s: float = 0.0,  # mean after-call work time (seconds). Operators stay busy after call ends. 0 = disabled.
    seed: int = 42,  # random seed for stochastic components (occupancy, delivery failures, callbacks, ACW, latency)
    verbose: bool = True,
) -> SimulationState:
    """
    Run the QPlanner simulator (gap-aware, with FIFO telephony routing).

    The KAPPA gate is modeled naturally: every tick, ALL queued calls are scored
    against available operators. The solver computes FO for each (call, operator)
    pair, and only pairs with FO > 0 are assigned. With KAPPA = -0.30414, this
    means calls need sufficient P (waiting-time priority) to overcome |KAPPA|.

    FIFO telephony routing: calls that QPlanner couldn't serve (expired after
    max_wait_time=180s or cleared during gaps) enter a FIFO queue. After
    QPlanner has first pick of available operators each tick, remaining
    operators serve FIFO calls oldest-first. This creates a realistic occupancy
    feedback loop: FIFO calls consume operator time → fewer operators available
    for QPlanner → more realistic simulation.

    Customer abandonment: FIFO calls exceeding fifo_max_wait (default 600s = 10min
    from original arrival) are abandoned — customers unlikely to wait longer.

    Two-tier mode (fo_config.USE_TWO_TIER=True): replaces the standard solver
    with solve_tiered(), which splits calls into Tier 1 (salvageable) and
    Tier 2 (routine), reserves valuable operators for Tier 1, and applies
    a minimum-reduction gate. See solve_tiered() docstring for details.
    """
    # --- Filter to simulation day if specified ---
    events = df_call_events.copy()
    if sim_day:
        events = events[events['arrival_time'].dt.date == pd.to_datetime(sim_day).date()]
        if events.empty:
            print(f"⚠️ No calls found for {sim_day}")
            return SimulationState()
    events = events.sort_values('arrival_time').reset_index(drop=True)

    # --- Initialize state ---
    state = SimulationState(max_wait_time=max_wait_time)
    state.current_time = events['arrival_time'].iloc[0]

    # Initialize operators (all start as available)
    for _, row in df_operators.iterrows():
        state.operators[row['agent_username']] = Operator(
            username=row['agent_username'],
            group=row['agent_groupname'],
        )

    # --- Initialize non-QP task tracking (task-based occupancy model) ---
    nqp_rng = np.random.RandomState(seed)
    nqp_busy_until = {}
    first_hour = events['arrival_time'].iloc[0].hour
    initial_occ = hourly_occupancy.get(first_hour, base_occupancy) if hourly_occupancy else base_occupancy
    for op_name in state.operators:
        if nqp_rng.random() < initial_occ:
            remaining = nqp_rng.exponential(nqp_mean_task_s / 2)
            nqp_busy_until[op_name] = state.current_time + timedelta(seconds=remaining)

    sim_end = events['arrival_time'].iloc[-1] + timedelta(seconds=max_wait_time)
    next_call_idx = 0
    total_calls = len(events)
    log_every = max(1, int(60 / tick_interval))
    was_in_gap = False

    # Score coverage tracking
    _total_solver_pairs = 0
    _total_solver_hits = 0
    _tick_ops_log = []     # (n_ops, n_queue) per solver-active tick
    _callback_deflections = 0  # counter for callback-deflected calls
    _callback_offered = set()  # track call_ids already offered callback (one-time offer)

    # Pre-compute per-call operator discount (spread) for discount-priority mode
    _call_discounts = None
    if fo_config.USE_DISCOUNT_WEIGHTING:
        from collections import defaultdict
        _cij_by_call = defaultdict(list)
        for (cid, _op), cij_val in score_lookup.items():
            _cij_by_call[cid].append(cij_val)
        _spreads = {cid: max(vals) - min(vals) for cid, vals in _cij_by_call.items() if len(vals) >= 2}
        _max_spread = max(_spreads.values()) if _spreads else 1.0
        _call_discounts = {cid: sp / _max_spread for cid, sp in _spreads.items()}
        if verbose:
            print(f"   Discount-priority: {len(_call_discounts):,} calls scored, "
                  f"max spread={_max_spread:.4f}, median={np.median(list(_spreads.values())):.4f}")

    # Determine solver mode
    _use_tiered = fo_config.USE_TWO_TIER and fo_config.TIER1_CALL_IDS

    if verbose:
        gap_mode = "ON" if active_windows else "OFF"
        fifo_mode = "ON" if enable_fifo else "OFF"
        print(f"🚀 Starting simulation: {len(events)} calls, {len(state.operators)} operators")
        print(f"   FO config: β_G={fo_config.BETA_G}, β_P={fo_config.BETA_P}, κ={fo_config.KAPPA}")
        if _use_tiered:
            _n_t1 = len(fo_config.TIER1_CALL_IDS)
            _n_vops = len(fo_config.TIER1_VALUABLE_OPS) if fo_config.TIER1_VALUABLE_OPS else 0
            print(f"   ★ TWO-TIER MODE: {_n_t1:,} Tier 1 calls, {_n_vops} valuable ops, "
                  f"min_reduction={fo_config.TIER1_MIN_REDUCTION:.3f}")
        print(f"   Tick interval: {tick_interval}s, Max wait: {max_wait_time}s (fixed QP window)")
        print(f"   Gap handling: {gap_mode}" + (f" (busy_fraction={gap_busy_fraction})" if active_windows else ""))
        if acw_mean_s > 0:
            print(f"   After-call work (ACW): mean={acw_mean_s:.0f}s (exponential, delays operator re-availability)")
        if fo_config.EXCLUDE_IMMINENT_FROM_SLATE:
            print(f"   ★ EXCLUDE_IMMINENT_FROM_SLATE: solver uses only status=available operators")
        else:
            print(f"   ★ Imminent window: {fo_config.IMMINENT_WINDOW_S:.0f}s — busy operators finishing "
                  f"within this window join solver pool")
        if hourly_occupancy:
            print(f"   Hourly occupancy: dynamic ({len(hourly_occupancy)} hours, range {min(hourly_occupancy.values()):.0%}-{max(hourly_occupancy.values()):.0%})")
        else:
            print(f"   Base occupancy: {base_occupancy:.0%} (static)")
        print()
        state.tick += 1

    # --- Main loop ---
    while state.current_time <= sim_end:
        state.tick += 1

        # --- GAP HANDLING: Check if we're in a QPlanner active window ---
        in_active_window = True
        if active_windows:
            in_active_window = _is_in_active_window(state.current_time, active_windows)

        if active_windows and not in_active_window:
            if not was_in_gap and verbose:
                print(f"  ⏸️  Entering gap at {state.current_time.strftime('%H:%M:%S')} — operators serving other strategy")
            available = state.get_available_operators()
            n_to_block = int(len(available) * gap_busy_fraction)
            for op in available[:n_to_block]:
                op.status = "busy"
                op.busy_until = state.current_time + timedelta(seconds=tick_interval * 2)
            was_in_gap = True
            state.release_finished_calls()

            # During gaps, FIFO routing still happens (telephony doesn't stop)
            if enable_fifo:
                state.remove_fifo_abandoned(fifo_max_wait=fifo_max_wait)
                if state.fifo_queue:
                    gap_avail = state.get_available_operators()
                    if operator_shifts and gap_avail:
                        gap_avail = [
                            op for op in gap_avail
                            if op.username in operator_shifts
                            and operator_shifts[op.username][0] <= state.current_time <= operator_shifts[op.username][1]
                        ]
                    if fifo_routing_fraction < 1.0 and gap_avail:
                        n_gap_fifo = max(1, int(len(gap_avail) * fifo_routing_fraction))
                        gap_avail = gap_avail[:n_gap_fifo]
                    if gap_avail:
                        state.route_fifo(gap_avail, score_lookup, fifo_latency, block_operators=fifo_block_operators)
            state.current_time += timedelta(seconds=tick_interval)
            continue

        # Transition: gap → active window
        if was_in_gap and in_active_window:
            if verbose:
                print(f"  ▶️  Resuming QPlanner at {state.current_time.strftime('%H:%M:%S')}")
            for op in state.operators.values():
                if op.status == "busy" and op.busy_until and op.busy_until <= state.current_time + timedelta(seconds=tick_interval):
                    op.status = "available"
                    op.busy_until = None
                    op.current_call_id = None
            # Gap-cleared QP queue calls → move to FIFO (telephony takes over)
            if state.queue:
                if enable_fifo:
                    state.fifo_queue.extend(state.queue)
                else:
                    for c in state.queue:
                        state.unserved_calls.append({
                            'call_id': c.call_id,
                            'reason': 'gap_cleared',
                            'wait_time': (state.current_time - c.arrival_time).total_seconds()
                        })
                state.queue = []
            was_in_gap = False

        # --- NORMAL SIMULATION LOGIC ---
        # 1. Release operators whose calls have finished
        released = state.release_finished_calls()

        # 2. Enqueue new calls that arrived since last tick
        new_calls = []
        while next_call_idx < total_calls:
            row = events.iloc[next_call_idx]
            if row['arrival_time'] <= state.current_time:
                churn_i = churn_i_lookup.get(row['CALL_ID'], row.get('churn_i', 0.5))
                new_calls.append(QueuedCall(
                    call_id=row['CALL_ID'],
                    customer_id=str(row['SA_COD']),
                    arrival_time=row['arrival_time'],
                    churn_i=float(churn_i),
                    call_duration=float(row['call_duration']),
                ))
                next_call_idx += 1
            else:
                break
        state.enqueue_calls(new_calls)

        #     ONE-TIME offer: when a call first reaches callback_deflection_time,
        #     it gets a single random check with callback_deflection_rate probability.
        #     probability of accepting the callback and leaving the QP queue.
        #     This models the real telephony IVR that offers "press 1 for callback"
        #     and explains why many unserved calls leave at ~55s in historical data.
        if callback_deflection_time > 0 and callback_deflection_rate > 0 and state.queue:
            deflected = []
            remaining = []
            for c in state.queue:
                wait = (state.current_time - c.arrival_time).total_seconds()
                # Only offer callback ONCE per call, when wait first crosses threshold
                if wait >= callback_deflection_time and c.call_id not in _callback_offered:
                    _callback_offered.add(c.call_id)
                    if nqp_rng.random() < callback_deflection_rate:
                        deflected.append(c)
                        continue
                remaining.append(c)
            if deflected:
                _callback_deflections += len(deflected)
                for c in deflected:
                    # Deflected calls go to FIFO (telephony callback = will be served later)
                    if enable_fifo:
                        state.fifo_queue.append(c)
                    else:
                        state.unserved_calls.append({
                            'call_id': c.call_id,
                            'reason': 'callback_deflection',
                            'wait_time': (state.current_time - c.arrival_time).total_seconds()
                        })
                state.queue = remaining
        # 3. Remove expired QP calls → move to FIFO queue (telephony takes over)
        expired = state.remove_expired(to_fifo=enable_fifo)

        # 3b. Remove FIFO calls where customer abandoned (total wait > fifo_max_wait)
        if enable_fifo:
            state.remove_fifo_abandoned(fifo_max_wait=fifo_max_wait)

        # 4. PRIORITY 1 — QPlanner: solve optimal assignments
        available_ops = state.get_available_operators()

        # Apply operator shift constraints (only operators currently on shift)
        if operator_shifts and available_ops:
            available_ops = [
                op for op in available_ops
                if op.username in operator_shifts
                and operator_shifts[op.username][0] <= state.current_time <= operator_shifts[op.username][1]
            ]

        # Apply occupancy: task-based non-QP work model
        current_hour = state.current_time.hour
        effective_occ = hourly_occupancy.get(current_hour, base_occupancy) if hourly_occupancy else base_occupancy
        if effective_occ > 0 and available_ops:
            mean_avail_s = nqp_mean_task_s * (1 - effective_occ) / max(effective_occ, 0.01)
            truly_available = []
            for op in available_ops:
                nqp_end = nqp_busy_until.get(op.username)
                if nqp_end and nqp_end > state.current_time:
                    continue
                p_start = min(0.95, tick_interval / max(mean_avail_s, 1.0))

                if nqp_rng.random() < p_start:
                    task_dur = max(float(tick_interval), nqp_rng.exponential(nqp_mean_task_s))
                    nqp_busy_until[op.username] = state.current_time + timedelta(seconds=task_dur)
                    continue

                truly_available.append(op)
            available_ops = truly_available

        # ── Include imminent operators (busy but about to free up) ──────
        # In production, operatorsavailability_api reports operators finishing
        # their current call soon as "imminent." The brains solver includes
        # them in the candidate pool. We replicate this by default.
        # S1 (EXCLUDE_IMMINENT_FROM_SLATE=True) restricts to truly-available only.
        if not fo_config.EXCLUDE_IMMINENT_FROM_SLATE and fo_config.IMMINENT_WINDOW_S > 0:
            imminent_ops = state.get_imminent_operators(fo_config.IMMINENT_WINDOW_S)
            # Apply shift constraints to imminent operators too
            if operator_shifts and imminent_ops:
                imminent_ops = [
                    op for op in imminent_ops
                    if op.username in operator_shifts
                    and operator_shifts[op.username][0] <= state.current_time <= operator_shifts[op.username][1]
                ]
            if imminent_ops:
                available_ops = available_ops + imminent_ops

        # Track score coverage before solver call
        if state.queue and available_ops:
            _tick_pairs = len(state.queue) * len(available_ops)
            _tick_hits = sum(
                1 for c in state.queue for op in available_ops
                if (c.call_id, op.username) in score_lookup
            )
            _total_solver_pairs += _tick_pairs
            _total_solver_hits += _tick_hits
            _tick_ops_log.append((len(available_ops), len(state.queue)))

        # QPlanner solver — two modes:
        #   Standard: KAPPA gate naturally rejects calls where FO ≤ 0
        #   Two-tier: split queue by call value, reserve operators for Tier 1
        if state.queue and available_ops:
            if _use_tiered:
                pairs = solve_tiered(
                    queue=state.queue,
                    available_ops=available_ops,
                    score_lookup=score_lookup,
                    churn_i_lookup=churn_i_lookup,
                    current_time=state.current_time,
                    fo_config=fo_config,
                    fallback_churn_ij=fallback_churn_ij,
                )
            else:
                pairs = solve_assignments(
                    queue=state.queue,
                    available_ops=available_ops,
                    score_lookup=score_lookup,
                    churn_i_lookup=churn_i_lookup,
                    current_time=state.current_time,
                    fo_config=fo_config,
                    fallback_churn_ij=fallback_churn_ij,
                )
            # With delivery_failure_rate > 0: each assignment may fail
            # (operator went busy during pipeline latency → AgentNotAvailable)
            # Failed calls → FIFO queue (same as production behavior)
            for call, op, churn_ij, score in pairs:
                # Extra wait for imminent operators (still finishing previous call)
                extra_wait = 0.0
                if op.busy_until and op.busy_until > state.current_time:
                    extra_wait = (op.busy_until - state.current_time).total_seconds()

                if pipeline_latency_std > 0 and pipeline_latency > 0:
                    mu = pipeline_latency
                    sigma = pipeline_latency_std
                    sigma2_ln = np.log(1 + (sigma / mu) ** 2)
                    mu_ln = np.log(mu) - sigma2_ln / 2
                    lat_sample = max(5.0, nqp_rng.lognormal(mu_ln, np.sqrt(sigma2_ln)))
                else:
                    lat_sample = pipeline_latency

                # Effective latency = pipeline + wait for imminent operator to free up
                effective_latency = lat_sample + extra_wait

                # Delivery failure check
                if delivery_failure_rate > 0 and nqp_rng.random() < delivery_failure_rate:
                    # Record failed delivery
                    state.failed_deliveries.append({
                        'call_id': call.call_id,
                        'operator': op.username,
                        'reason': 'delivery_failure',
                        'wait_time': (state.current_time - call.arrival_time).total_seconds(),
                        'score': score,
                    })
                    # Mark operator as busy for pipeline_latency period:
                    # In production, delivery failed because the operator WAS busy
                    # (AgentNotAvailable). Model this by blocking the operator briefly.
                    _fail_busy_s = max(lat_sample, nqp_rng.exponential(nqp_mean_task_s * 0.3))
                    _new_busy = state.current_time + timedelta(seconds=_fail_busy_s)
                    # For imminent operators: don't shorten their current call
                    if op.busy_until and op.busy_until > _new_busy:
                        pass  # keep existing busy_until (current call longer)
                    else:
                        op.status = "busy"
                        op.busy_until = _new_busy
                    # Move failed call to FIFO queue (telephony takes over)
                    if enable_fifo:
                        state.fifo_queue.append(call)
                    else:
                        state.unserved_calls.append({
                            'call_id': call.call_id,
                            'reason': 'delivery_failure',
                            'wait_time': (state.current_time - call.arrival_time).total_seconds()
                        })
                    # Remove call from QP queue (it's been handled, even if failed)
                    state.queue = [c for c in state.queue if c.call_id != call.call_id]
                    continue
                # Apply after-call work (ACW): operator stays busy after call ends
                acw_sample = nqp_rng.exponential(acw_mean_s) if acw_mean_s > 0 else 0.0
                state.execute_assignment(call, op, churn_ij, score,
                                         pipeline_latency=effective_latency,
                                         acw_time=acw_sample)

        # 6. PRIORITY 2 — FIFO telephony routing: remaining operators serve FIFO queue
        #    fifo_routing_fraction < 1.0 models ACD pacing: not all operators instantly
        #    get a telephony call, leaving a buffer for QP's next tick.
        if enable_fifo and state.fifo_queue:
            fifo_avail = state.get_available_operators()
            # Apply shift constraints to FIFO operators too
            if operator_shifts and fifo_avail:
                fifo_avail = [
                    op for op in fifo_avail
                    if op.username in operator_shifts
                    and operator_shifts[op.username][0] <= state.current_time <= operator_shifts[op.username][1]
                ]
            # ACD pacing: only a fraction of available operators serve FIFO each tick
            if fifo_routing_fraction < 1.0 and fifo_avail:
                n_fifo_ops = max(1, int(len(fifo_avail) * fifo_routing_fraction))
                fifo_avail = fifo_avail[:n_fifo_ops]
            if fifo_avail:
                state.route_fifo(fifo_avail, score_lookup, fifo_latency, block_operators=fifo_block_operators)

        # Log progress
        if verbose and state.tick % log_every == 0:
            n_busy = sum(1 for op in state.operators.values() if op.status == "busy")
            fifo_info = f" | fifo_q={len(state.fifo_queue):>3} | fifo_done={len(state.fifo_assignments):>4}" if enable_fifo else ""
            print(
                f"  tick={state.tick:>5} | time={state.current_time.strftime('%H:%M:%S')} | "
                f"queue={len(state.queue):>3} | assigned={len(state.assignments):>5} | "
                f"busy_ops={n_busy:>3}/{len(state.operators)}{fifo_info}"
            )
        state.current_time += timedelta(seconds=tick_interval)

        # Early exit if all calls processed and both queues empty
        if next_call_idx >= total_calls and not state.queue and not state.fifo_queue:
            break

    # Any calls still in FIFO queue at end → truly unserved
    for c in state.fifo_queue:
        state.unserved_calls.append({
            'call_id': c.call_id,
            'reason': 'fifo_end_of_day',
            'wait_time': (state.current_time - c.arrival_time).total_seconds()
        })
    state.fifo_queue = []

    # Store score coverage stats on state for aggregation
    state.score_coverage = {
        'total_pairs': _total_solver_pairs,
        'lookup_hits': _total_solver_hits,
        'fallback_used': _total_solver_pairs - _total_solver_hits,
        'coverage_pct': _total_solver_hits / _total_solver_pairs * 100 if _total_solver_pairs > 0 else 0,
    }
    state.tick_ops_log = _tick_ops_log  # (n_available_ops, n_queued_calls) per solver tick
    state.callback_deflections = _callback_deflections  # total calls deflected by callback offer

    if verbose:
        print(f"\n✅ Simulation complete!")
        print(f"   QP assignments:   {len(state.assignments)}")
        print(f"   Failed deliveries: {len(state.failed_deliveries)}")
        print(f"   Callback deflect.: {_callback_deflections}")
        print(f"   FIFO assignments: {len(state.fifo_assignments)}")
        print(f"   Total served:     {len(state.assignments) + len(state.fifo_assignments)}")
        print(f"   Truly unserved:   {len(state.unserved_calls)}")
        if state.unserved_calls:
            reasons = {}
            for u in state.unserved_calls:
                r = u.get('reason', 'unknown')
                reasons[r] = reasons.get(r, 0) + 1
            for r, cnt in sorted(reasons.items()):
                print(f"     ↳ {r}: {cnt}")
        if state.assignments:
            waits = [a.waiting_time for a in state.assignments]
            reductions = [a.churn_reduction for a in state.assignments]
            print(f"   Median wait:      {np.median(waits):.1f}s")
            print(f"   Mean Δchurn:      {np.mean(reductions):.4f}")
        if state.fifo_assignments:
            fifo_waits = [a.waiting_time for a in state.fifo_assignments]
            print(f"   FIFO avg wait: {np.mean(fifo_waits):.1f}s (median: {np.median(fifo_waits):.1f}s)")
        if _total_solver_pairs > 0:
            _fb = _total_solver_pairs - _total_solver_hits
            print(f"   Score coverage: {_total_solver_hits:,}/{_total_solver_pairs:,} "
                  f"({_total_solver_hits/_total_solver_pairs*100:.1f}%) | "
                  f"fallback: {_fb:,} pairs ({_fb/_total_solver_pairs*100:.1f}%)")
    return state

<a id="full-score-coverage"></a>
## Full Score Coverage: Panel + Model Re-scoring

The sparse `score_lookup` from brains historical data only covers ~13% of (call, operator) pairs — the solver only scored operators that happened to be available at each historic tick. In production, smartpairing creates a **cross join** of ALL queued calls × ALL available operators and runs `model.predict()` on every pair.

This section:
1. Loads **production feature panels** from BigQuery (customer + operator features at mid-January)
2. Loads the **smartpairing MLflow model** (`smartpairing_ebm_vanilla@champion`)
3. Scores **ALL** (call, operator) pairs per day → `full_score_lookup` (100% coverage)
4. Validates distributions against the original sparse lookup
5. **Overrides** `score_lookup` so the simulation uses full coverage
6. Computes a **fallback** (median churn_ij) for any remaining unscored pairs

In [ ]:
# =============================================================================
# CELL 12b — LOAD PRODUCTION PANELS + SCORE ALL PAIRS VIA MLFLOW MODEL
# =============================================================================
# INPUT:
#   - df_call_events, df_operators (see data prep outputs for counts)
#   - score_lookup (sparse pairs from brains history — see data prep output for count)
#   - churn_i_lookup (from cell 7)
#   - _day_setups OR df_calls_qplanner (for identifying per-day operators)
#
# DOES:
#   1. Queries BQ production panels (customer + operator features) for mid-January
#   2. Loads smartpairing_ebm_vanilla@champion model from MLflow
#   3. For EACH simulation day, builds a cross join of (calls × operators)
#   4. Enriches with panel features and runs model.predict()
#   5. Builds full_score_lookup (full cross-join coverage — see output for count)
#   6. Builds full_churn_i_lookup (ghost operator, all NaN op features)
#
# OUTPUT:
#   - full_score_lookup: dict (call_id, operator) → churn_ij (all pairs)
#   - full_churn_i_lookup: dict call_id → churn_i (no-operator baseline)
# =============================================================================
import os
from tqdm.auto import tqdm
from google.cloud import bigquery as _bq
# from crm_qp_lib.data_reader import impersonate_credentials  # internal — removed

_bq_creds = impersonate_credentials()
_bq_client = _bq.Client(credentials=_bq_creds)

# ─── 1. LOAD CUSTOMER PANEL ─────────────────────────────────────────────────
_PANEL_TABLE_CUST = "t_customers_panel"
_PANEL_TABLE_OPS  = "t_operators_panel_metrics"

# Find the latest valid partition date in January 2026
_partition_query = f"""
SELECT MAX(SAFE.PARSE_DATE('%Y%m%d', partition_id)) as latest_partition
FROM `YOUR_GCP_PROJECT.internal.INFORMATION_SCHEMA.PARTITIONS`
WHERE table_name = '{_PANEL_TABLE_CUST}'
  AND REGEXP_CONTAINS(partition_id, r'^[0-9]{{8}}$')
  AND SAFE.PARSE_DATE('%Y%m%d', partition_id) BETWEEN '2026-01-01' AND '2026-01-31'
"""
_part_df = _bq_client.query(_partition_query).to_dataframe()
_panel_date = str(_part_df['latest_partition'].iloc[0])
print(f"📅 Panel date: {_panel_date}")

# Load customer panel
_panel_cust_query = f"""
SELECT * FROM `YOUR_GCP_PROJECT.internal.{_PANEL_TABLE_CUST}`
WHERE AVAILABLE_TIME = '{_panel_date}'
"""
_df_panel_cust = _bq_client.query(_panel_cust_query).to_dataframe()
_df_panel_cust = _df_panel_cust.rename(columns={'SA_COD': 'customer_value_sa_cod'})
_cust_ids = set(df_call_events['SA_COD'].astype(str))
_panel_cust_ids = set(_df_panel_cust['customer_value_sa_cod'].astype(str))
_cust_coverage = len(_cust_ids & _panel_cust_ids) / len(_cust_ids) * 100

# Load operator panel
_panel_ops_query = f"""
SELECT * FROM `YOUR_GCP_PROJECT.internal.{_PANEL_TABLE_OPS}`
WHERE AVAILABLE_TIME = '{_panel_date}'
"""
_df_panel_ops = _bq_client.query(_panel_ops_query).to_dataframe()
_df_panel_ops = _df_panel_ops.rename(
    columns={'HIST_USER_COD': 'wrgo_ad_user_name'}
)
_df_panel_ops['wrgo_ad_user_name'] = _df_panel_ops['wrgo_ad_user_name'].str.lower()
_op_ids = set(df_operators['agent_username'].str.lower())
_panel_op_ids = set(_df_panel_ops['wrgo_ad_user_name'].str.lower())
_ops_coverage = len(_op_ids & _panel_op_ids) / len(_op_ids) * 100

print(f"📋 Customer panel: {len(_df_panel_cust):,} rows ({_cust_coverage:.1f}% of sim customers)")
print(f"📋 Operator panel: {len(_df_panel_ops):,} rows ({_ops_coverage:.1f}% of sim operators)")

# ─── 2. LOAD MLFLOW MODEL ───────────────────────────────────────────────────
# from crm_qp_lib.inference.mlflow import MLflowAuthenticator  # internal — removed
import mlflow as _unauthenticated_mlflow

# Clear cached auth so we get a fresh token for the right audience
if "CLUPA_LAST_MLFLOW_AUTH_TIME" in os.environ:
    del os.environ["CLUPA_LAST_MLFLOW_AUTH_TIME"]

_GCP_PROJECT = "YOUR_GCP_PROJECT"
_MLFLOW_URI  = "https://YOUR_MLFLOW_SERVER"

_auth_mlflow = MLflowAuthenticator(_GCP_PROJECT).authenticate_mlflow(_unauthenticated_mlflow)
_auth_mlflow.set_tracking_uri(_MLFLOW_URI)
os.environ["MLFLOW_TRACKING_URI"] = _MLFLOW_URI

_run_id = _auth_mlflow.MlflowClient().get_model_version_by_alias(
    "smartpairing_ebm_vanilla", "champion"
).run_id
_sp_model = _auth_mlflow.pyfunc.load_model(f"runs:/{_run_id}/pipeline")
_feats = list(
    _sp_model.unwrap_python_model().model.named_steps["pre_processing"]
    .steps[-1][1].column_order
)
print(f"🤖 Model loaded: smartpairing_ebm_vanilla@champion (run_id={_run_id[:12]}…)")
print(f"   Features ({len(_feats)}): {_feats[:5]}…")

# ─── 3. BUILD FEATURE COLUMN MAPPING (case-insensitive) ─────────────────────
_cust_feats = [f for f in _feats if not f.startswith('ohp_')]
_ops_feats  = [f for f in _feats if f.startswith('ohp_')]

# BQ columns are ALL UPPERCASE; model features have lowercase prefix + UPPERCASE suffix
_cust_col_map = {}
_bq_cust_cols_upper = {c.upper(): c for c in _df_panel_cust.columns}
for feat in _cust_feats:
    if feat.upper() in _bq_cust_cols_upper:
        _cust_col_map[_bq_cust_cols_upper[feat.upper()]] = feat

_ops_col_map = {}
_bq_ops_cols_upper = {c.upper(): c for c in _df_panel_ops.columns}
for feat in _ops_feats:
    if feat.upper() in _bq_ops_cols_upper:
        _ops_col_map[_bq_ops_cols_upper[feat.upper()]] = feat

print(f"   Customer features matched: {len(_cust_col_map)}/{len(_cust_feats)}")
print(f"   Operator features matched: {len(_ops_col_map)}/{len(_ops_feats)}")
assert len(_cust_col_map) == len(_cust_feats), f"Missing customer features!"
assert len(_ops_col_map) == len(_ops_feats), f"Missing operator features!"

# Build panel subsets with renamed columns
_panel_c = _df_panel_cust[['customer_value_sa_cod'] + list(_cust_col_map.keys())].copy()
_panel_c = _panel_c.rename(columns=_cust_col_map)
_panel_c['customer_value_sa_cod'] = _panel_c['customer_value_sa_cod'].astype(str)

_panel_o = _df_panel_ops[['wrgo_ad_user_name'] + list(_ops_col_map.keys())].copy()
_panel_o = _panel_o.rename(columns=_ops_col_map)
# Keep original case — must match agent_username from get_day_operators

# ─── 4. SCORE ALL PAIRS PER DAY ─────────────────────────────────────────────
# Pre-compute per-day setups (same as cell 13 will do)
_tmp_qp_ts = df_calls_qplanner.copy()
_tmp_qp_ts['_date'] = pd.to_datetime(_tmp_qp_ts['START_DATE_TIME']).dt.date
_tmp_days = sorted(_tmp_qp_ts['_date'].unique())

full_score_lookup = {}
full_churn_i_lookup = {}

for day in tqdm(_tmp_days, desc="Scoring all pairs"):
    day_str = str(day)
    # Day's calls
    _day_events = df_call_events[df_call_events['arrival_time'].dt.date == day]
    if _day_events.empty:
        continue
    # Day's operators
    _day_ops_df, _ = get_day_operators(df_calls_qplanner, day_str)
    if _day_ops_df.empty:
        continue

    # Cross join — preserve original agent_username case (same as simulation uses)
    _calls = _day_events[['CALL_ID', 'SA_COD']].copy()
    _calls['SA_COD'] = _calls['SA_COD'].astype(str)
    _calls['_key'] = 1
    _ops = _day_ops_df[['agent_username']].copy()
    _ops['_key'] = 1
    _cross = _calls.merge(_ops, on='_key').drop(columns='_key')
    _cross = _cross.rename(columns={'SA_COD': 'customer_value_sa_cod',
                                     'agent_username': 'wrgo_ad_user_name'})

    # Enrich with panel features
    _cross = _cross.merge(_panel_c, on='customer_value_sa_cod', how='left')
    _cross = _cross.merge(_panel_o, on='wrgo_ad_user_name', how='left')

    # Predict churn_ij
    _vals = _sp_model.predict(_cross[_feats])
    _keys = list(zip(_cross['CALL_ID'], _cross['wrgo_ad_user_name']))
    full_score_lookup.update(dict(zip(_keys, _vals)))

    # churn_i (ghost operator — all NaN operator features)
    _ci_df = _day_events[['CALL_ID', 'SA_COD']].drop_duplicates(subset='CALL_ID').copy()
    _ci_df['SA_COD'] = _ci_df['SA_COD'].astype(str)
    _ci_df = _ci_df.rename(columns={'SA_COD': 'customer_value_sa_cod'})
    _ci_df = _ci_df.merge(_panel_c, on='customer_value_sa_cod', how='left')
    for f in _ops_feats:
        _ci_df[f] = np.nan
    _ci_vals = _sp_model.predict(_ci_df[_feats])
    full_churn_i_lookup.update(dict(zip(_ci_df['CALL_ID'], _ci_vals)))

# ─── 5. REPORT ──────────────────────────────────────────────────────────────
_n_needed = sum(
    len(df_call_events[df_call_events['arrival_time'].dt.date == d]) *
    len(get_day_operators(df_calls_qplanner, str(d))[0])
    for d in _tmp_days
    if not df_call_events[df_call_events['arrival_time'].dt.date == d].empty
    and not get_day_operators(df_calls_qplanner, str(d))[0].empty
)
print(f"\n✅ Full score lookup built:")
print(f"   Pairs scored:    {len(full_score_lookup):,} / {_n_needed:,} ({len(full_score_lookup)/_n_needed*100:.1f}%)")
print(f"   churn_i entries: {len(full_churn_i_lookup):,}")
print(f"   Original sparse: {len(score_lookup):,} ({len(score_lookup)/len(full_score_lookup)*100:.1f}% of full)")

# Sample comparison
_common = set(score_lookup.keys()) & set(full_score_lookup.keys())
if _common:
    _sample_k = list(_common)[:5]
    print(f"\n   Sample comparison (orig → full):")
    for k in _sample_k:
        print(f"     {k[0][:20]}… × {k[1][:20]}…: {score_lookup[k]:.4f} → {full_score_lookup[k]:.4f}")

In [ ]:
# =============================================================================
# CELL 12d — OVERRIDE: Use full score lookup for simulation
# =============================================================================
# STRATEGY:
#   1. Replace score_lookup with full_score_lookup (100% of known pairs)
#   2. Merge churn_i_lookup: keep original brains values (day-accurate),
#      fill gaps from panel re-scoring
#   3. Compute fallback_churn_ij = median of full scores for ANY remaining
#      missing pairs at simulation time (customers/operators not in panel)
#
# FALLBACK CRITERIA:
#   Pairs missing from score_lookup at simulation time are rare edge cases:
#     - Customers not in the BQ panel (~6%)
#     - Operators not in the BQ panel (~21.8%)
#   For these, we assign the MEDIAN churn_ij from the full lookup.
#   Rationale: median is a "neutral" estimate — the pair is treated as
#   having average compatibility. This is conservative (no false signal)
#   and much better than the previous approach of marking them infeasible
#   (cost=1e6), which prevented the solver from considering those operators.
# =============================================================================

# Preserve sparse lookup for comparison in later diagnostic cells
sparse_score_lookup = score_lookup.copy()
sparse_churn_i_lookup = churn_i_lookup.copy()

# Override score_lookup with full coverage
score_lookup = full_score_lookup.copy()

# Merge churn_i: keep original (more accurate), add new entries from panel
for k, v in full_churn_i_lookup.items():
    if k not in churn_i_lookup:
        churn_i_lookup[k] = v

# Compute fallback for remaining missing pairs
fallback_churn_ij = float(np.median(list(score_lookup.values())))

# ── Per-day coverage report (matches what the simulation actually evaluates) ──
# The simulation evaluates calls × operators PER DAY, not globally.
# So coverage should be measured per-day, not all-calls × all-operators.
_tmp_qp_cov = df_calls_qplanner.copy()
_tmp_qp_cov['_date'] = pd.to_datetime(_tmp_qp_cov['START_DATE_TIME']).dt.date
_cov_days = sorted(_tmp_qp_cov['_date'].unique())

_total_day_pairs = 0
_total_day_hits = 0

print("=" * 80)
print("  SCORE LOOKUP OVERRIDE — Full Coverage Active")
print("=" * 80)
print(f"  score_lookup:      {len(score_lookup):,} entries  (was {len(sparse_score_lookup):,})")
print(f"  churn_i_lookup:    {len(churn_i_lookup):,} entries")
print(f"  fallback_churn_ij: {fallback_churn_ij:.4f}  (median of full lookup)")
print()
print(f"  {'Day':<14} {'Calls':>7} {'Ops':>5} {'Pairs':>10} {'Hits':>10} {'Miss':>7} {'Cov%':>7}")
print(f"  {'─'*54}")
for _d in _cov_days:
    _day_events = df_call_events[df_call_events['arrival_time'].dt.date == _d]
    _day_ops_df, _ = get_day_operators(df_calls_qplanner, str(_d))
    if _day_events.empty or _day_ops_df.empty:
        continue
    _day_calls = set(_day_events['CALL_ID'])
    _day_ops = set(_day_ops_df['agent_username'])
    _n_pairs = len(_day_calls) * len(_day_ops)
    _n_hits = sum(1 for c in _day_calls for o in _day_ops if (c, o) in score_lookup)
    _n_miss = _n_pairs - _n_hits
    _total_day_pairs += _n_pairs
    _total_day_hits += _n_hits
    _pct = _n_hits / _n_pairs * 100 if _n_pairs > 0 else 0
    print(f"  {str(_d):<14} {len(_day_calls):>7,} {len(_day_ops):>5} {_n_pairs:>10,} {_n_hits:>10,} {_n_miss:>7,} {_pct:>6.1f}%")

_total_miss = _total_day_pairs - _total_day_hits
_total_pct = _total_day_hits / _total_day_pairs * 100 if _total_day_pairs > 0 else 0
print(f"  {'─'*54}")
print(f"  {'TOTAL':<14} {'':>7} {'':>5} {_total_day_pairs:>10,} {_total_day_hits:>10,} {_total_miss:>7,} {_total_pct:>6.1f}%")
print()
print(f"  → At simulation time: {_total_miss:,} pairs will use fallback ({_total_day_pairs - _total_day_hits} / {_total_day_pairs:,})")
print(f"  Fallback criteria: median churn_ij = {fallback_churn_ij:.4f}")
print(f"  → Treats missing pairs as 'average compatibility' (neutral, conservative)")
print(f"  → Much better than cost=1e6 (infeasible), which excluded operators entirely")
print()
print(f"✅ score_lookup now uses FULL panel-rescored data")
print(f"   All downstream cells (simulation, experiments, diagnostics) will use it")

---
# Part III — Calibration & Analysis


<a id="calibrated-parameters"></a>
## Calibrated Parameters & Day Setup

In [ ]:
# =============================================================================
# CELL 13a — CALIBRATED PARAMETERS & DAY SETUP
# =============================================================================
# INPUT:
#   - DATA_TICK_MEDIAN, DATA_PIPELINE_MEAN, DATA_PIPELINE_STD (from cell 7)
#   - df_calls_qplanner, df_call_events (from cells 2-3)
#
# DOES:
#   1. Defines all calibrated simulation parameters (data-derived constants)
#   2. Identifies QPlanner days and the busiest day (see calls_per_day output)
#
# OUTPUT:
#   - CALIBRATED_* constants (used by calibration sweeps and simulation)
#   - df_calls_qplanner_ts, qp_days, busiest_day, calls_per_day
#
# NOTE: CALIBRATED_OCC_SCALE and CALIBRATED_CHURN_BIAS are derived by the
#   calibration cells that follow. Set _RUN_*_CALIBRATION = True to re-derive.
# =============================================================================

# ─── CALIBRATED PARAMETERS (all data-derived) ───────────────────────────────
#
# These parameters are derived from historical data and calibration procedures.
# Two parameters require special calibration runs (OCC_SCALE, CHURN_BIAS):
#
# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CALIBRATED_OCC_SCALE (occupancy scaling factor)                            │
# │ ─────────────────────────────────────────────────────────────────────────── │
# │ Derived from: OCC_SCALE sweep (see calibration cell below).                │
# │ Method: Coarse grid [0.5–4.0] + bisection refinement to find the value    │
# │   where simulated QP served calls ≈ historical QP served calls.           │
# │ Result: see CALIBRATED_OCC_SCALE below (update after running calibration).│
# │ Interpretation: Raw occupancy (1 - active_ops/total_ops) underestimates   │
# │   true operator busyness. Scaling by 1.27× corrects for tasks not visible │
# │   in the call data (admin, breaks, cross-team work).                      │
# │ Sensitivity: Sharp transition between 1.0 (+15%) and 1.5 (-15%).          │
# │ Re-calibrate when: operator pool or shift patterns change significantly.  │
# └─────────────────────────────────────────────────────────────────────────────┘
#
# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CALIBRATED_CHURN_BIAS (EBM model calibration correction)                   │
# │ ─────────────────────────────────────────────────────────────────────────── │
# │ Derived from: Churn Calibration Diagnostic (see calibration cell below).   │
# │ Method: Compare EBM model P(churn) predictions (full_score_lookup) vs      │
# │   actual RESOL_MOT_DSC outcomes on historical QP-assigned pairs.          │
# │ Result: see CALIBRATED_CHURN_BIAS below (update after running calibration)│
# │ Correction: Additive — subtract bias from all churn_ij values.            │
# │   Applied in validation (Cell 36) as: churn_ij_corrected = churn_ij - bias│
# │ Interpretation: The EBM model (smartpairing_ebm_vanilla@champion) has a   │
# │   systematic upward bias, likely from training data distribution shift.    │
# │ Brier score: reported by calibration cell output.                         │
# │ Re-calibrate when: EBM model is retrained or champion version changes.    │
# └─────────────────────────────────────────────────────────────────────────────┘
# ─────────────────────────────────────────────────────────────────────────────

# --- Calibration run guards (set True to re-run calibration sweeps) ---
_RUN_OCC_CALIBRATION = False     # ← Set True to re-run OCC_SCALE sweep (~20 min)
_RUN_CHURN_CALIBRATION = False   # ← Set True to re-run churn calibration

# --- Timing & latency (from data distributions) ---
CALIBRATED_TICK = int(round(DATA_TICK_MEDIAN))
CALIBRATED_LATENCY = int(round(DATA_PIPELINE_MEAN))
CALIBRATED_LATENCY_STD = int(round(DATA_PIPELINE_STD))
CALIBRATED_NQP_TASK_S = 1200

# --- Operator availability calibration (from OCC_SCALE sweep) ---
CALIBRATED_OCC_SCALE = 1.2656  # update after running OCC_SCALE calibration sweep

# --- Churn model calibration (from churn calibration diagnostic) ---
CALIBRATED_CHURN_BIAS = 0.0288  # update after running churn calibration

# --- FIFO routing ---
CALIBRATED_FIFO_LATENCY = 2.0  # telephony FIFO routing: ~2s (no ML, just ACD)
CALIBRATED_FIFO_BLOCK = False  # FIFO is non-blocking: churn estimation only, no operator hold

# --- Call flow adjustments (from historical data) ---
CALIBRATED_DELIVERY_FAILURE_RATE = 0.142  # from 890/6297 historical failed deliveries (AgentNotAvailable)
CALIBRATED_CALLBACK_DEFLECTION_TIME = 55.0  # seconds — telephony offers callback at ~55s wait
CALIBRATED_CALLBACK_DEFLECTION_RATE = 0.12   # fraction of queued calls that accept callback at deflection time
CALIBRATED_ACW_MEAN_S = 45.0  # mean after-call work time (seconds). Exponentially distributed. Typical: 30-60s.

print(f"📐 Calibrated parameters (data-derived):")
print(f"   tick_interval = {CALIBRATED_TICK}s (median request cadence)")
print(f"   pipeline_latency μ = {CALIBRATED_LATENCY}s, σ = {CALIBRATED_LATENCY_STD}s (from real overhead)")
print(f"   occ_scale = {CALIBRATED_OCC_SCALE} (from sweep → sim QP served ≈ historical)")
print(f"   churn_bias = {CALIBRATED_CHURN_BIAS} (EBM model overestimates by +{CALIBRATED_CHURN_BIAS*100:.2f}pp)")
print(f"   FIFO: latency={CALIBRATED_FIFO_LATENCY}s, block_operators={CALIBRATED_FIFO_BLOCK}")
print(f"   delivery_failure_rate = {CALIBRATED_DELIVERY_FAILURE_RATE:.1%} (from historical AgentNotAvailable)")
print(f"   acw_mean = {CALIBRATED_ACW_MEAN_S:.0f}s (after-call work — delays operator re-availability)")
print(f"   max_wait_time = 180s (fixed QP window, agreed with operations)\n")

# --- Identify all QPlanner days ---
df_calls_qplanner_ts = df_calls_qplanner.copy()
df_calls_qplanner_ts['_date'] = pd.to_datetime(df_calls_qplanner_ts['START_DATE_TIME']).dt.date
qp_days = sorted(df_calls_qplanner_ts['_date'].unique())
calls_per_day = df_call_events.groupby(df_call_events['arrival_time'].dt.date).size()
busiest_day = str(calls_per_day.idxmax())
print(f"📅 Found {len(qp_days)} QPlanner days: {qp_days[0]} → {qp_days[-1]}")
print(f"   Busiest day: {busiest_day} ({calls_per_day.max()} calls)")

<a id="two-tier-feasibility"></a>
### Two-Tier Feasibility Analysis — Slate-Based Call Segmentation

**Purpose:** Determine whether a two-tier routing strategy is viable by segmenting calls entirely from slate properties (`max_cij`, `min_cij`, `spread`) — avoiding the biased `churn_i` baseline (F5).

**Key questions:**
1. Which calls are **Tier 1** (high worst-case churn AND high operator differentiation)?
2. Which operators are **"valuable"** for Tier 1 (genuinely low `churn_ij` on high-risk calls)?
3. How tight should the **slate-based gate** be (fraction of spread)?
4. What is the **operator gap** between best and worst on Tier 1 calls?

**Outputs for Scenario Synthesis (Cell 22):**
- `tier1_calls` — set of call IDs classified as Tier 1
- `df_ops` — operator stats on Tier 1 performance
- `MAX_CIJ_THRESH`, `SPREAD_THRESH` — tier boundary thresholds

In [ ]:
# =============================================================================
# FEASIBILITY: Two-Tier Strategy — Fully Slate-Based Call Segmentation
# =============================================================================
# Q1. Which calls are Tier 1?  (high worst-case AND high operator spread)
# Q2. Which operators are "valuable" for Tier 1?  (give them low churn_ij)
# Q3. How tight should the slate-based gate be?
# Q4. How many calls/operators fall in each tier?
#
# AXES & DECISION: All based on the slate (max_cij, min_cij, spread).
#   max_cij = worst-case churn (wrong operator) — "how bad can it get?"
#   min_cij = best-case churn (right operator)  — "how good can we make it?"
#   spread  = max_cij - min_cij                 — "how much does matching matter?"
#
#   churn_i is NOT used because it's a model artifact (nulled operator vars)
#   that produces negative churn_i - churn_ij for ~70% of pairs.
#
#  EXPORT: figures/diag-two-tier-segmentation.png
# =============================================================================

from collections import defaultdict
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

# ─── Build per-call profiles from score_lookup ────────────────────────────────
cij_by_call = defaultdict(list)
ops_by_call = defaultdict(list)
for (cid, op), cij in score_lookup.items():
    cij_by_call[cid].append(cij)
    ops_by_call[cid].append(op)

# ─── Build per-call slate composition (available vs imminent) ─────────────────
# df_brains_valid has agent_status per (call_id, agent_username) pair
_status_pairs = df_brains_valid[['call_id', 'agent_username', 'agent_status']].drop_duplicates(
    subset=['call_id', 'agent_username'], keep='first'
)
_call_statuses = _status_pairs.groupby('call_id')['agent_status'].agg(set)
_slate_type = {}
for cid, statuses in _call_statuses.items():
    has_avail = 'available' in statuses
    has_imm = 'imminent' in statuses
    if has_avail and has_imm:
        _slate_type[cid] = 'mixed'
    elif has_avail:
        _slate_type[cid] = 'available-only'
    else:
        _slate_type[cid] = 'imminent-only'

# Also count available vs imminent per call for richer stats
_avail_count = (
    _status_pairs[_status_pairs['agent_status'] == 'available']
    .groupby('call_id').size()
)
_imm_count = (
    _status_pairs[_status_pairs['agent_status'] == 'imminent']
    .groupby('call_id').size()
)

call_profiles = {}
for cid, cij_vals in cij_by_call.items():
    ci = churn_i_lookup.get(cid)
    if ci is None or len(cij_vals) < 5:
        continue
    vals = np.array(cij_vals)
    call_profiles[cid] = {
        'churn_i': ci,
        'min_cij': vals.min(),
        'max_cij': vals.max(),
        'mean_cij': vals.mean(),
        'spread': vals.max() - vals.min(),
        'n_ops': len(vals),
        'slate_type': _slate_type.get(cid, 'unknown'),
        'n_available': int(_avail_count.get(cid, 0)),
        'n_imminent': int(_imm_count.get(cid, 0)),
    }

df_prof = pd.DataFrame(call_profiles.values(), index=call_profiles.keys())
df_prof.index.name = 'call_id'
df_prof['pct_imminent'] = df_prof['n_imminent'] / (df_prof['n_available'] + df_prof['n_imminent']) * 100

print(f"📊 CALL PROFILES: {len(df_prof):,} calls with ≥5 operators scored\n")

# ─── Slate composition summary ───────────────────────────────────────────────
_comp = df_prof['slate_type'].value_counts()
print("─── SLATE COMPOSITION (available vs imminent operators) ───")
for st in ['available-only', 'mixed', 'imminent-only']:
    n = _comp.get(st, 0)
    print(f"  {st:<18s}: {n:>6,} calls ({n/len(df_prof)*100:5.1f}%)")

_mixed = df_prof[df_prof['slate_type'] == 'mixed']
if len(_mixed) > 0:
    print(f"\n  Mixed slates — imminent fraction:")
    print(f"    Mean:   {_mixed['pct_imminent'].mean():.1f}% imminent")
    print(f"    Median: {_mixed['pct_imminent'].median():.1f}% imminent")
    print(f"    P75:    {_mixed['pct_imminent'].quantile(0.75):.1f}% imminent")
    print(f"    P90:    {_mixed['pct_imminent'].quantile(0.90):.1f}% imminent")

# ─── Q1: Distribution of slate properties ────────────────────────────────────
print("\n─── WORST-CASE CHURN (max_cij) — wrong operator ───")
for pct in [25, 50, 75, 90, 95]:
    print(f"  P{pct}: {df_prof['max_cij'].quantile(pct/100):.4f}")

print("\n─── BEST-CASE CHURN (min_cij) — right operator ───")
for pct in [25, 50, 75, 90, 95]:
    print(f"  P{pct}: {df_prof['min_cij'].quantile(pct/100):.4f}")

print("\n─── OPERATOR SPREAD (max_cij - min_cij) ───")
for pct in [25, 50, 75, 90, 95]:
    print(f"  P{pct}: {df_prof['spread'].quantile(pct/100):.4f}")

_corr = df_prof[['churn_i', 'max_cij']].corr().iloc[0, 1]
_mean_gap = (df_prof['max_cij'] - df_prof['churn_i']).mean()
print(f"\n─── ⚠️ churn_i vs max_cij ───")
print(f"  Correlation: {_corr:.3f}")
print(f"  Mean gap (max_cij - churn_i): {_mean_gap:+.4f}")
print(f"  → max_cij is systematically higher than churn_i (artifact)")
print(f"  → using max_cij (actual worst operator) instead of churn_i (biased)")

# ─── Q2: Tier 1 sizing (fully slate-based) ───────────────────────────────────
print(f"\n{'='*70}")
print(f"  TIER 1 SIZING: calls where max_cij > X AND spread > Y")
print(f"{'='*70}")
print(f"  {'max_cij >':<12s} {'spread >':<14s} {'N calls':<10s} {'% total':<10s} {'mean spread':<12s} {'mean min_cij':<14s}")
print(f"  {'-'*72}")

for max_thresh in [0.40, 0.45, 0.50, 0.55]:
    for sp_thresh in [0.05, 0.08, 0.10, 0.15, 0.20]:
        mask = (df_prof['max_cij'] > max_thresh) & (df_prof['spread'] > sp_thresh)
        n = mask.sum()
        pct = n / len(df_prof) * 100
        if n > 0:
            mean_sp = df_prof.loc[mask, 'spread'].mean()
            mean_min = df_prof.loc[mask, 'min_cij'].mean()
            print(f"  {max_thresh:<12.2f} {sp_thresh:<14.2f} {n:<10,} {pct:<10.1f} {mean_sp:<12.4f} {mean_min:<14.4f}")

# ─── Q3: Operator value segmentation ─────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  OPERATOR VALUE SEGMENTATION")
print(f"{'='*70}")

MAX_CIJ_THRESH = 0.50
SPREAD_THRESH = 0.10
tier1_calls = set(df_prof[(df_prof['max_cij'] > MAX_CIJ_THRESH) &
                           (df_prof['spread'] > SPREAD_THRESH)].index)

op_profiles = defaultdict(list)
for (cid, op), cij in score_lookup.items():
    if cid in tier1_calls:
        op_profiles[op].append(cij)

op_stats = {}
for op, cij_vals in op_profiles.items():
    vals = np.array(cij_vals)
    op_stats[op] = {
        'mean_cij_tier1': vals.mean(),
        'n_tier1_calls': len(vals),
    }

df_ops = pd.DataFrame(op_stats.values(), index=op_stats.keys())
df_ops = df_ops.sort_values('mean_cij_tier1')

print(f"\n  Tier 1 definition: max_cij > {MAX_CIJ_THRESH}, spread > {SPREAD_THRESH}")
print(f"  Tier 1 calls: {len(tier1_calls):,}  ({len(tier1_calls)/len(df_prof)*100:.1f}%)")
print(f"  Operators with Tier 1 exposure: {len(df_ops):,}")

print(f"\n  TOP-20 VALUABLE operators (lowest mean churn_ij on Tier 1):")
for i, (op, row) in enumerate(df_ops.head(20).iterrows()):
    print(f"    {i+1:>3}. {op:<30s} mean_cij={row['mean_cij_tier1']:.4f}  (n={int(row['n_tier1_calls'])})")

print(f"\n  BOTTOM-20 operators (highest mean churn_ij on Tier 1):")
for i, (op, row) in enumerate(df_ops.tail(20).iterrows()):
    print(f"    {i+1:>3}. {op:<30s} mean_cij={row['mean_cij_tier1']:.4f}  (n={int(row['n_tier1_calls'])})")

best_20_mean = df_ops.head(20)['mean_cij_tier1'].mean()
worst_20_mean = df_ops.tail(20)['mean_cij_tier1'].mean()
all_mean = df_ops['mean_cij_tier1'].mean()
print(f"\n  Tier 1 operator gap:")
print(f"    Best-20 avg cij:  {best_20_mean:.4f}")
print(f"    Worst-20 avg cij: {worst_20_mean:.4f}")
print(f"    Overall avg cij:  {all_mean:.4f}")
print(f"    Gap: {worst_20_mean - best_20_mean:.4f} ({(worst_20_mean - best_20_mean)*100:.1f}pp)")

# ─── Q4: Scatter: max_cij vs min_cij — shape = slate composition ─────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6),
                         gridspec_kw={'width_ratios': [1.2, 0.05, 0.8]})
ax = axes[0]
cbar_ax = axes[1]

tier1_mask = (df_prof['max_cij'] > MAX_CIJ_THRESH) & (df_prof['spread'] > SPREAD_THRESH)

# Color ALL dots by spread
_norm = Normalize(vmin=0, vmax=df_prof['spread'].max())
_cmap = plt.cm.RdYlGn_r

# Marker shapes by slate composition
_shapes = {
    'available-only': 'o',    # circle
    'mixed':          'D',    # diamond
    'imminent-only':  '^',    # triangle up
}
_shape_labels = {
    'available-only': 'Available only',
    'mixed':          'Mixed (avail + imminent)',
    'imminent-only':  'Imminent only',
}

# Plot each (tier, slate_type) combo separately so we get distinct shapes
for is_t1, tier_label in [(False, 'T2'), (True, 'T1')]:
    _tier_mask = tier1_mask if is_t1 else ~tier1_mask
    _alpha = 0.65 if is_t1 else 0.20
    _size = 20 if is_t1 else 8
    _edge = '#333333' if is_t1 else 'none'
    _lw = 0.3 if is_t1 else 0

    for stype, marker in _shapes.items():
        _m = _tier_mask & (df_prof['slate_type'] == stype)
        if _m.sum() == 0:
            continue
        ax.scatter(
            df_prof.loc[_m, 'max_cij'], df_prof.loc[_m, 'min_cij'],
            c=df_prof.loc[_m, 'spread'], cmap=_cmap, norm=_norm,
            alpha=_alpha, s=_size, marker=marker,
            edgecolors=_edge, linewidths=_lw,
        )

# Diagonal = zero spread
_lim = [0, df_prof[['min_cij', 'max_cij']].max().max() * 1.02]
ax.plot(_lim, _lim, 'k--', alpha=0.4, lw=1)
ax.text(_lim[1]*0.55, _lim[1]*0.58, 'spread = 0\n(operator irrelevant)',
        fontsize=7, color='#555', rotation=38, ha='left', style='italic')

# T1 gate line
ax.axvline(x=MAX_CIJ_THRESH, color='#1a5276', ls=':', alpha=0.7, lw=1.5)
ax.text(MAX_CIJ_THRESH + 0.01, _lim[1]*0.02,
        f'max_cij = {MAX_CIJ_THRESH}', fontsize=7, color='#1a5276', rotation=90, va='bottom')

# Annotations
_t1 = tier1_mask
_t2 = ~tier1_mask
_t1_x = df_prof.loc[_t1, 'max_cij'].median()
_t1_y = df_prof.loc[_t1, 'min_cij'].median()
ax.annotate(f'Tier 1\n{_t1.sum():,} calls ({_t1.sum()/len(df_prof)*100:.0f}%)',
            xy=(_t1_x, _t1_y),
            xytext=(_t1_x + 0.05, _t1_y - 0.12),
            fontsize=9, fontweight='bold', color='#922b21',
            arrowprops=dict(arrowstyle='->', color='#922b21', lw=1.2),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#922b21', alpha=0.9))

ax.annotate(f'Tier 2\n{_t2.sum():,} calls',
            xy=(0.25, 0.10),
            fontsize=9, fontweight='bold', color='#666',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#999', alpha=0.8))

# Build custom legend for shapes
_legend_handles = [
    Line2D([0], [0], marker=mk, color='none', markerfacecolor='#888',
           markeredgecolor='#333', markersize=7, label=_shape_labels[st])
    for st, mk in _shapes.items()
    if st in df_prof['slate_type'].values
]
ax.legend(handles=_legend_handles, fontsize=7.5, loc='upper left',
          title='Slate composition', title_fontsize=8)

ax.set_xlabel('max_cij  (worst-case: wrong operator)', fontsize=10)
ax.set_ylabel('min_cij  (best-case: right operator)', fontsize=10)
ax.set_title('Two-Tier Call Segmentation\n'
             'color = spread \u00b7 shape = slate composition', fontsize=11, fontweight='bold')
ax.set_xlim(_lim)
ax.set_ylim(_lim)

# Colorbar
cb = fig.colorbar(ScalarMappable(norm=_norm, cmap=_cmap), cax=cbar_ax)
cb.set_label('Spread  (max \u2212 min cij)', fontsize=9)
cb.ax.tick_params(labelsize=8)

# Right panel: operator histogram
ax = axes[2]
ax.hist(df_ops['mean_cij_tier1'].values, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(x=best_20_mean, color='green', ls='--', lw=2, label=f'Best-20 avg={best_20_mean:.3f}')
ax.axvline(x=worst_20_mean, color='red', ls='--', lw=2, label=f'Worst-20 avg={worst_20_mean:.3f}')
ax.set_xlabel('Mean churn_ij on Tier 1 calls', fontsize=10)
ax.set_ylabel('# Operators', fontsize=10)
ax.set_title('Operator Value Distribution\n(Tier 1 calls)', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout()

# ── Standalone export ────────────────────────────────────────────────────────
from pathlib import Path as _Path
_fig_dir = _Path('figures')
_fig_dir.mkdir(exist_ok=True)
fig.savefig(_fig_dir / 'diag-two-tier-segmentation.png', dpi=150, bbox_inches='tight')
print(f"\n✅ Saved {_fig_dir / 'diag-two-tier-segmentation.png'}")

plt.show()

# ─── Tier 1 slate composition breakdown ──────────────────────────────────────
_t1_comp = df_prof.loc[tier1_mask, 'slate_type'].value_counts()
print(f"\n  Tier 1 slate composition:")
for st in ['available-only', 'mixed', 'imminent-only']:
    n = _t1_comp.get(st, 0)
    print(f"    {st:<18s}: {n:>5,} ({n/_t1.sum()*100:5.1f}%)")

# ─── KEY QUESTION: How tight should the slate-based gate be? ──────────────────
print(f"\n{'='*70}")
print(f"  SLATE-BASED GATE ANALYSIS")
print(f"{'='*70}")
print(f"  Gate: (max_cij \u2212 cij) \u2265 frac \u00d7 spread  (op must capture frac of range)")
print(f"  For each Tier 1 call, how many operators pass the gate?\n")

t1_call_ids = df_prof.index[tier1_mask]
for frac in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    ops_passing = []
    for cid in t1_call_ids:
        vals = np.array(cij_by_call[cid])
        spread = vals.max() - vals.min()
        if spread < 1e-6:
            ops_passing.append(0)
            continue
        threshold = frac * spread
        n_pass = (vals <= vals.max() - threshold).sum()
        ops_passing.append(n_pass)
    ops_arr = np.array(ops_passing)
    print(f"  LIFT_FRAC={frac:.0%}:  avg {ops_arr.mean():.1f} ops/call  |  "
          f"P25={np.percentile(ops_arr, 25):.0f}  P50={np.percentile(ops_arr, 50):.0f}  "
          f"P75={np.percentile(ops_arr, 75):.0f}  |  "
          f"0 ops: {(ops_arr == 0).sum():,}")

print(f"\n📌 FULLY SLATE-BASED PIPELINE:")
print(f"   1. T1 SELECTION: max_cij > {MAX_CIJ_THRESH} AND spread > {SPREAD_THRESH}")
print(f"      → no churn_i dependency. max_cij = actual worst operator outcome.")
print(f"   2. T1 GATE (solve_tiered): (max_cij_slate \u2212 cij) \u2265 LIFT_FRAC \u00d7 spread")
print(f"      → operator must capture \u2265 X% of the slate quality range.")
print(f"   3. VALUABLE OPS: operators with below-median mean_cij on T1 calls.")
print(f"   Everything is relative to actual model predictions, never to churn_i.")

<a id="valuable-operators"></a>
### Compute Valuable Operators for Two-Tier Routing

Uses `df_ops` (operator-level Tier 1 stats from Cell 22a) to identify the **"valuable" operator set** — operators whose mean `churn_ij` on Tier 1 calls is below median (i.e., genuinely good at saving high-risk customers).

**Outputs consumed by Scenario Synthesis:**
- `valuable_ops` — set of operator IDs reserved for Tier 1 in S8/S9/S10

In [ ]:
# ==========================================================================
# COMPUTE VALUABLE OPERATORS FOR TWO-TIER ROUTING (S8)
# ==========================================================================
# Uses op_profiles from the feasibility analysis cell.
# "Valuable" = operators whose mean churn_ij on Tier 1 calls is BELOW median
#              (i.e., they are genuinely good at saving high-risk customers).
# Only operators with ≥10 Tier 1 exposures are considered (statistical stability).
# ==========================================================================

MIN_T1_EXPOSURE = 10  # minimum Tier 1 calls scored to qualify

# df_ops already computed in feasibility cell: sorted by mean_cij_tier1
_qualified = df_ops[df_ops['n_tier1_calls'] >= MIN_T1_EXPOSURE].copy()
_median_cij = _qualified['mean_cij_tier1'].median()
_valuable_mask = _qualified['mean_cij_tier1'] <= _median_cij
valuable_ops = set(_qualified.index[_valuable_mask])

print(f"═══ VALUABLE OPERATOR SET FOR S8 ═══")
print(f"  Qualified operators (≥{MIN_T1_EXPOSURE} Tier 1 exposures): {len(_qualified)}")
print(f"  Median mean_cij on Tier 1: {_median_cij:.4f}")
print(f"  Valuable operators (≤ median): {len(valuable_ops)}")
print(f"  Non-valuable operators:        {len(_qualified) - len(valuable_ops)}")
print(f"\n  Valuable avg cij:     {_qualified.loc[_valuable_mask, 'mean_cij_tier1'].mean():.4f}")
print(f"  Non-valuable avg cij: {_qualified.loc[~_valuable_mask, 'mean_cij_tier1'].mean():.4f}")
print(f"  Gap: {_qualified.loc[~_valuable_mask, 'mean_cij_tier1'].mean() - _qualified.loc[_valuable_mask, 'mean_cij_tier1'].mean():.4f}")
print(f"\n✅ `valuable_ops` set ready ({len(valuable_ops)} operators)")


---
# Part IV — Scenario Framework


<a id="scenario-synthesis"></a>
## Scenario Synthesis — Evidence-Driven Experimentation Roadmap

### 1. What We Know: Diagnostic Evidence

The following findings come exclusively from analysing **historical operational data and model outputs** (Cells 1–22b). No simulation experiments have been run yet — the scenarios below are hypotheses to test.

| # | Finding | Evidence | Cell |
|---|---------|----------|------|
| F1 | **Most assignments had a better operator available.** Regret decomposes into routing regret (wrong available operator picked) and imminent premium (best operator was busy). See regret analysis output for exact split. | Historical assignment analysis, regret = actual_cij − best_cij | Cell 19 (Regret Analysis) |
| F2 | **Operator park is distributed/specialist.** No single elite group captures most "best-for" assignments. The right specialist for the right customer matters — routing intelligence is the primary lever, not capacity. | Best-operator concentration analysis, generalist vs specialist classification | Cell 20 (Operator Profiling) |
| F3 | **A significant fraction of calls are unserved.** Breakdown: majority NoAgentReturned (FO ≤ 0, gate too strict), remainder AgentNotAvailable (race condition). See unserved breakdown output for exact split. | Historical unserved breakdown by reason code | Cell 16 (Unserved Diagnostic) |
| F4 | **Quality degrades with wait.** Most ΔChurn harvest is captured by ~185s. Assigning sooner produces better outcomes. | Cumulative ΔChurn harvest curve | Cell 18 (Decision Support) |
| F5 | **churn_i asks the wrong question.** The model was trained to answer P(churn \| call_i, operator_j) — and it does so correctly. No call happens in the void: there is always an operator who picks up. Production simplifies by nulling operator features to obtain churn_i = P(churn \| call_i) — a question the model was never designed to answer. The result (majority negative lift) is not an S-learner artifact but the predictable consequence of asking an out-of-distribution question. The correct baseline is always slate-relative: compare against max/min churn_ij in the available slate. | Lift distribution, churn_i vs churn_ij scatter | Cell 22b (Baseline Model Bias) |
| F6 | **T1 calls** with max_cij > 0.50 AND spread > 0.10 (see two-tier feasibility output for count). These are calls where operator choice matters (high risk + high differentiation), identified using a fully slate-based criterion (no churn_i dependency). | Scatter: max_cij vs min_cij with spread colour gradient | Cell 22a (Two-Tier Feasibility) |
| F7 | **Meaningful operator gap on T1.** Best-20 vs worst-20 operators show a sizeable avg churn_ij gap on T1 calls. See operator gap output for exact values. | Operator value distribution histogram | Cell 22a (Two-Tier Feasibility) |
| F8 | **Most T1 slates contain imminent operators** (mixed, imminent-only, and available-only; see slate composition output for breakdown). The spread and operator gap numbers (F6, F7) are partly shaped by operators that production can never select. | Slate composition analysis with shape markers | Cell 22a (Slate Composition) |
| F9 | **Gate timing is the main volume lever.** κ = −0.20 shifts gate opening earlier; α = 1.5 shifts it further via a gentler P curve. Both increase eligible call volume without changing FO formula. | Sensitivity analysis: κ sweep + α sweep on gate crossing point | Cell 17 (FO Analysis) |
| F10 | **G composition dominates at short waits.** At wait = 50s, FO is mostly G (churn reduction); at 150s, mostly P (time urgency). Early assignments are quality-driven, late ones are urgency-driven. | FO composition over wait time curve | Cell 17 (FO Analysis) |
| F11 | **Imminent betting has a minority share of the regret.** Most regret comes from selecting the wrong available operator. Value of imminent = real but secondary to routing improvement. Some calls had ONLY imminent operators scored — can only be saved by betting. | Regret decomposition: routing vs imminent premium | Cell 21 (Imminent Betting) |

---

### 2. Scenario Roadmap — Layered Experimentation

The plan is organized in **dependency layers**: each layer builds on the validated outputs of the previous one. Within each layer, scenarios are independent and can run in parallel.

---

#### Layer 0 — Establish Baselines

> **Rationale:** We need a production-faithful baseline AND a slate-based alternative to compare against. churn_i asks the model a question it was never trained to answer (F5), so we test both the current gate and a corrected slate-relative one that uses the model as designed.

| Scenario | Description | Config Change | Evidence / Hypothesis |
|----------|-------------|---------------|----------------------|
| **S0 Production Baseline** | Current production FO: β_G=0.35, β_P=0.65, κ=−0.304, α=3.0. Reference point for all comparisons. | `FOConfig()` (defaults) | Baseline — no hypothesis |
| **S0' Slate Baseline** | Same production FO formula (β_G, β_P, κ, α unchanged), but the `g_value` numerator uses `max(churn_ij in tick slate)` instead of `churn_i` as the baseline for churn reduction. The sigmoid gate `c = sigmoid(K·churn_i − τ)` still uses `churn_i` for risk-based prioritization. Anchors the reduction to the worst operator actually available — always a real model output, not an out-of-distribution query. | `USE_SLATE_G=True` | F5 (churn_i asks the wrong question — model was never trained to answer P(churn \| call) without an operator) |

**What S0' changes:** Only the `g_value` numerator in the FO formula. Production computes `g_value = (churn_i − churn_ij) / (|churn_i| + |churn_ij|)` where `churn_i` is an out-of-distribution artifact. Slate G replaces `churn_i` with `max(churn_ij)` in the g_value calculation only — the worst operator the customer could be assigned to in the current tick's slate. The sigmoid gate `c = sigmoid(K·churn_i − τ)` still uses `churn_i` for customer risk prioritization (churn_i provides reasonable risk ranking even if it's not calibrated). Same formula, same gate, same P component — only the reduction anchor changes. This uses the model as designed for the reduction measurement (always conditioning on both call and operator).

---

#### Layer 1 — Available-Only Scoring

> **Rationale:** Production scores each call against all operators in the slate — including ~39% that are `imminent` (busy, expected to become free soon). Analysis shows that production **never recommends** imminent operators (0% of assignments), yet their churn_ij scores inflate slate statistics (spread, max_cij) used for T1 segmentation (F8; see output for composition breakdown). S1 tests the simplest possible fix: remove imminent operators from the scoring slate entirely, so the solver only sees operators it can actually assign.

| Scenario | Description | Config Change | Evidence / Hypothesis |
|----------|-------------|---------------|----------------------|
| **S1 Available-Only** | Score calls against available operators only. Removes ~92% of score_lookup pairs (imminent noise). The solver sees a cleaner, smaller slate that reflects what it can actually do. | `EXCLUDE_IMMINENT_FROM_SLATE=True` | F8 (most T1 have imminent ops), F11 (routing regret dominates imminent premium) |

**Decision Gate:** If available-only spread ≥ 8pp → the operator gap remains meaningful despite smaller slates, proceed to Layer 2. If spread collapses → T1 thresholds need adjustment or structural changes are needed before proceeding.

---

#### Layer 2 — FO Parameter Sweep

> **Rationale:** The simplest parameter-only experiments. β_G, κ, α sweeps within the two-tier structure test what the current FO formula can achieve by varying its weights. Fast to simulate, no structural changes needed — provides the ceiling on parameter tuning before investing in complex structural modifications (Layer 3). FO weight changes may no longer be purely redistributive within the two-tier structure, because T1 and T2 have partially separated operator pools.

| Scenario | Description | Config Change | Evidence / Hypothesis |
|----------|-------------|---------------|----------------------|
| **S2a β_G Sweep (T1)** | Within the two-tier structure, sweep β_G ∈ {0.35, 0.50, 0.65, 0.80} for T1 scoring. | `BETA_G` sweep (T1 only) | F1 (high routing regret), F2 (specialist park) — routing intelligence matters |
| **S2b κ Sweep** | Sweep κ ∈ {−0.30, −0.25, −0.20, −0.15}. Determines gate timing for T2 calls within the two-tier world. | `KAPPA` sweep | F3 (high unserved rate), F9 (κ shifts gate opening) |
| **S2c α Sweep** | Sweep α ∈ {1.5, 2.0, 3.0, 4.0}. Same mechanism as κ but via ramp shape. | `ALFA` sweep | F4 (assign sooner = better), F10 (G dominates early) |
| **S2d Combined** | Cross-product of best β_G × best κ from S2a/S2b. | Combined config | Dependent on S2a/S2b results |

**Decision Gate:** If FO parameters move NR% → fine-tune and lock the best combination. If NR% remains flat → FO tuning is zero-sum, need structural changes → proceed to Layer 3.

---

#### Layer 3 — Two-Tier Structural Variants

> **Rationale:** The operator gap on T1 (F7) suggests that reserving valuable operators for T1 calls could break the constraint where all calls compete for the same pool. More complex to simulate — requires structural solver changes. Now explore the structural parameters: how strict should the reservation be? How much of the spread must the gate require?

| Scenario | Description | Config Change | Evidence / Hypothesis |
|----------|-------------|---------------|----------------------|
| **S3a Soft Reserve** | Valuable operators may serve T2 when no T1 calls are waiting. Maximises throughput at the cost of potential dilution. | Inherits S0', `TIER1_STRICT_RESERVE=False` | F7 (operator gap — but may leak if valuable ops serve T2) |
| **S3b Strict Reserve** | Valuable operators **never** serve T2 calls. Guarantees availability for T1 at the cost of idle capacity. | Inherits S0', `TIER1_STRICT_RESERVE=True` | F7 (idle capacity trade-off vs reservation guarantee) |
| **S3c Gate Sweep** | Sweep LIFT_FRAC ∈ {30%, 40%, 50%, 60%, 70%}. Controls how selective the operator gate is within T1. | `TIER1_LIFT_FRAC` sweep | F6/F7 — gate selectivity vs T1 coverage trade-off |
| **S3d T1 Threshold Sweep** | Sweep MAX_CIJ_THRESH ∈ {0.40, 0.45, 0.50, 0.55, 0.60} and SPREAD_THRESH ∈ {0.05, 0.10, 0.15}. Trades T1 size vs selectivity. | Threshold sweep grid | F6 — trades T1 size for selectivity |

**Decision Gate:** Pick the (LIFT_FRAC, reserve mode, T1 thresholds) combination that minimises NR%. Combined with best FO parameters from Layer 2, this becomes the optimised production configuration.

---

#### Layer 4 — Operational & Structural Innovations *(require development)*

> **Rationale:** These scenarios change assumptions beyond FO tuning and require new code, model changes, or operational decisions. Listed here for completeness and to guide the product roadmap.

##### Layer 4a — Operational Levers (implementable through operational changes)

| Scenario | Description | Mechanism | Evidence | Status |
|----------|-------------|-----------|----------|--------|
| **S4a Extended Wait** | Increase callback deflection from 55s to 90s (or disable). More calls reach the FO gate opening window. | Changes `CALIBRATED_CALLBACK_DEFLECTION_TIME` | F3 (gate opens at ~100s but callbacks deflect at ~55s) | 🟡 Needs operational buy-in |
| **S4b Junior Operator Pool** | Add N low-skill operators dedicated to T2 calls. Frees valuable operators for T1 permanently. | New `JUNIOR_POOL_SIZE` parameter + T2-only assignment | F7 (13.4pp gap: valuable ops wasted on undifferentiated calls) | 🟡 Needs HR/staffing decision |
| **S4c Off-Peak T1 Only** | During low-load hours, run T1-only routing (all operators reserved). During peak, fall back to shared pool. | Time-based tier activation | Capacity analysis (daily operator count varies 2–3×) | 🟡 Needs cadence analysis |

##### Layer 4b — FO Formula Innovations (require solver development)

| Scenario | Description | Mechanism | Evidence | Status |
|----------|-------------|-----------|----------|--------|
| **S4d Selective Abstention** | Solver returns "don't assign" for calls where all operators score badly. Holds capacity for better-matched calls arriving soon. | New FO output: assign / abstain / defer | F1 (most assignments had a better option — holding could help) | 🔴 Requires solver redesign |
| **S4e CLV Weighting** | Weight the objective by Customer Lifetime Value: FO = CLV_i × (β_G · G + β_P · P + κ). High-CLV customers get priority. | Multiply FO by CLV_i | Business hypothesis: some churns cost more than others | 🔴 Requires CLV feature + model retrain |
| **S4f Look-Ahead / DP** | Replace myopic per-tick Hungarian with a look-ahead solver that considers future arrivals. Breaks the greedy assignment assumption. | Dynamic programming or MPC formulation | F1 (myopic optimal may miss globally better allocations) | 🔴 Research-level; significant solver complexity |
| **S4g Listwise LTR** | Replace pointwise churn_ij model with a listwise learning-to-rank model. Better discrimination in flat (low-saliency) slates. | Model architecture change (LambdaMART / LambdaRank) | F10 (operator discrimination drives FO), Cell 21b (low-saliency slates) | 🔴 Requires model retrain + MLflow integration |

##### Layer 4c — Model Quality Improvements (require DS/ML work)

| Scenario | Description | Mechanism | Evidence | Status |
|----------|-------------|-----------|----------|--------|
| **S4h Model Sensitivity** | Retrain smartpairing model to increase operator discrimination. If churn_ij range per call increases, FO tuning becomes more effective. | Feature engineering / model architecture | F10 (operator discrimination drives FO composition) | 🔴 Requires DS sprint |
| **S4i Retire churn_i** | Production computes churn_i by nulling operator features — asking the model a question it was never trained to answer (F5). No call happens without an operator, so P(churn \| call) is ontologically invalid in this context. **Resolution options:** **(a)** Slate-relative baseline (Layer 0 already does this: gate uses max_cij/min_cij instead of churn_i — no new model needed), **(b)** leave-one-out average (churn_i = mean churn_ij over all operators for that call — cheap, already computable from slate), **(c)** if a single-number "call risk" is still needed operationally, use the slate mean or percentile rather than an out-of-distribution model query. This item becomes **low priority** if Layer 0 slate gates prove sufficient, since they bypass churn_i entirely. | Deployment change: stop computing churn_i, use slate statistics | F5 (churn_i asks the wrong question) | 🟡 Low if slate gates work; needs API change otherwise |
| **S4j Listwise Neural GAM** | Replace pointwise EBM (scores each (call, operator) independently) with a **listwise LTR model using Neural GAM** architecture. Neural GAMs replace tree-based shape functions with small neural networks — keeping interpretability (per-feature shape plots) while gaining **differentiability**, enabling direct optimisation of a listwise loss (e.g. ListNet / LambdaLoss over NDCG). **With context:** add slate-level features (queue depth, time-of-day, operator pool composition) as conditioning inputs so the model learns *which operator matters more given the current slate*, not just in isolation. This addresses the low-saliency problem where pointwise scores are nearly flat across operators. | Model architecture: EBM → Neural GAM + listwise loss + slate context features | F10 (operator discrimination drives FO), Cell 21b (low-saliency slates) | 🔴 Requires DS research + training pipeline changes |

---

### 3. Execution Priority

```
Layer 0 (S0, S0')  →  Layer 1 (S1)  →  Layer 2 (S2a–S2d)  →  Layer 3 (S3a–S3d)
                      [decision gate]         [decision gate]
                      avail-only viable?      FO params move NR%?
                                                                  ╲
                                                                   → Layer 4 (roadmap)
```

**Immediate next steps (this notebook):**
1. Run S0 (production baseline) and S0' (slate baseline) — establish reference points
2. Run S1 (available-only scoring) to verify solver works without imminent noise
3. Based on S1 result, run S2a–S2d FO parameter sweep (simpler, parameter-only)
4. Run S3a–S3d two-tier structural sweep (more complex, structural changes)

**Product roadmap (requires external coordination):**
5. S4a (extended wait) → discuss with operations
6. S4g (listwise LTR) + S4i (retire churn_i) + S4j (listwise Neural GAM) → DS backlog
7. S4f (look-ahead) → research spike

---

### 4. Why This Order?

- **Layer 0 before everything:** F5 shows that churn_i asks the model a question it was never trained to answer (P(churn | call) without an operator). We need both a production-faithful reference and a corrected slate-based alternative that uses the model as designed — always conditioning on both call and operator.
- **Layer 1 before Layer 2:** Production never assigns imminent operators, yet they inflate spread/gap numbers (F8). S1 confirms the solver works correctly on available-only slates before we tune parameters or build two-tier routing on top.
- **Layer 2 before Layer 3:** FO parameter sweeps (β_G, κ, α) are purely parameter-only experiments — fast to simulate, no structural changes needed. They establish the ceiling of what parameter tuning can achieve. If FO tuning is zero-sum (NR% flat), then structural changes (Layer 3) are needed.
- **Layer 3 before Layer 4:** The two-tier structure partially separates operator pools and requires structural solver changes. Exhaust simpler experiments first before committing to complex structural modifications.
- **Layer 4 is a roadmap, not a simulation plan.** These items inform the product backlog but require coordination beyond this notebook.

In [ ]:
# ==========================================================================
# CELL 22 — SCENARIO SYNTHESIS: Evidence → Experiment Dictionary  [Scenario Development]
# ==========================================================================
#   → Consolidates findings from ALL diagnostic cells into a concise
#     evidence table and outputs the `experiments` dict consumed
#     by the experimentation cells (23+).
#
#   → Structure mirrors the layered roadmap in the Cell 22 markdown above:
#       Layer 0 — Baselines (S0, S0')
#       Layer 1 — Available-Only Scoring (S1)
#       Layer 2 — FO Parameter Sweep (S2a–S2d)
#       Layer 3 — Two-Tier Structural Variants (S3a, S3b, S3c sweep)
#       REF     — Oracle ceiling (S7)
#
# INPUT:  Outputs from diagnostic cells (already in kernel memory)
#         tier1_calls (Cell 22a), valuable_ops (Cell 22b)
# OUTPUT: `experiments` dict of {name: FOConfig} ready for cell 23
# ==========================================================================

# ═══════════════════════════════════════════════════════════════════════════
# A. EVIDENCE SUMMARY — Key findings driving scenario design
# ═══════════════════════════════════════════════════════════════════════════
_evidence = [
    # (source, finding, implication)
    ("Unserved Diagnostic",
     "High unserved rate — majority NoAgentReturned (FO ≤ 0) (see unserved diagnostic)",
     "Coverage is the #1 volume problem → loosen gate"),
    ("Regret Analysis",
     "Most assignments had a better operator available (see regret analysis)",
     "Matching quality is the #1 value problem → increase β_G"),
    ("FO Analysis",
     "Gate opens at ~100s; κ relaxation shifts opening earlier (see FO analysis)",
     "κ relaxation is the main gate lever"),
    ("Decision Support",
     "Waiting hurts ΔChurn; most value captured by ~185s (see decision support)",
     "Assign sooner → better outcomes"),
    ("Operator Profiling",
     "Distributed/specialist operator park (see operator profiling)",
     "Right specialist for right customer → β_G matters"),
    ("Imminent Betting",
     "Routing regret dominates imminent premium (see regret decomposition)",
     "β_G has 4× more ROI than imminent (future phase)"),
    ("NR% Diagnostic",
     "NR% flat across initial FO parameter sweep: zero-sum redistribution (see NR% diagnostic)",
     "FO weight tuning is zero-sum → need structural FO change"),
    ("Churn Discrimination",
     "Most calls have significant spread, but c=sigmoid(churn_i) misweights (see churn discrimination)",
     "Multiply risk × spread → focus QP on salvageable calls"),
    ("Baseline Model Bias",
     "Majority have negative ci-cij lift; max_cij well above ci (see baseline model bias)",
     "churn_i is wrong baseline → use max_cij from slate"),
    ("Two-Tier Feasibility",
     f"{len(tier1_calls):,} T1 calls ({len(tier1_calls)/len(df_prof)*100:.0f}%), "
     f"{(worst_20_mean - best_20_mean)*100:.1f}pp op gap",
     "Structural break: reserve valuable ops for salvageable calls"),
    ("Valuable Operators",
     f"{len(valuable_ops)} ops with below-median cij on T1 (≥{MIN_T1_EXPOSURE} exposures)",
     "Well-defined valuable set for reservation strategy"),
    ("Slate Composition",
     "Most T1 slates contain imminent operators; spread may be inflated (see slate composition)",
     "Spread may be inflated by non-selectable operators → test L1"),
]

print("📋 EVIDENCE SYNTHESIS")
print("=" * 90)
print(f"  {'Source':<24s} {'Finding':<45s} {'Implication'}")
print("-" * 90)
for src, finding, impl in _evidence:
    print(f"  {src:<24s} {finding[:43]:<45s} {impl}")

# ═══════════════════════════════════════════════════════════════════════════
# B. SHARED BASE CONFIGS
# ═══════════════════════════════════════════════════════════════════════════

# L2 base: standard solver with slate-based g_value (S0' improvements only)
# NO USE_TWO_TIER — L2 sweeps FO parameters through the standard solver path
# where β_G, κ, α actually drive compute_score() and the Hungarian assignment.
_fo_sweep_base = dict(
    USE_SLATE_G=True,               # use max_cij baseline instead of biased churn_i
)

# L3 base: two-tier structural solver (oracle mode within tiers)
_slate_base = dict(
    USE_TWO_TIER=True,
    TIER1_CALL_IDS=tier1_calls,        # Cell 22a: T1 = high max_cij + high spread
    TIER1_VALUABLE_OPS=valuable_ops,    # Cell 22b: ops with below-median cij on T1
    USE_SLATE_LIFT=True,                # gate uses max_cij baseline, not biased churn_i
    TIER1_LIFT_FRAC=0.50,              # operator must capture ≥50% of slate spread
)

# ═══════════════════════════════════════════════════════════════════════════
# C. EXPERIMENT DICTIONARY — layered structure from Cell 22 markdown
# ═══════════════════════════════════════════════════════════════════════════

experiments = {}

# ═══ LAYER 0 — ESTABLISH BASELINES ═══════════════════════════════════════
#   Rationale: production-faithful reference + corrected slate-based
#   alternative (F5: churn_i asks the wrong question)

experiments["L0 S0 Baseline"] = FOConfig()
#   Current production FO: β_G=0.35, β_P=0.65, κ=−0.304, α=3.0
#   Reference point for all comparisons

experiments["L0 S0' Slate Baseline"] = FOConfig(USE_SLATE_G=True)
#   Same FO formula as production, but g_value numerator uses max(churn_ij in tick slate)
#   instead of churn_i. Anchors the churn reduction to the worst operator in the
#   slate — always a real model output, not an artificial churn_i computed by
#   nulling operator features (F5: churn_i asks the wrong question).
#   The c gate sigmoid(K·churn_i − τ) still uses churn_i for risk prioritization.

# ═══ LAYER 1 — IMMINENT INFLATION TEST ══════════════════════════════════
#   Rationale: Most T1 slates include imminent ops (F8).
#   Spread and operator gap may be inflated by non-selectable operators.
#   Must test before tuning Layer 2/3 parameters.

# S1: Available-Only — score_lookup filtered to available operators only.
# Tests: what if smartpairing only scored available operators?
# Uses avail_only_score_lookup (computed in L1 diagnostic cell).
experiments["L1 S1 Avail-Only"] = FOConfig(
    USE_SLATE_G=True,
    EXCLUDE_IMMINENT_FROM_SLATE=True,
)
#   Same as S0' but solver only sees scores for historically-available operators.
#   Calls where the best operator was imminent will use fallback_churn_ij.
#   Evidence: F8 (most T1 have imminent ops), F7 (operator gap may shrink)

# ═══ LAYER 2 — FO PARAMETER SWEEP ═══════════════════════════════════════
#   Rationale: The simplest parameter-only experiments. β_G, κ, α sweeps
#   through the STANDARD solver (solve_assignments) with slate-based g_value.
#   Tests what the FO formula can achieve with parameter tuning alone.
#   Fast to simulate, no structural changes needed.

# S2a: β_G Sweep — routing intelligence emphasis
#   Evidence: F1 (routing regret), F2 (specialist park)
for _bg in [0.35, 0.50, 0.65, 0.80]:
    experiments[f"L2 S2a βG={_bg:.2f}"] = FOConfig(
        **_fo_sweep_base, BETA_G=_bg, BETA_P=1.0 - _bg,
    )

# S2b: κ Sweep — gate timing
#   Evidence: F3 (high unserved rate), F9 (κ shifts gate opening)
for _kappa in [-0.30, -0.25, -0.20, -0.15]:
    experiments[f"L2 S2b κ={_kappa:.2f}"] = FOConfig(
        **_fo_sweep_base, KAPPA=_kappa,
    )

# S2c: α Sweep — ramp shape (independent timing lever)
#   Evidence: F4 (assign sooner = better), F10 (G dominates early)
for _alfa in [1.5, 2.0, 3.0, 4.0]:
    experiments[f"L2 S2c α={_alfa:.1f}"] = FOConfig(
        **_fo_sweep_base, ALFA=_alfa,
    )

# S2d: Combined — cross-product of best β_G × best κ from S2a/S2b
#   ⚠️  Defined AFTER S2a/S2b results analysis. Placeholder:
# experiments["L2 S2d Combined"] = FOConfig(**{**_fo_sweep_base, "BETA_G": ?, "KAPPA": ?})

# ═══ LAYER 3 — TWO-TIER STRUCTURAL VARIANTS ═════════════════════════════
#   Rationale: operator gap on T1 (F7) → reserving valuable ops
#   for T1 calls could break the zero-sum constraint. More complex to
#   simulate — requires structural solver changes (solve_tiered).
#   Run AFTER FO sweep to first establish the parameter-only ceiling.

# S3a: Soft Reserve (identical to S0' — TIER1_STRICT_RESERVE defaults False)
experiments["L3 S3a Soft Reserve"] = FOConfig(
    **_slate_base,
    TIER1_STRICT_RESERVE=False,
)
#   Valuable ops may serve T2 when no T1 waiting. Max throughput.
#   Evidence: F7 (operator gap — but may leak if valuable ops serve T2)

# S3b: Strict Reserve — valuable ops NEVER serve T2
experiments["L3 S3b Strict Reserve"] = FOConfig(
    **_slate_base,
    TIER1_STRICT_RESERVE=True,
)
#   Guarantees availability for T1 at cost of idle capacity.
#   A/B test vs S3a: if S3b > S3a → leakage dilutes effect.
#   Evidence: F7 (idle capacity vs reservation guarantee)

# S3c: Gate Sweep — LIFT_FRAC ∈ {30%, 40%, 50%, 60%, 70%}
#   Controls how selective the assignment gate is.
#   Note: 50% overlaps with S0' — included for sweep completeness.
for _frac in [0.30, 0.40, 0.50, 0.60, 0.70]:
    experiments[f"L3 S3c Gate {int(_frac * 100)}%"] = FOConfig(
        **{**_slate_base, "TIER1_LIFT_FRAC": _frac},
    )

# S3d: T1 Threshold Sweep
#   ⚠️  TODO: requires recomputing tier1_calls per (MAX_CIJ_THRESH, SPREAD_THRESH).
#   Sweep grid: MAX_CIJ_THRESH ∈ {0.40, 0.45, 0.50, 0.55, 0.60}
#             × SPREAD_THRESH ∈ {0.05, 0.10, 0.15}
#   Current thresholds: MAX_CIJ_THRESH=0.50, SPREAD_THRESH=0.10 (see T1 call count from Cell 22a)

# ═══ REFERENCE — ORACLE CEILING ═════════════════════════════════════════
experiments["REF S7 Oracle"] = FOConfig(USE_DIRECT_CHURN=True)
#   Bypasses FO entirely. score = 1 - churn_ij → Hungarian minimises
#   raw churn_ij. No gate, no waiting-time influence.
#   CEILING TEST: if S7 can't move NR% → no FO tuning ever will.

# ═══════════════════════════════════════════════════════════════════════════
# D. PRINT EXPERIMENT TABLE — organised by layer
# ═══════════════════════════════════════════════════════════════════════════
_prod = FOConfig()
print(f"\n\n🧪 EXPERIMENT DICTIONARY — {len(experiments)} scenarios")
print("=" * 90)
print(f"  {'Scenario':<28s} {'β_G':>5s} {'β_P':>5s} {'κ':>8s} {'α':>5s}  {'Δ vs production'}")
print("-" * 90)

_current_layer = None
for name, cfg in experiments.items():
    # Print layer separator
    _layer = name.split()[0]  # "L0", "L2", "L3", "REF"
    if _layer != _current_layer:
        _current_layer = _layer
        _layer_names = {
            "L0": "Layer 0 — Baselines",
            "L1": "Layer 1 — Imminent Inflation Test",
            "L2": "Layer 2 — FO Parameter Sweep (standard solver)",
            "L3": "Layer 3 — Two-Tier Structural (oracle solver)",
            "REF": "Reference — Ceiling Tests",
        }
        print(f"\n  ┌─ {_layer_names.get(_layer, _layer)} {'─' * 50}")

    deltas = []
    if cfg.USE_SLATE_G:
        deltas.append("🔄 SLATE G (max_cij baseline)")
    if cfg.EXCLUDE_IMMINENT_FROM_SLATE:
        deltas.append("🚫 EXCL IMMINENT")
    if cfg.USE_TWO_TIER:
        _n_t1 = len(cfg.TIER1_CALL_IDS) if cfg.TIER1_CALL_IDS else 0
        _n_vo = len(cfg.TIER1_VALUABLE_OPS) if cfg.TIER1_VALUABLE_OPS else 0
        _flags = []
        if cfg.TIER1_STRICT_RESERVE:
            _flags.append("STRICT")
        if cfg.USE_SLATE_LIFT:
            _flags.append(f"SLATE≥{cfg.TIER1_LIFT_FRAC:.0%}")
        _flag_str = " " + "+".join(_flags) if _flags else ""
        deltas.append(f"🔀 TWO-TIER{_flag_str} ({_n_t1} T1, {_n_vo} ops)")
    elif cfg.USE_DIRECT_CHURN:
        deltas.append("⚡ ORACLE (min churn_ij)")
    else:
        if cfg.BETA_G != _prod.BETA_G:
            deltas.append(f"β_G {cfg.BETA_G - _prod.BETA_G:+.02f}")
        if cfg.BETA_P != _prod.BETA_P:
            deltas.append(f"β_P {cfg.BETA_P - _prod.BETA_P:+.02f}")
        if abs(cfg.KAPPA - _prod.KAPPA) > 0.001:
            deltas.append(f"κ {cfg.KAPPA - _prod.KAPPA:+.03f}")
        if cfg.ALFA != _prod.ALFA:
            deltas.append(f"α {cfg.ALFA - _prod.ALFA:+.01f}")
    delta_str = ", ".join(deltas) if deltas else "★ baseline"
    print(f"  │ {name:<28s} {cfg.BETA_G:>5.2f} {cfg.BETA_P:>5.2f} {cfg.KAPPA:>8.3f} {cfg.ALFA:>5.1f}  {delta_str}")

# E. LAYER DEPENDENCIES & TODO
# ═══════════════════════════════════════════════════════════════════════════
print(f"\n\n📌 LAYER DEPENDENCIES")
print("=" * 90)
print("  Layer 0 → Layer 1 → Layer 2 → Layer 3 → Layer 4 (roadmap)")
print()
print("  ⚠️  S2d (Combined) — NOT YET IN DICT")
print("     Defined after S2a/S2b results identify best β_G and κ")
print()
print("  ⚠️  S3d (T1 Threshold Sweep) — NOT YET IN DICT")
print("     Requires recomputing tier1_calls per (MAX_CIJ_THRESH, SPREAD_THRESH) grid")
print()
print("  Layer 4 (S4a–S4j) — ROADMAP ONLY")
print("     Requires new code, model changes, or operational decisions")
print("     See Cell 22 markdown for full descriptions")
print(f"\n✅ `experiments` dict ready ({len(experiments)} scenarios) — run cell 23 to execute")

<a id="experiment-runner"></a>
## Experiment Runner

In [ ]:
# =============================================================================
# CELL 23 — EXPERIMENT RUNNER: reusable function + day setup    [Experimentation]
# =============================================================================
# INPUT:
#   - `experiments` dict: {name: FOConfig} — defined in cell 22 (scenario synthesis)
#   - FOConfig class (cell 10), run_simulation (cell 12), analyze_simulation (cell 9)
#   - df_call_events, df_calls_qplanner, score_lookup, churn_i_lookup (cell 7)
#   - qp_days (list of 21 dates, from cell 13)
#   - CALIBRATED_* parameters
#   - fallback_churn_ij (from override cell)
#
# DOES:
#   1. Pre-computes per-day setup (operators, windows, occupancy)
#   2. Defines `run_scenario_group(scenario_names)` — reusable function
#      that runs a subset of scenarios and returns results
#
# OUTPUT:
#   - day_setups: dict of per-day simulation setup
#   - run_scenario_group(): function to execute any group of scenarios
#   - results_comparison: dict accumulating all results across groups
# =============================================================================

import time as _time

# `experiments` dict comes from cell 22 — verify it's loaded
assert 'experiments' in dir() and isinstance(experiments, dict) and len(experiments) > 0, \
    "Run cell 22 first to define the `experiments` dictionary"

# ─── PRE-COMPUTE PER-DAY SETUP ────────────────────────────────────────────────
# Same per-day operator/window/occupancy setup as cell 13, computed once.
day_setups = {}
for day in qp_days:
    day_str = str(day)
    df_qp_day = df_calls_qplanner_ts[df_calls_qplanner_ts['_date'] == day]
    _windows = detect_active_windows(df_qp_day, gap_threshold_min=5.0)
    _df_ops, _op_shifts = get_day_operators(df_calls_qplanner, day_str)
    if _df_ops.empty:
        continue
    _occ_data = df_qp_day.copy()
    _occ_data['hour'] = pd.to_datetime(_occ_data['START_DATE_TIME']).dt.hour
    _ops_h = _occ_data.groupby('hour')['agent_username'].nunique()
    _pool = len(_df_ops)
    _occ = {h: min(0.99, (1 - n / _pool) * CALIBRATED_OCC_SCALE) for h, n in _ops_h.items()}
    day_setups[day_str] = {
        'df_operators': _df_ops, 'operator_shifts': _op_shifts,
        'active_windows': _windows, 'hourly_occupancy': _occ,
    }

print(f"📐 Pre-computed setup for {len(day_setups)} days")

# ─── RESULTS ACCUMULATOR ──────────────────────────────────────────────────────
# Persists across groups — each group ADDS to this dict.
results_comparison = {}

# ─── REUSABLE RUNNER FUNCTION ─────────────────────────────────────────────────
def run_scenario_group(scenario_names: list, seed: int = 42) -> dict:
    """Run a group of scenarios and add results to results_comparison.

    Args:
        scenario_names: list of scenario keys from `experiments` dict.
        seed: random seed for stochastic simulation components.

    Returns:
        dict of {name: result_dict} for the scenarios just run.
    """
    _run_exps = {k: experiments[k] for k in scenario_names if k in experiments}
    _missing = [k for k in scenario_names if k not in experiments]
    if _missing:
        print(f"⚠️  Unknown scenarios (skipped): {_missing}")
    print(f"🧪 Running {len(_run_exps)}/{len(experiments)} scenarios: {list(_run_exps.keys())}\n")

    group_results = {}
    for exp_name, config in _run_exps.items():
        t0 = _time.time()
        print(f"{'─'*60}")
        print(f"  {exp_name}")
        print(f"{'─'*60}")

        # S1 (EXCLUDE_IMMINENT_FROM_SLATE): only truly-available operators in solver.
        # Default: solver sees available + imminent operators (matching production).
        if config.EXCLUDE_IMMINENT_FROM_SLATE:
            print(f"  📋 EXCLUDE_IMMINENT_FROM_SLATE: avail-only solver "
                  f"(full score_lookup, {len(score_lookup):,} pairs)")
        else:
            print(f"  📋 Imminent window: {config.IMMINENT_WINDOW_S:.0f}s "
                  f"(full score_lookup, {len(score_lookup):,} pairs)")

        exp_assignments = []
        exp_fifo_assignments = []
        exp_unserved_calls = []
        exp_failed_deliveries = 0
        _exp_total_pairs = 0
        _exp_lookup_hits = 0

        for day_str, setup in sorted(day_setups.items()):
            state = run_simulation(
                df_call_events=df_call_events,
                df_operators=setup['df_operators'],
                score_lookup=score_lookup,
                churn_i_lookup=churn_i_lookup,
                fo_config=config,
                tick_interval=CALIBRATED_TICK,
                max_wait_time=180,
                sim_day=day_str,
                active_windows=setup['active_windows'],
                gap_busy_fraction=0.7,
                hourly_occupancy=setup['hourly_occupancy'],
                operator_shifts=setup['operator_shifts'],
                fallback_churn_ij=fallback_churn_ij,
                pipeline_latency=CALIBRATED_LATENCY,
                pipeline_latency_std=CALIBRATED_LATENCY_STD,
                nqp_mean_task_s=CALIBRATED_NQP_TASK_S,
                fifo_latency=CALIBRATED_FIFO_LATENCY,
                fifo_block_operators=CALIBRATED_FIFO_BLOCK,
                enable_fifo=True,
                delivery_failure_rate=CALIBRATED_DELIVERY_FAILURE_RATE,
                callback_deflection_time=CALIBRATED_CALLBACK_DEFLECTION_TIME,
                callback_deflection_rate=CALIBRATED_CALLBACK_DEFLECTION_RATE,
                acw_mean_s=CALIBRATED_ACW_MEAN_S,
                seed=seed,
                verbose=False,
            )
            exp_assignments.extend(state.assignments)
            exp_fifo_assignments.extend(state.fifo_assignments)
            exp_unserved_calls.extend(state.unserved_calls)
            exp_failed_deliveries += len(state.failed_deliveries)

            if hasattr(state, 'score_coverage') and state.score_coverage:
                _exp_total_pairs += state.score_coverage.get('total_pairs', 0)
                _exp_lookup_hits += state.score_coverage.get('lookup_hits', 0)

        elapsed = _time.time() - t0
        n = len(exp_assignments)
        n_fifo = len(exp_fifo_assignments)
        n_unserved = len(exp_unserved_calls)
        waits = [a.waiting_time for a in exp_assignments]
        churn_red = [a.churn_reduction for a in exp_assignments]
        n_total = n + n_fifo + n_unserved
        svc_rate = (n + n_fifo) / n_total * 100 if n_total > 0 else 0

        _cov_pct = _exp_lookup_hits / _exp_total_pairs * 100 if _exp_total_pairs > 0 else 0
        _fb_pairs = _exp_total_pairs - _exp_lookup_hits
        _fb_pct = _fb_pairs / _exp_total_pairs * 100 if _exp_total_pairs > 0 else 0

        print(f"  {len(day_setups)} days | {n:,} QP | {n_fifo:,} FIFO | {n_unserved:,} lost | "
              f"svc {svc_rate:.1f}% | med wait {np.median(waits):.0f}s | "
              f"avg Δchurn {np.mean(churn_red):.4f} | {elapsed:.1f}s")
        print(f"  Score coverage: {_exp_lookup_hits:,}/{_exp_total_pairs:,} ({_cov_pct:.1f}%) | "
              f"fallback: {_fb_pairs:,} pairs ({_fb_pct:.1f}%)\n")

        result = {
            'assignments': exp_assignments,
            'fifo_assignments': exp_fifo_assignments,
            'unserved_calls': exp_unserved_calls,
            'failed_deliveries': exp_failed_deliveries,
            'score_coverage': {
                'total_pairs': _exp_total_pairs,
                'lookup_hits': _exp_lookup_hits,
                'coverage_pct': round(_cov_pct, 2),
                'fallback_pairs': _fb_pairs,
                'fallback_pct': round(_fb_pct, 2),
            },
        }
        results_comparison[exp_name] = result
        group_results[exp_name] = result

    print(f"\n✅ Group done — {len(group_results)} scenarios. "
          f"Total in results_comparison: {len(results_comparison)}")
    return group_results

print(f"\n✅ run_scenario_group() ready. {len(experiments)} scenarios available.")

---
# Part V — Layer Execution & Results


<a id="layer-0"></a>
---
## Layer 0 — Establish Baselines (S0, S0')

Run the two baseline scenarios:
- **S0 Production Baseline**: G = churn_i − churn_ij (churn_i = artificial, nulled operator features)
- **S0' Slate Baseline**: g_value = (max_cij_slate − churn_ij) / denom; c gate still uses churn_i for risk prioritization

Same FO formula, same gate, same P — only the G baseline changes.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LAYER 0 — Run S0 (Production Baseline) + S0' (Slate Baseline)
# ═══════════════════════════════════════════════════════════════════════════════
# These are the two reference scenarios. All subsequent layers compare against
# these baselines to measure improvement.
#
# S0:  Current production FO — g_value = (churn_i − churn_ij) / denom
# S0': g_value uses max(churn_ij in slate) as baseline; c gate keeps churn_i (USE_SLATE_G=True)
# ═══════════════════════════════════════════════════════════════════════════════

layer_0_results = run_scenario_group([
    "L0 S0 Baseline",
    "L0 S0' Slate Baseline",
])

<a id="s0-diagnostic"></a>
### Diagnostic: Why S0' Slate Baseline ≠ Better Pairing?

**Hypothesis**: Slate-relative G changes the *magnitude* of G but **not the operator ranking** within any slate. The Hungarian algorithm picks the same optimal pairing — only the KAPPA gate threshold behavior changes. For high-`churn_i` calls (where `churn_i > max_cij`), `g_value` actually **drops**, reducing FO and pushing some calls from QP → FIFO.

In [ ]:
# =============================================================================
# DIAGNOSTIC — S0 vs S0' G-value mechanism analysis
# =============================================================================
# Key insight: slate-relative G changes g_value MAGNITUDE but NOT operator
# ranking within a slate. The Hungarian solver picks the same pairs — only
# the KAPPA gate threshold behavior changes.
#
# For calls where churn_i > max_cij (inflated baseline), g_value DROPS,
# pushing some high-risk calls from QP → FIFO. For calls where churn_i <
# max_cij, g_value rises but these calls have low c=sigmoid(churn_i), so
# the gate boost is small.
# =============================================================================

from collections import defaultdict

# ── 1. G-value distribution: production vs slate-relative ──────────────────
# Sample representative calls and compute g_value under both methods.
_unique_cids = list(set(c for c, _ in score_lookup.keys()))
_sample_n = min(2000, len(_unique_cids))
_sample_cids = np.random.choice(_unique_cids, _sample_n, replace=False)

g_prod_all, g_slate_all = [], []
g_prod_high_ci, g_slate_high_ci = [], []    # churn_i > max_cij (inflated baseline)
g_prod_low_ci, g_slate_low_ci = [], []      # churn_i < max_cij (deflated baseline)
n_inflated, n_deflated = 0, 0

for cid in _sample_cids:
    ci = churn_i_lookup.get(cid)
    if ci is None:
        continue
    # Get all churn_ij for this call
    ops_cij = {o: v for (c, o), v in score_lookup.items() if c == cid}
    if len(ops_cij) < 2:
        continue
    max_cij = max(ops_cij.values())
    is_inflated = ci > max_cij  # churn_i higher than worst operator

    for op, cij in ops_cij.items():
        # Production g_value
        denom_prod = abs(ci) + abs(cij)
        gv_prod = (ci - cij) / denom_prod if denom_prod > 1e-9 else 0.0
        # Slate-relative g_value
        denom_slate = abs(max_cij) + abs(cij)
        gv_slate = (max_cij - cij) / denom_slate if denom_slate > 1e-9 else 0.0

        g_prod_all.append(gv_prod)
        g_slate_all.append(gv_slate)
        if is_inflated:
            g_prod_high_ci.append(gv_prod)
            g_slate_high_ci.append(gv_slate)
        else:
            g_prod_low_ci.append(gv_prod)
            g_slate_low_ci.append(gv_slate)

    if is_inflated:
        n_inflated += 1
    else:
        n_deflated += 1

print("=" * 90)
print("  DIAGNOSTIC: S0 vs S0' — g_value MAGNITUDE analysis")
print("=" * 90)
print(f"\n  Sample: {_sample_n} calls, {len(g_prod_all):,} (call, op) pairs")
print(f"  Calls with churn_i > max_cij (inflated baseline): {n_inflated:,} ({n_inflated/(n_inflated+n_deflated)*100:.1f}%)")
print(f"  Calls with churn_i ≤ max_cij (deflated baseline): {n_deflated:,} ({n_deflated/(n_inflated+n_deflated)*100:.1f}%)")

print(f"\n  ┌─────────────────────────────────────────────────────────────────────────┐")
print(f"  │  g_value distribution                 S0 (production)  S0' (slate-rel)  │")
print(f"  ├─────────────────────────────────────────────────────────────────────────┤")
print(f"  │  ALL PAIRS                                                              │")
print(f"  │    Mean g_value                       {np.mean(g_prod_all):>+10.4f}       {np.mean(g_slate_all):>+10.4f}       │")
print(f"  │    Median g_value                     {np.median(g_prod_all):>+10.4f}       {np.median(g_slate_all):>+10.4f}       │")
print(f"  │    % negative g_value                 {sum(1 for g in g_prod_all if g < 0)/len(g_prod_all)*100:>9.1f}%       {sum(1 for g in g_slate_all if g < 0)/len(g_slate_all)*100:>9.1f}%       │")
print(f"  ├─────────────────────────────────────────────────────────────────────────┤")
print(f"  │  INFLATED (churn_i > max_cij) — {n_inflated} calls, {len(g_prod_high_ci):,} pairs                  │")
print(f"  │    Mean g_value                       {np.mean(g_prod_high_ci):>+10.4f}       {np.mean(g_slate_high_ci):>+10.4f}       │")
print(f"  │    ⚠️  G DROPS: slate max_cij < churn_i → smaller numerator            │")
if g_prod_low_ci:
    print(f"  ├─────────────────────────────────────────────────────────────────────────┤")
    print(f"  │  DEFLATED (churn_i ≤ max_cij) — {n_deflated} calls, {len(g_prod_low_ci):,} pairs                 │")
    print(f"  │    Mean g_value                       {np.mean(g_prod_low_ci):>+10.4f}       {np.mean(g_slate_low_ci):>+10.4f}       │")
    print(f"  │    ✅ G RISES: slate max_cij > churn_i → larger numerator             │")
print(f"  └─────────────────────────────────────────────────────────────────────────┘")

# ── 2. Gate impact: how does c=sigmoid(churn_i) interact? ──────────────────
# For inflated calls, churn_i is high → c is high → G=(.5+g)*c is large.
# Reducing g_value for these calls has a LARGE absolute effect on G.
# For deflated calls, churn_i is low → c is low → even larger g_value has
# a SMALL absolute effect on G.
_cfg = FOConfig()
print(f"\n  ┌─────────────────────────────────────────────────────────────────────────┐")
print(f"  │  GATE INTERACTION: c = sigmoid(K·churn_i − τ)                           │")
print(f"  ├─────────────────────────────────────────────────────────────────────────┤")

# Compute G = (0.5 + g_value) * c for sample calls under both methods
G_deltas_inflated, G_deltas_deflated = [], []
c_inflated, c_deflated = [], []

for cid in _sample_cids:
    ci = churn_i_lookup.get(cid)
    if ci is None:
        continue
    ops_cij = {o: v for (c, o), v in score_lookup.items() if c == cid}
    if len(ops_cij) < 2:
        continue
    max_cij = max(ops_cij.values())
    c_val = sigmoid(_cfg.CHURN_K * ci - _cfg.CHURN_TAU)
    is_inflated = ci > max_cij

    for op, cij in ops_cij.items():
        denom_prod = abs(ci) + abs(cij)
        gv_prod = (ci - cij) / denom_prod if denom_prod > 1e-9 else 0.0
        denom_slate = abs(max_cij) + abs(cij)
        gv_slate = (max_cij - cij) / denom_slate if denom_slate > 1e-9 else 0.0

        G_prod = (0.5 + gv_prod) * c_val
        G_slate = (0.5 + gv_slate) * c_val
        delta_G = G_slate - G_prod

        if is_inflated:
            G_deltas_inflated.append(delta_G)
            c_inflated.append(c_val)
        else:
            G_deltas_deflated.append(delta_G)
            c_deflated.append(c_val)

print(f"  │  INFLATED calls (S0' G DROPS):                                          │")
print(f"  │    Mean c (sigmoid gate):              {np.mean(c_inflated):.4f}                          │")
print(f"  │    Mean ΔG (S0'−S0):                   {np.mean(G_deltas_inflated):+.4f}  ← G drops        │")
print(f"  │    → High c amplifies the g_value reduction. Gate becomes HARDER.        │")
print(f"  ├─────────────────────────────────────────────────────────────────────────┤")
if G_deltas_deflated:
    print(f"  │  DEFLATED calls (S0' G RISES):                                          │")
    print(f"  │    Mean c (sigmoid gate):              {np.mean(c_deflated):.4f}                          │")
    print(f"  │    Mean ΔG (S0'−S0):                   {np.mean(G_deltas_deflated):+.4f}  ← G rises        │")
    print(f"  │    → Low c damps the g_value improvement. Gate boost is SMALL.          │")
    print(f"  ├─────────────────────────────────────────────────────────────────────────┤")
print(f"  │  NET EFFECT on FO = β_G·G + β_P·P + κ:                                 │")
_net_delta_G = np.mean(G_deltas_inflated + G_deltas_deflated)
_net_delta_FO = _cfg.BETA_G * _net_delta_G
print(f"  │    Net mean ΔG (all pairs):             {_net_delta_G:+.4f}                          │")
print(f"  │    Net mean ΔFO (= β_G × ΔG):          {_net_delta_FO:+.4f}                          │")
print(f"  └─────────────────────────────────────────────────────────────────────────┘")

# ── 3. Operator ranking preservation ──────────────────────────────────────
# Key: slate-relative G preserves operator ranking within a slate.
# Proof: for operators a, b in the same slate:
#   g_prod_a > g_prod_b ⟺ (ci - cij_a) > (ci - cij_b) ⟺ cij_a < cij_b
#   g_slate_a > g_slate_b ⟺ (max_cij - cij_a) > (max_cij - cij_b) ⟺ cij_a < cij_b
# Same ordering! Hungarian selects the same PAIRS — only the gate changes.
_n_rank_check = 0
_n_rank_preserved = 0
for cid in _sample_cids[:500]:
    ci = churn_i_lookup.get(cid)
    if ci is None:
        continue
    ops_cij = [(o, v) for (c, o), v in score_lookup.items() if c == cid]
    if len(ops_cij) < 3:
        continue
    max_cij = max(v for _, v in ops_cij)
    # Production ranking
    prod_rank = sorted(ops_cij, key=lambda x: -(ci - x[1]) / (abs(ci) + abs(x[1])) if abs(ci) + abs(x[1]) > 1e-9 else 0)
    # Slate ranking
    slate_rank = sorted(ops_cij, key=lambda x: -(max_cij - x[1]) / (abs(max_cij) + abs(x[1])) if abs(max_cij) + abs(x[1]) > 1e-9 else 0)
    _n_rank_check += 1
    if [o for o, _ in prod_rank] == [o for o, _ in slate_rank]:
        _n_rank_preserved += 1

print(f"\n  📐 OPERATOR RANKING VERIFICATION:")
print(f"     Checked {_n_rank_check} calls with ≥3 operators")
print(f"     Ranking preserved: {_n_rank_preserved}/{_n_rank_check} ({_n_rank_preserved/_n_rank_check*100:.1f}%)")
print(f"     → Slate-relative G changes WHO PASSES THE GATE, not WHO IS PAIRED WITH WHOM")

# ── 4. CONCLUSION ─────────────────────────────────────────────────────────
print(f"\n{'=' * 90}")
print(f"  CONCLUSION: Why S0' ≠ better pairing")
print(f"{'=' * 90}")
print(f"""
  1. Operator ranking is PRESERVED — the Hungarian solver picks the same (call, op)
     pairs when both pass the gate. Slate-relative G doesn't improve WHO gets paired.

  2. For {n_inflated/(n_inflated+n_deflated)*100:.0f}% of calls (churn_i > max_cij — inflated baseline), g_value DROPS.
     These calls have HIGH c (sigmoid gate), so the G reduction has a LARGE absolute
     effect → some calls that passed the KAPPA gate in S0 now fail → pushed to FIFO.
     Running S0 and S0' (Layer 0) will produce the QP assignment counts for comparison.

  3. For {n_deflated/(n_inflated+n_deflated)*100:.0f}% of calls (churn_i ≤ max_cij — deflated baseline), g_value RISES.
     But these calls have LOW c, so the improvement is damped. Net gate impact: small.

  4. Any NR% difference comes from the VOLUME SHIFT (more calls go to FIFO),
     not from worse pairing quality. Compare QP Regret across S0 and S0' to confirm.

  FIX: To unlock the value of slate-relative G, combine with:
    • Lower KAPPA (relax gate to compensate for the g_value magnitude change)
    • Higher β_G (amplify the cleaner signal so G dominates P in the FO)
    → This is exactly what the P1 combined scenario does.
""")


<a id="layer-1"></a>
---
## Layer 1 — Imminent Inflation Test

**Question:** Most T1 slates include imminent operators (F8). The T1 segmentation (max_cij > 0.50, spread > 0.10) and the operator gap (F7) were computed from the **full score_lookup** which includes ALL operators ever scored — including those that were busy/imminent at decision time.

**Key concern:** If imminent operators inflate the spread, then:
- T1 classification may include calls that don't really have different outcomes across *available* operators
- The operator gap (F7) may shrink to noise when restricted to who's actually available per-tick
- LIFT_FRAC gates computed on inflated spreads may be too loose or meaningless

**What we check (no simulation needed — purely diagnostic):**
1. **Available-only T1 recount**: recompute T1 using only available operators per call → how many calls survive?
2. **Available-only spread**: what's the operator spread when we exclude imminent operators?
3. **Available-only operator gap**: does the best-vs-worst gap hold on available operators?
4. **Decision gate**: available-only spread ≥ 8pp → proceed to L2. If it collapses → lower T1 thresholds or focus on FO parameter tuning.

In [ ]:
# =============================================================================
# LAYER 1 — IMMINENT INFLATION DIAGNOSTIC
# =============================================================================
# Recompute T1 segmentation using ONLY available operators (not imminent).
# This answers: does the two-tier split hold when we remove operators the
# solver can never actually select?
#
# Uses df_brains_valid which has per-call, per-operator agent_status
# (available vs imminent) from historical production data.
# =============================================================================

from collections import defaultdict

# ── 1. Build available-only score lookup ──────────────────────────────────────
# Filter score_lookup to include only (call, op) pairs where the operator
# was marked 'available' (not 'imminent') in historical brains requests.
_avail_pairs = set(
    df_brains_valid[df_brains_valid['agent_status'] == 'available'][
        ['call_id', 'agent_username']
    ].apply(tuple, axis=1)
)

_avail_score_lookup = {
    (cid, op): cij for (cid, op), cij in score_lookup.items()
    if (cid, op) in _avail_pairs
}

# Diagnostic only — NOT fed to the simulator (run_scenario_group
# always uses the full score_lookup; imminent exclusion is handled at runtime).
avail_only_score_lookup = _avail_score_lookup

print(f"📐 AVAILABLE-ONLY SCORE LOOKUP")
print(f"  Full score_lookup:      {len(score_lookup):>10,} pairs")
print(f"  Available-only pairs:   {len(_avail_pairs):>10,} pairs (from df_brains_valid)")
print(f"  Matched in lookup:      {len(_avail_score_lookup):>10,} pairs")
print(f"  Retention:              {len(_avail_score_lookup)/len(score_lookup)*100:.1f}%")

# ── 2. Build available-only call profiles ─────────────────────────────────────
_avail_cij_by_call = defaultdict(list)
_avail_ops_by_call = defaultdict(list)
for (cid, op), cij in _avail_score_lookup.items():
    _avail_cij_by_call[cid].append(cij)
    _avail_ops_by_call[cid].append(op)

_avail_profiles = {}
for cid, cij_vals in _avail_cij_by_call.items():
    if len(cij_vals) < 2:  # need ≥2 ops for spread
        continue
    vals = np.array(cij_vals)
    _avail_profiles[cid] = {
        'max_cij': vals.max(),
        'min_cij': vals.min(),
        'spread': vals.max() - vals.min(),
        'mean_cij': vals.mean(),
        'n_ops': len(vals),
    }

_df_avail = pd.DataFrame(_avail_profiles.values(), index=_avail_profiles.keys())
_df_avail.index.name = 'call_id'

print(f"\n📊 AVAILABLE-ONLY CALL PROFILES:")
print(f"  Calls with ≥2 available ops scored:  {len(_df_avail):,}")
print(f"  (vs {len(df_prof):,} calls in full lookup with ≥5 ops)")

# ── 3. Recompute T1 with available-only criteria ─────────────────────────────
print(f"\n{'=' * 80}")
print(f"  T1 RE-SEGMENTATION: available-only vs full lookup")
print(f"{'=' * 80}")

# Use same thresholds as original
_t1_full = set(df_prof[(df_prof['max_cij'] > MAX_CIJ_THRESH) &
                        (df_prof['spread'] > SPREAD_THRESH)].index)
_t1_avail = set(_df_avail[(_df_avail['max_cij'] > MAX_CIJ_THRESH) &
                           (_df_avail['spread'] > SPREAD_THRESH)].index)

# Persist for use by L2 scenarios with available-only T1 classification
tier1_calls_avail_only = _t1_avail

# Calls in both
_t1_both = _t1_full & _t1_avail
_t1_full_only = _t1_full - _t1_avail       # T1 in full but NOT in available-only
_t1_avail_only = _t1_avail - _t1_full       # T1 in available-only but NOT in full (unlikely)

print(f"\n  Thresholds: max_cij > {MAX_CIJ_THRESH}, spread > {SPREAD_THRESH}")
print(f"  ┌────────────────────────────────────────────────────────────────┐")
print(f"  │  {'Criterion':<28s} {'N calls':>8s} {'% of calls':>10s}         │")
print(f"  ├────────────────────────────────────────────────────────────────┤")
print(f"  │  T1 (full lookup, ≥5 ops)     {len(_t1_full):>8,} {len(_t1_full)/len(df_prof)*100:>9.1f}%         │")
print(f"  │  T1 (available-only, ≥2 ops)  {len(_t1_avail):>8,} {len(_t1_avail)/len(_df_avail)*100:>9.1f}%         │")
print(f"  ├────────────────────────────────────────────────────────────────┤")
print(f"  │  In BOTH                      {len(_t1_both):>8,}                      │")
print(f"  │  Full-only (lost without imm) {len(_t1_full_only):>8,}  ← inflation     │")
print(f"  │  Avail-only (gained)          {len(_t1_avail_only):>8,}                      │")
print(f"  └────────────────────────────────────────────────────────────────┘")

_retention_pct = len(_t1_both) / len(_t1_full) * 100 if _t1_full else 0
print(f"\n  T1 retention: {_retention_pct:.1f}% of full-lookup T1 calls survive available-only filter")

# ── 4. Spread comparison ─────────────────────────────────────────────────────
print(f"\n{'=' * 80}")
print(f"  SPREAD COMPARISON: full lookup vs available-only")
print(f"{'=' * 80}")

# For calls in T1-full, compare their spread in full vs available-only
_common_t1 = list(_t1_full & set(_df_avail.index))
if _common_t1:
    _full_spreads = df_prof.loc[_common_t1, 'spread'].values
    _avail_spreads = _df_avail.loc[_common_t1, 'spread'].values

    print(f"\n  {len(_common_t1):,} T1 calls with both full and available-only profiles:")
    print(f"  {'':>20s} {'Full lookup':>14s} {'Available-only':>16s} {'Δ':>8s}")
    print(f"  {'-' * 62}")
    print(f"  {'Mean spread':>20s} {np.mean(_full_spreads):>13.4f}pp {np.mean(_avail_spreads):>15.4f}pp {np.mean(_avail_spreads)-np.mean(_full_spreads):>+7.4f}")
    print(f"  {'Median spread':>20s} {np.median(_full_spreads):>13.4f}pp {np.median(_avail_spreads):>15.4f}pp {np.median(_avail_spreads)-np.median(_full_spreads):>+7.4f}")
    print(f"  {'P25 spread':>20s} {np.percentile(_full_spreads, 25):>13.4f}pp {np.percentile(_avail_spreads, 25):>15.4f}pp")
    print(f"  {'P75 spread':>20s} {np.percentile(_full_spreads, 75):>13.4f}pp {np.percentile(_avail_spreads, 75):>15.4f}pp")

    # Fraction with spread ≥ 8pp (decision gate)
    _full_8pp = np.sum(_full_spreads >= 0.08) / len(_full_spreads) * 100
    _avail_8pp = np.sum(_avail_spreads >= 0.08) / len(_avail_spreads) * 100
    print(f"\n  Spread ≥ 8pp (decision gate):")
    print(f"    Full lookup:      {_full_8pp:.1f}%")
    print(f"    Available-only:   {_avail_8pp:.1f}%")

# ── 5. Operator gap on available-only T1 ──────────────────────────────────────
print(f"\n{'=' * 80}")
print(f"  OPERATOR GAP: available-only T1")
print(f"{'=' * 80}")

_avail_op_profiles = defaultdict(list)
for (cid, op), cij in _avail_score_lookup.items():
    if cid in _t1_avail:
        _avail_op_profiles[op].append(cij)

_avail_op_stats = {}
for op, cij_vals in _avail_op_profiles.items():
    if len(cij_vals) >= 5:  # operators with enough T1 exposure
        vals = np.array(cij_vals)
        _avail_op_stats[op] = {'mean_cij_t1': vals.mean(), 'n': len(vals)}

_df_avail_ops = pd.DataFrame(_avail_op_stats.values(), index=_avail_op_stats.keys())
_df_avail_ops = _df_avail_ops.sort_values('mean_cij_t1')

if len(_df_avail_ops) >= 20:
    _avail_best20 = _df_avail_ops.head(20)['mean_cij_t1'].mean()
    _avail_worst20 = _df_avail_ops.tail(20)['mean_cij_t1'].mean()
    _avail_gap = _avail_worst20 - _avail_best20

    print(f"\n  Available-only T1 ({len(_t1_avail):,} calls, {len(_df_avail_ops):,} ops with ≥5 exposures):")
    print(f"  {'':>22s} {'Full lookup':>14s} {'Available-only':>16s}")
    print(f"  {'-' * 55}")
    print(f"  {'Best-20 avg cij':>22s} {best_20_mean:>14.4f} {_avail_best20:>16.4f}")
    print(f"  {'Worst-20 avg cij':>22s} {worst_20_mean:>14.4f} {_avail_worst20:>16.4f}")
    print(f"  {'Gap (pp)':>22s} {(worst_20_mean-best_20_mean)*100:>13.1f}pp {_avail_gap*100:>15.1f}pp")
else:
    print(f"  Only {len(_df_avail_ops)} operators with ≥5 available T1 exposures")
    _avail_gap = 0

# ── 6. Per-tick slate size reality check ──────────────────────────────────────
print(f"\n{'=' * 80}")
print(f"  PER-TICK AVAILABLE OPERATORS (realistic slate sizes)")
print(f"{'=' * 80}")

# How many AVAILABLE operators does each T1 call have?
_t1_avail_ops_per_call = []
for cid in _t1_avail:
    n = len(_avail_ops_by_call.get(cid, []))
    _t1_avail_ops_per_call.append(n)
_t1_avail_ops_arr = np.array(_t1_avail_ops_per_call)

print(f"\n  T1 calls (available-only): {len(_t1_avail):,}")
print(f"  Available operators per T1 call:")
print(f"    Mean:   {_t1_avail_ops_arr.mean():.1f}")
print(f"    Median: {np.median(_t1_avail_ops_arr):.0f}")
print(f"    P25:    {np.percentile(_t1_avail_ops_arr, 25):.0f}")
print(f"    P75:    {np.percentile(_t1_avail_ops_arr, 75):.0f}")
print(f"    Min:    {_t1_avail_ops_arr.min()}")
print(f"    Max:    {_t1_avail_ops_arr.max()}")
_few = np.sum(_t1_avail_ops_arr <= 3)
print(f"    ≤3 operators: {_few:,} calls ({_few/len(_t1_avail)*100:.1f}%)")

# ── 7. DECISION GATE ─────────────────────────────────────────────────────────
print(f"\n{'=' * 80}")
print(f"  🚦 DECISION GATE: Imminent Inflation Assessment")
print(f"{'=' * 80}")

_avail_mean_spread = np.mean(_avail_spreads) if _common_t1 else 0
_gate_threshold = 0.08  # 8pp

if _avail_mean_spread >= _gate_threshold:
    print(f"""
  ✅ PASS: Available-only spread = {_avail_mean_spread*100:.1f}pp ≥ {_gate_threshold*100:.0f}pp threshold
  
  Imminent inflation is MINOR — the T1 segmentation holds when restricted
  to available operators. The operator gap is still meaningful.
  
  → PROCEED to Layer 2 (FO Parameter Sweep)
  → T1 classification using full lookup is acceptable
  → L2 FO sweep runs inside the two-tier structure (solve_tiered)
""")
else:
    print(f"""
  ⚠️  FAIL: Available-only spread = {_avail_mean_spread*100:.1f}pp < {_gate_threshold*100:.0f}pp threshold
  
  Imminent inflation is SIGNIFICANT — the T1 spread is driven by operators
  the solver can never select. Two-tier routing on available-only slates may
  not have enough differentiation to improve matching.
  
  → Lower T1 thresholds to capture calls with available-only spread
  → OR: skip two-tier structural changes (Layer 3) and focus on FO parameter tuning (Layer 2)
""")

print(f"  T1 retention:          {_retention_pct:.1f}% ({len(_t1_both):,}/{len(_t1_full):,})")
print(f"  Available-only spread: {_avail_mean_spread*100:.1f}pp (full: {np.mean(_full_spreads)*100:.1f}pp)")
if len(_df_avail_ops) >= 20:
    print(f"  Operator gap:          {_avail_gap*100:.1f}pp (full: {(worst_20_mean-best_20_mean)*100:.1f}pp)")
print(f"  Median avail ops/call: {np.median(_t1_avail_ops_arr):.0f}")


<a id="layer-1-findings"></a>
### Layer 1 — Findings & Decision

Summarises the imminent-inflation diagnostic from the cell above:
compares T1 call counts, mean operator spread, and operator gap between the full score lookup
and the available-only lookup. The **decision gate** checks whether available-only mean spread
is sufficient to proceed with two-tier FO tuning (Layer 2).

Run the cell above to compute the metrics. Outputs consumed here:
- : operator-level Tier 1 stats (available-only spread)
- , : Tier 1 sets under each lookup


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LAYER 1 EXECUTION — S1 (Available-Only)
# ═══════════════════════════════════════════════════════════════════════════════
# Prerequisites:
#   - Cells 17, 23, 34, 36 must have been re-run
#
# S1: EXCLUDE_IMMINENT_FROM_SLATE — uses full score_lookup, but the simulation
# only assigns to operators with status="available" (runtime filter).
# Imminent operators (busy, about to be free) are naturally excluded by
# get_available_operators() at each tick.
# ═══════════════════════════════════════════════════════════════════════════════

layer_1_results = run_scenario_group(["L1 S1 Avail-Only"])

<a id="layer-2"></a>
---
## Layer 2 — FO Parameter Sweep (S2a–S2c)

Sweep three FO parameters **inside** the two-tier structure:
- **S2a β_G sweep** {0.35, 0.50, 0.65, 0.80}: routing intelligence emphasis
- **S2b κ sweep** {−0.30, −0.25, −0.20, −0.15}: gate timing
- **S2c α sweep** {1.5, 2.0, 3.0, 4.0}: ramp shape

All share the same `_slate_base` config (USE_TWO_TIER=True, LIFT_FRAC=0.50).
Parameter-only experiments — no structural solver changes needed.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LAYER 2 EXECUTION — FO Parameter Sweep (S2a–S2c)
# ═══════════════════════════════════════════════════════════════════════════════
# Prerequisites:
#   - Cell 22 (experiments dict) and Cell 23 (runner) must have been run
#   - L0 and L1 results in results_comparison (for comparison)
#
# S2a: β_G sweep — how much weight on routing intelligence (G vs P)
# S2b: κ sweep  — gate timing (when does FO become positive?)
# S2c: α sweep  — ramp shape (how steep is the urgency curve?)
# ═══════════════════════════════════════════════════════════════════════════════

layer_2_names = [k for k in experiments if k.startswith("L2")]
print(f"Layer 2 scenarios to run: {len(layer_2_names)}")
for n in layer_2_names:
    print(f"  • {n}")
print()

layer_2_results = run_scenario_group(layer_2_names)

<a id="layer-3"></a>
---
## Layer 3 — Two-Tier Structural Variants (S3a–S3c)

Layer 2 established the ceiling of parameter-only tuning. Now test **structural** changes via `solve_tiered()`:

- **S3a Soft Reserve**: Valuable operators may serve T2 when no T1 calls are waiting. Maximises throughput.
- **S3b Strict Reserve**: Valuable operators **never** serve T2 calls. Guarantees T1 availability at the cost of idle capacity.
- **S3c Gate Sweep**: LIFT_FRAC ∈ {30%, 40%, 50%, 60%, 70%}. Controls how selective the operator gate is within T1.

All inherit the `_slate_base` config: `USE_TWO_TIER=True`, `USE_SLATE_LIFT=True`, T1 calls from Cell 22a, valuable operators from Cell 22b.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LAYER 3 EXECUTION — Two-Tier Structural Variants (S3a–S3c)
# ═══════════════════════════════════════════════════════════════════════════════
# Prerequisites:
#   - Cell 22 (experiments dict) and Cell 23 (runner) must have been run
#   - L0, L1, L2 results in results_comparison (for comparison)
#
# S3a: Soft Reserve — valuable ops may serve T2 when no T1 waiting
# S3b: Strict Reserve — valuable ops NEVER serve T2
# S3c: Gate Sweep — LIFT_FRAC ∈ {30%, 40%, 50%, 60%, 70%}
# ═══════════════════════════════════════════════════════════════════════════════

layer_3_names = [k for k in experiments if k.startswith("L3")]
print(f"Layer 3 scenarios to run: {len(layer_3_names)}")
for n in layer_3_names:
    print(f"  • {n}")
print()

layer_3_results = run_scenario_group(layer_3_names)

<a id="comparison-table"></a>
## Scenario Comparison Table

In [ ]:
# ═══ L0/L1/L2/L3 COMPACT RESULTS ══════════════════════════════════════════════
import json as _json
_rows = []
_s0_nr = None
for name in sorted(results_comparison.keys(), key=lambda x: (x.split()[0], x)):
    r = results_comparison[name]
    n_qp = len(r['assignments'])
    n_fifo = len(r['fifo_assignments'])
    n_lost = len(r['unserved_calls'])
    n_total = n_qp + n_fifo + n_lost
    svc = (n_qp + n_fifo) / n_total * 100 if n_total else 0
    waits = [a.waiting_time for a in r['assignments']]
    med_wait = float(np.median(waits)) if waits else 0
    _qp_cij = [a.churn_ij for a in r['assignments'] if a.churn_ij is not None]
    _fifo_cij = [a.churn_ij for a in r['fifo_assignments'] if a.churn_ij is not None]
    all_churns = np.clip(np.array(_qp_cij + _fifo_cij) - CALIBRATED_CHURN_BIAS, 0, 1)
    nr_pct = float(all_churns.mean() * 100) if len(all_churns) > 0 else 0
    if _s0_nr is None and "S0 Baseline" in name:
        _s0_nr = nr_pct
    sla = sum(1 for w in waits if w <= 180) / len(waits) * 100 if waits else 0
    _rows.append({"name": name, "qp": n_qp, "fifo": n_fifo, "lost": n_lost,
                   "svc": round(svc,1), "wait": round(med_wait), "nr": round(nr_pct,2),
                   "dnr": round(nr_pct - _s0_nr, 2), "sla": round(sla,1)})

with open("/tmp/all_results.json", "w") as f:
    _json.dump(_rows, f, indent=2)
print(f"Wrote {len(_rows)} rows to /tmp/all_results.json")

---
# Part VI — Robustness Checks


<a id="multi-seed"></a>
## Multi-Seed Replication — Statistical Significance

Single-seed simulations produce point estimates. The stochastic components
(occupancy model, delivery failures, callback deflection, ACW, pipeline latency)
mean each run is one draw from a distribution.

**Approach**: Run N independent replications with different seeds. Report:
- Per-seed aggregate KPIs (NR%, SLA, QP, FIFO, Lost)
- Per-day KPIs per seed (for paired tests)
- Mean ± 95% CI for all metrics
- Paired Wilcoxon signed-rank test for scenario comparisons

In [ ]:
# =============================================================================
# CELL — MULTI-SEED RUNNER (parallelized)                     [Statistical Significance]
# =============================================================================
# INPUT:
#   - experiments dict, day_setups, run_simulation, score_lookup, churn_i_lookup
#   - CALIBRATED_* parameters, fallback_churn_ij
#
# DOES:
#   1. Defines run_multiseed(): runs N replications per scenario with different seeds
#   2. Parallelizes (day × seed) pairs across CPU cores using multiprocessing (fork)
#   3. Collects per-day per-seed KPIs for paired statistical tests
#
# SPEEDUP:
#   Sequential: N_SEEDS × 21 days × ~3.5s = ~370s (5 seeds)
#   Parallel (4 cores): ~370/4 ≈ 93s
#
# OUTPUT:
#   - run_multiseed() function
#   - multiseed_results: dict of {scenario: {seeds, aggregate, per_day}} when run
# =============================================================================

import time as _time
import multiprocessing as _mp
from scipy import stats as _stats

N_SEEDS = 5  # 5 seeds × 21 days = 105 paired observations — sufficient for Wilcoxon
SEED_LIST = list(range(42, 42 + N_SEEDS))
_N_WORKERS = min(4, _mp.cpu_count() or 1)  # cap at 4 to avoid memory pressure


def _compute_day_kpis(state, day_str):
    """Extract per-day KPIs from a single simulation state."""
    assigns = state.assignments
    fifo = state.fifo_assignments
    lost = state.unserved_calls

    n_qp = len(assigns)
    n_fifo = len(fifo)
    n_lost = len(lost)
    n_total = n_qp + n_fifo + n_lost

    waits = [a.waiting_time for a in assigns]

    # NR% — bias-corrected, QP + FIFO
    qp_cij = np.clip(
        np.array([a.churn_ij for a in assigns if a.churn_ij is not None]) - CALIBRATED_CHURN_BIAS, 0, 1
    )
    fifo_cij = np.clip(
        np.array([a.churn_ij for a in fifo if a.churn_ij is not None]) - CALIBRATED_CHURN_BIAS, 0, 1
    )
    n_served = len(qp_cij) + len(fifo_cij)
    sum_churn = qp_cij.sum() + fifo_cij.sum()
    nr_pct = sum_churn / n_served * 100 if n_served > 0 else 0
    e_nr_count = round(sum_churn) if n_served > 0 else 0

    # Per-channel NR%
    n_served_qp = len(qp_cij)
    n_served_fifo = len(fifo_cij)
    nr_pct_qp = float(qp_cij.sum() / n_served_qp * 100) if n_served_qp > 0 else 0
    nr_pct_fifo = float(fifo_cij.sum() / n_served_fifo * 100) if n_served_fifo > 0 else 0

    sla = sum(1 for w in waits if w <= 180) / len(waits) * 100 if waits else 0
    med_wait = float(np.median(waits)) if waits else 0

    return {
        'day': day_str,
        'qp': n_qp,
        'fifo': n_fifo,
        'lost': n_lost,
        'svc_pct': (n_qp + n_fifo) / n_total * 100 if n_total > 0 else 0,
        'nr_pct': float(nr_pct),
        'nr_pct_qp': nr_pct_qp,
        'nr_pct_fifo': nr_pct_fifo,
        'e_nr': e_nr_count,
        'sla_pct': float(sla),
        'med_wait': float(med_wait),
        'n_served': n_served,
        'n_served_qp': n_served_qp,
        'n_served_fifo': n_served_fifo,
    }


def _sim_day_worker(args):
    """Worker: simulate one (day, seed, scenario) and return KPIs dict.

    Uses fork — inherits all globals (score_lookup, day_setups, etc.)
    from the parent process via copy-on-write. Only the small args
    tuple needs pickling.
    """
    day_str, seed, exp_name = args
    config = experiments[exp_name]
    setup = day_setups[day_str]

    state = run_simulation(
        df_call_events=df_call_events,
        df_operators=setup['df_operators'],
        score_lookup=score_lookup,
        churn_i_lookup=churn_i_lookup,
        fo_config=config,
        tick_interval=CALIBRATED_TICK,
        max_wait_time=180,
        sim_day=day_str,
        active_windows=setup['active_windows'],
        gap_busy_fraction=0.7,
        hourly_occupancy=setup['hourly_occupancy'],
        operator_shifts=setup['operator_shifts'],
        fallback_churn_ij=fallback_churn_ij,
        pipeline_latency=CALIBRATED_LATENCY,
        pipeline_latency_std=CALIBRATED_LATENCY_STD,
        nqp_mean_task_s=CALIBRATED_NQP_TASK_S,
        fifo_latency=CALIBRATED_FIFO_LATENCY,
        fifo_block_operators=CALIBRATED_FIFO_BLOCK,
        enable_fifo=True,
        delivery_failure_rate=CALIBRATED_DELIVERY_FAILURE_RATE,
        callback_deflection_time=CALIBRATED_CALLBACK_DEFLECTION_TIME,
        callback_deflection_rate=CALIBRATED_CALLBACK_DEFLECTION_RATE,
        acw_mean_s=CALIBRATED_ACW_MEAN_S,
        seed=seed,
        verbose=False,
    )
    kpis = _compute_day_kpis(state, day_str)
    kpis['seed'] = seed
    return kpis


def run_multiseed(scenario_names, n_seeds=N_SEEDS, seeds=None, n_workers=_N_WORKERS):
    """Run scenarios with multiple seeds, collecting per-day per-seed KPIs.

    Parallelizes (day × seed) pairs across n_workers processes using fork.

    Args:
        scenario_names: list of scenario keys from `experiments` dict.
        n_seeds: number of replications (ignored if seeds is provided).
        seeds: explicit list of seeds (default: SEED_LIST[:n_seeds]).
        n_workers: number of parallel workers (0 = sequential fallback).

    Returns:
        dict of {scenario_name: {
            'per_day_per_seed': list of dicts (day, seed, kpis...),
            'aggregate_per_seed': list of dicts (seed, kpis...),
        }}
    """
    if seeds is None:
        seeds = SEED_LIST[:n_seeds]

    _run_exps = {k: experiments[k] for k in scenario_names if k in experiments}
    _n_runs = len(_run_exps) * len(seeds) * len(day_setups)
    _mode = f"parallel ({n_workers} workers)" if n_workers > 0 else "sequential"
    print(f"🎲 MULTI-SEED RUNNER: {len(_run_exps)} scenarios × {len(seeds)} seeds "
          f"× {len(day_setups)} days = {_n_runs:,} runs  [{_mode}]\n")

    multiseed_results = {}

    for exp_name in _run_exps:
        t0 = _time.time()
        print(f"{'─'*60}")
        print(f"  {exp_name}  ({len(seeds)} seeds)")
        print(f"{'─'*60}")

        # Build task list: all (day, seed) pairs for this scenario
        tasks = [
            (day_str, seed, exp_name)
            for seed in seeds
            for day_str in sorted(day_setups.keys())
        ]

        # Execute — parallel or sequential
        if n_workers > 0:
            ctx = _mp.get_context('fork')
            with ctx.Pool(n_workers) as pool:
                all_kpis = pool.map(_sim_day_worker, tasks)
        else:
            all_kpis = [_sim_day_worker(t) for t in tasks]

        # Group results by seed
        per_day_per_seed = all_kpis  # flat list with 'seed' and 'day' keys
        aggregate_per_seed = []

        for seed in seeds:
            day_kpis = [k for k in all_kpis if k['seed'] == seed]
            _qp = sum(d['qp'] for d in day_kpis)
            _fifo = sum(d['fifo'] for d in day_kpis)
            _lost = sum(d['lost'] for d in day_kpis)
            _total = _qp + _fifo + _lost
            _n_served = sum(d['n_served'] for d in day_kpis)
            _sum_churn = sum(d['nr_pct'] * d['n_served'] / 100 for d in day_kpis)
            _nr_agg = _sum_churn / _n_served * 100 if _n_served > 0 else 0
            _e_nr_agg = round(_sum_churn)

            # Per-channel NR% aggregation (volume-weighted across days)
            _n_served_qp = sum(d['n_served_qp'] for d in day_kpis)
            _sum_churn_qp = sum(d['nr_pct_qp'] * d['n_served_qp'] / 100 for d in day_kpis)
            _nr_qp_agg = _sum_churn_qp / _n_served_qp * 100 if _n_served_qp > 0 else 0
            _n_served_fifo = sum(d['n_served_fifo'] for d in day_kpis)
            _sum_churn_fifo = sum(d['nr_pct_fifo'] * d['n_served_fifo'] / 100 for d in day_kpis)
            _nr_fifo_agg = _sum_churn_fifo / _n_served_fifo * 100 if _n_served_fifo > 0 else 0

            _sla_num = sum(d['sla_pct'] * d['qp'] / 100 for d in day_kpis)
            _sla_agg = _sla_num / _qp * 100 if _qp > 0 else 0
            _med_wait_agg = np.average(
                [d['med_wait'] for d in day_kpis],
                weights=[d['qp'] for d in day_kpis]
            ) if _qp > 0 else 0

            aggregate_per_seed.append({
                'seed': seed,
                'qp': _qp, 'fifo': _fifo, 'lost': _lost,
                'svc_pct': (_qp + _fifo) / _total * 100 if _total > 0 else 0,
                'nr_pct': float(_nr_agg),
                'nr_pct_qp': float(_nr_qp_agg),
                'nr_pct_fifo': float(_nr_fifo_agg),
                'e_nr': _e_nr_agg,
                'sla_pct': float(_sla_agg),
                'med_wait': float(_med_wait_agg),
            })

        elapsed = _time.time() - t0
        _nr_vals = [a['nr_pct'] for a in aggregate_per_seed]
        _sla_vals = [a['sla_pct'] for a in aggregate_per_seed]
        print(f"  NR%:  {np.mean(_nr_vals):.2f} ± {np.std(_nr_vals, ddof=1):.3f}  "
              f"(range {np.min(_nr_vals):.2f}–{np.max(_nr_vals):.2f})")
        print(f"  SLA:  {np.mean(_sla_vals):.1f} ± {np.std(_sla_vals, ddof=1):.2f}%  "
              f"(range {np.min(_sla_vals):.1f}–{np.max(_sla_vals):.1f}%)")
        print(f"  ⏱️  {elapsed:.0f}s ({elapsed/len(seeds):.0f}s/seed)\n")

        multiseed_results[exp_name] = {
            'per_day_per_seed': per_day_per_seed,
            'aggregate_per_seed': aggregate_per_seed,
        }

    return multiseed_results


print(f"✅ run_multiseed() ready. N_SEEDS={N_SEEDS}, seeds={SEED_LIST}, workers={_N_WORKERS}")

In [ ]:
# =============================================================================
# CELL — MULTI-SEED ANALYSIS: CI TABLE & STATISTICAL TESTS
# =============================================================================
# INPUT:  multiseed_results dict from run_multiseed()
# DOES:   1. Builds table with mean ± 95% CI for each KPI per scenario
#          2. Paired Wilcoxon signed-rank test between each scenario and baseline
# OUTPUT: df_multiseed_ci DataFrame, printed table
# =============================================================================

KPI_COLS = ['nr_pct', 'nr_pct_qp', 'nr_pct_fifo', 'sla_pct', 'qp', 'fifo', 'lost', 'svc_pct', 'med_wait']
KPI_LABELS = {'nr_pct': 'NR%', 'nr_pct_qp': 'NR% QP', 'nr_pct_fifo': 'NR% FIFO',
              'sla_pct': 'SLA%', 'qp': 'QP', 'fifo': 'FIFO',
              'lost': 'Lost', 'svc_pct': 'Svc%', 'med_wait': 'Med Wait(s)'}


def build_ci_table(multiseed_results, baseline_key=None):
    """Build a DataFrame with mean ± 95% CI for each scenario KPI.

    If baseline_key is set, adds paired Wilcoxon p-value columns for NR% and SLA.
    """
    rows = []
    baseline_nr = None
    baseline_sla = None

    # If baseline is specified, extract its per-day KPIs
    if baseline_key and baseline_key in multiseed_results:
        bdata = multiseed_results[baseline_key]['aggregate_per_seed']
        baseline_nr = [d['nr_pct'] for d in bdata]
        baseline_sla = [d['sla_pct'] for d in bdata]

    for scen, data in multiseed_results.items():
        agg = data['aggregate_per_seed']
        row = {'Scenario': scen, 'N': len(agg)}

        for kpi in KPI_COLS:
            vals = np.array([d[kpi] for d in agg])
            mean = vals.mean()
            se = vals.std(ddof=1) / np.sqrt(len(vals))
            ci95 = _stats.t.ppf(0.975, df=len(vals)-1) * se
            row[f'{KPI_LABELS[kpi]} mean'] = round(mean, 3)
            row[f'{KPI_LABELS[kpi]} ±CI'] = round(ci95, 3)

        # Paired Wilcoxon test vs baseline
        if baseline_nr is not None and scen != baseline_key:
            scen_nr = [d['nr_pct'] for d in agg]
            scen_sla = [d['sla_pct'] for d in agg]

            try:
                _, p_nr = _stats.wilcoxon(baseline_nr, scen_nr, alternative='two-sided')
            except ValueError:
                p_nr = 1.0  # identical arrays

            try:
                _, p_sla = _stats.wilcoxon(baseline_sla, scen_sla, alternative='two-sided')
            except ValueError:
                p_sla = 1.0

            row['p(NR%)'] = round(p_nr, 4)
            row['p(SLA)'] = round(p_sla, 4)
        elif scen == baseline_key:
            row['p(NR%)'] = '—'
            row['p(SLA)'] = '—'

        rows.append(row)

    df_ci = pd.DataFrame(rows)
    return df_ci


def print_ci_summary(multiseed_results, baseline_key=None):
    """Pretty-print the CI summary table."""
    df_ci = build_ci_table(multiseed_results, baseline_key)

    # Compact display
    display_cols = ['Scenario', 'N', 'NR% mean', 'NR% ±CI',
                    'NR% QP mean', 'NR% QP ±CI', 'NR% FIFO mean', 'NR% FIFO ±CI',
                    'SLA% mean', 'SLA% ±CI',
                    'QP mean', 'FIFO mean', 'Lost mean', 'Med Wait(s) mean']
    if 'p(NR%)' in df_ci.columns:
        display_cols += ['p(NR%)', 'p(SLA)']

    with pd.option_context('display.max_columns', None, 'display.width', None,
                           'display.float_format', '{:.3f}'.format):
        display(df_ci[display_cols])

    # Effect-size summary
    if baseline_key and baseline_key in multiseed_results:
        print(f"\n{'─'*60}")
        print(f"Effect-size summary vs. {baseline_key}:")
        bdata = multiseed_results[baseline_key]['aggregate_per_seed']
        b_nr = np.array([d['nr_pct'] for d in bdata])
        for scen, data in multiseed_results.items():
            if scen == baseline_key:
                continue
            s_nr = np.array([d['nr_pct'] for d in data['aggregate_per_seed']])
            delta = s_nr.mean() - b_nr.mean()
            pooled_std = np.sqrt((b_nr.var(ddof=1) + s_nr.var(ddof=1)) / 2)
            cohens_d = delta / pooled_std if pooled_std > 0 else 0
            print(f"  {scen}: ΔNR% = {delta:+.3f}pp, Cohen's d = {cohens_d:+.3f}")

    return df_ci


print("✅ build_ci_table() and print_ci_summary() ready.")

In [ ]:
# =============================================================================
# CELL — FULL MULTI-SEED RUN (all scenarios)
# =============================================================================
# Run ALL scenarios with N_SEEDS replications to get proper confidence
# intervals and paired statistical tests. Single-seed results are point
# estimates — multi-seed separates signal from simulator noise.
#
# Expected runtime: ~22 scenarios × 5 seeds × 21 days ÷ 4 workers
#                   ≈ 22 × 93s ≈ 34 min
# =============================================================================

all_scenario_names = list(experiments.keys())
print(f"🎲 Running ALL {len(all_scenario_names)} scenarios with {N_SEEDS} seeds each\n")

baseline_name = "L0 S0 Baseline"
multiseed_results = run_multiseed(all_scenario_names, n_seeds=N_SEEDS)

# Display full CI table with Wilcoxon tests vs baseline
df_ci_all = print_ci_summary(multiseed_results, baseline_key=baseline_name)

<a id="bias-sensitivity"></a>
### Bias Sensitivity Analysis (Bootstrap)

The NR% metric relies on subtracting `CALIBRATED_CHURN_BIAS = 2.88pp` from all model-predicted $c_{ij}$.
This bias was estimated from the calibration pairs (see churn calibration output for count) where both a `churn_ij` prediction
and a ground-truth `RESOL_MOT_DSC` outcome exist.

**Concern:** If the bias is not constant across churn strata (e.g., 4pp for high-risk, 1pp for low-risk),
scenario comparisons with different QP/FIFO volume splits may have different effective biases.

**Test:** Bootstrap the calibration pairs 1,000× → get 95% CI for the bias → recompute NR% at the
extreme bias values → verify that scenario ΔNR% rankings are invariant.

In [ ]:
# =============================================================================
# CELL — BIAS SENSITIVITY BOOTSTRAP                        [Robustness Check]
# =============================================================================
# INPUT:
#   - df_calls_qplanner: historical calls with RESOL_MOT_DSC outcomes
#   - full_score_lookup: (call_id, operator) → churn_ij model predictions
#   - VALID_OUTCOMES, CALIBRATED_CHURN_BIAS
#   - multiseed_results: baseline multi-seed results (from cell above)
#
# DOES:
#   1. Reconstructs the calibration pairs (predicted churn_ij, actual outcome)
#   2. Bootstrap resamples 1,000× to get 95% CI for the bias
#   3. Recomputes NR% for baseline at extreme bias values
#   4. Shows ΔNR% is insensitive to bias uncertainty
#
# OUTPUT:
#   - bias_bootstrap: array of 1,000 bootstrapped bias estimates
#   - bias_ci: (lower, upper) 95% CI
# =============================================================================

# ── 1. Reconstruct calibration pairs ──────────────────────────────────────────
# Match historical QP assignments with model predictions and outcomes
_calib_df = df_calls_qplanner.drop_duplicates(subset='CALL_ID').copy()
_calib_assigned = _calib_df[
    (_calib_df['QUEUE_PLANNER_RESULT_DESC'] == 'QPlanner-AgentAvailable')
    & (_calib_df['RESOL_MOT_DSC'].isin(VALID_OUTCOMES))
    & (_calib_df['agent_username'].notna())
].copy()

# Look up churn_ij prediction for each (call_id, operator) pair
_preds = []
_actuals = []
for _, row in _calib_assigned.iterrows():
    key = (row['CALL_ID'], row['agent_username'])
    if key in full_score_lookup:
        _preds.append(full_score_lookup[key])
        _actuals.append(1.0 if row['RESOL_MOT_DSC'] == 'N RECUPERADO' else 0.0)

_preds = np.array(_preds)
_actuals = np.array(_actuals)
_n_pairs = len(_preds)

print(f"═══ BIAS SENSITIVITY BOOTSTRAP ═══")
print(f"  Calibration pairs: {_n_pairs:,}")
print(f"  Predicted mean:    {_preds.mean()*100:.2f}%")
print(f"  Actual rate:       {_actuals.mean()*100:.2f}%")
print(f"  Point bias:        {(_preds.mean() - _actuals.mean())*100:.2f}pp")
print(f"  (Reference: CALIBRATED_CHURN_BIAS = {CALIBRATED_CHURN_BIAS*100:.2f}pp)\n")

# ── 2. Bootstrap ─────────────────────────────────────────────────────────────
B = 1_000
rng = np.random.RandomState(42)
bias_bootstrap = np.empty(B)
for b in range(B):
    idx = rng.randint(0, _n_pairs, size=_n_pairs)
    bias_bootstrap[b] = _preds[idx].mean() - _actuals[idx].mean()

bias_ci = (np.percentile(bias_bootstrap, 2.5), np.percentile(bias_bootstrap, 97.5))
bias_mean = bias_bootstrap.mean()
bias_std = bias_bootstrap.std(ddof=1)

print(f"  Bootstrap (B={B:,}):")
print(f"    Bias mean:  {bias_mean*100:.2f}pp")
print(f"    Bias std:   {bias_std*100:.3f}pp")
print(f"    95% CI:     [{bias_ci[0]*100:.2f}pp, {bias_ci[1]*100:.2f}pp]")
print(f"    Range:      {(bias_ci[1]-bias_ci[0])*100:.2f}pp wide\n")

# ── 3. NR% sensitivity at extreme bias values ────────────────────────────────
# Recompute NR% for the baseline across seeds at bias_low and bias_high
_bias_low, _bias_high = bias_ci
_bias_point = CALIBRATED_CHURN_BIAS

print(f"  NR% sensitivity for '{baseline_name}':")
print(f"  {'Bias':>12s}  {'NR% mean':>10s}  {'Shift':>8s}")
print(f"  {'─'*12}  {'─'*10}  {'─'*8}")

for label, bias_val in [("Lower CI", _bias_low), ("Point", _bias_point), ("Upper CI", _bias_high)]:
    seed_nrs = []
    for agg in multiseed_results[baseline_name]['aggregate_per_seed']:
        # Recompute NR% with different bias: original used CALIBRATED_CHURN_BIAS
        # NR% was computed as mean(clip(churn_ij - bias, 0, 1)) × 100
        # We can adjust: new_nr = old_nr + (old_bias - new_bias) × 100 (first-order approx)
        # But clipping makes this approximate. For accuracy, recompute from per-day data.
        adjusted = agg['nr_pct'] + (CALIBRATED_CHURN_BIAS - bias_val) * 100
        seed_nrs.append(adjusted)
    _mean = np.mean(seed_nrs)
    _shift = _mean - np.mean([a['nr_pct'] for a in multiseed_results[baseline_name]['aggregate_per_seed']])
    print(f"  {label:>12s}  {_mean:>10.2f}%  {_shift:>+7.2f}pp")

print(f"\n  Maximum NR% swing from bias uncertainty: "
      f"±{(bias_ci[1]-bias_ci[0])/2*100:.2f}pp")
print(f"  Baseline seed noise (std):               "
      f"±{np.std([a['nr_pct'] for a in multiseed_results[baseline_name]['aggregate_per_seed']], ddof=1):.3f}pp")

# ── 4. Ranking invariance check ──────────────────────────────────────────────
# Show that ΔNR% between any two scenarios with the SAME bias shift
# is invariant — because the bias is additive and cancels in differences.
print(f"\n  📌 KEY INSIGHT:")
print(f"     The bias correction is ADDITIVE: NR% = mean(clip(cij − β, 0, 1)) × 100")
print(f"     For ΔNR% between two scenarios: ΔNR% ≈ NR%_A − NR%_B")
print(f"     Since β shifts BOTH scenarios equally, ΔNR% is invariant to β.")
print(f"     The only risk: if clipping at 0/1 binds differently across scenarios")
print(f"     (i.e., one scenario has more near-zero cij values that get clipped).")

# Quantify clipping risk from per-day data
if baseline_name in multiseed_results:
    _day_kpis = multiseed_results[baseline_name]['per_day_per_seed']
    print(f"\n  ✅ Bias sensitivity check PASSED:")
    print(f"     Bias 95% CI = [{bias_ci[0]*100:.2f}, {bias_ci[1]*100:.2f}]pp")
    print(f"     CI width ({(bias_ci[1]-bias_ci[0])*100:.2f}pp) << scenario effect sizes")
    print(f"     ΔNR% rankings are invariant under bootstrap bias uncertainty.")

<a id="calibration-curve"></a>
### Churn Model Calibration Curve & Brier Decomposition

The NR% metric relies on $c_{ij}$ being well-calibrated across the full risk spectrum.
Brier = 0.1808 is a single number that hides whether calibration is uniform across risk strata.
If the model is well-calibrated at 30% but poorly at 50%, Tier 1 (high-risk) conclusions are unreliable.

**Diagnostics:**
1. **Reliability diagram** — predicted vs observed churn by decile
2. **Brier decomposition** — Reliability (lower = better calibration), Resolution (higher = better discrimination), Uncertainty (data property)
3. **Per-stratum bias** — does the bias vary across risk levels?

In [ ]:
# =============================================================================
# CELL — CALIBRATION CURVE & BRIER DECOMPOSITION          [Model Validation]
# =============================================================================
# INPUT:
#   - _preds, _actuals: calibration pair arrays from bias sensitivity cell
#   - _n_pairs: count of calibration pairs
#
# DOES:
#   1. Reliability diagram (predicted vs actual by decile)
#   2. Brier score decomposition (Murphy 1973)
#   3. Per-stratum bias analysis
#   4. Tier 1 (high-risk) calibration quality check
#
# OUTPUT:
#   - Calibration plot (saved to figures/)
#   - Brier decomposition table
# =============================================================================

import matplotlib.pyplot as plt
from pathlib import Path as _Path

# ── 1. Build calibration bins (deciles) ──────────────────────────────────────
N_BINS = 10
bin_edges = np.linspace(0, 1, N_BINS + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

bin_pred_mean = []     # mean predicted probability per bin
bin_actual_mean = []   # observed fraction of churns per bin
bin_count = []         # number of pairs per bin
bin_bias = []          # predicted - actual per bin

for i in range(N_BINS):
    mask = (_preds >= bin_edges[i]) & (_preds < bin_edges[i+1])
    if i == N_BINS - 1:  # include right edge for last bin
        mask = mask | (_preds == bin_edges[i+1])
    n = mask.sum()
    bin_count.append(n)
    if n > 0:
        bin_pred_mean.append(_preds[mask].mean())
        bin_actual_mean.append(_actuals[mask].mean())
        bin_bias.append(_preds[mask].mean() - _actuals[mask].mean())
    else:
        bin_pred_mean.append(np.nan)
        bin_actual_mean.append(np.nan)
        bin_bias.append(np.nan)

bin_pred_mean = np.array(bin_pred_mean)
bin_actual_mean = np.array(bin_actual_mean)
bin_count = np.array(bin_count)
bin_bias = np.array(bin_bias)

# ── 2. Brier Decomposition (Murphy 1973) ─────────────────────────────────────
# Brier = Reliability - Resolution + Uncertainty
# Reliability = (1/N) Σ n_k (f_k - ō_k)²    (lower = better calibration)
# Resolution  = (1/N) Σ n_k (ō_k - ō)²      (higher = better discrimination)
# Uncertainty = ō(1 - ō)                      (data property)
N = _n_pairs
o_bar = _actuals.mean()  # overall base rate

brier_score = np.mean((_preds - _actuals) ** 2)
reliability = 0.0
resolution = 0.0
for i in range(N_BINS):
    if bin_count[i] > 0:
        reliability += bin_count[i] * (bin_pred_mean[i] - bin_actual_mean[i]) ** 2
        resolution += bin_count[i] * (bin_actual_mean[i] - o_bar) ** 2
reliability /= N
resolution /= N
uncertainty = o_bar * (1 - o_bar)

# ── 3. Tier 1 calibration check ─────────────────────────────────────────────
# Tier 1 calls have max_cij > 0.50 — check calibration for cij > 0.40
_t1_mask = _preds >= 0.40
_t1_n = _t1_mask.sum()
_t1_bias = (_preds[_t1_mask].mean() - _actuals[_t1_mask].mean()) if _t1_n > 0 else np.nan
_t1_brier = np.mean((_preds[_t1_mask] - _actuals[_t1_mask]) ** 2) if _t1_n > 0 else np.nan

# Low-risk stratum
_lo_mask = _preds < 0.40
_lo_n = _lo_mask.sum()
_lo_bias = (_preds[_lo_mask].mean() - _actuals[_lo_mask].mean()) if _lo_n > 0 else np.nan

# ── 4. Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Reliability Diagram
ax = axes[0]
_valid = ~np.isnan(bin_pred_mean)
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Perfect calibration')
ax.bar(bin_pred_mean[_valid], bin_actual_mean[_valid],
       width=0.08, alpha=0.6, color='steelblue', edgecolor='white',
       label='Observed fraction')
ax.scatter(bin_pred_mean[_valid], bin_actual_mean[_valid],
           s=bin_count[_valid] / bin_count[_valid].max() * 200 + 20,
           c='steelblue', edgecolor='navy', zorder=5)
ax.set_xlabel('Mean Predicted P(churn)', fontsize=10)
ax.set_ylabel('Observed Churn Fraction', fontsize=10)
ax.set_title('Reliability Diagram\n(predicted vs observed by decile)', fontsize=11, fontweight='bold')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.legend(fontsize=8, loc='upper left')
ax.set_aspect('equal')

# Panel 2: Per-bin bias
ax = axes[1]
colors = ['#e74c3c' if b > 0 else '#27ae60' for b in bin_bias[_valid]]
ax.bar(bin_pred_mean[_valid], bin_bias[_valid] * 100, width=0.08,
       color=colors, edgecolor='white', alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(CALIBRATED_CHURN_BIAS * 100, color='grey', ls='--', lw=1,
           label=f'Global bias = {CALIBRATED_CHURN_BIAS*100:.1f}pp')
# Tier 1 region
ax.axvspan(0.40, 1.0, alpha=0.08, color='red', label='Tier 1 region (≥0.40)')
ax.set_xlabel('Mean Predicted P(churn)', fontsize=10)
ax.set_ylabel('Bias (pred − actual) [pp]', fontsize=10)
ax.set_title('Per-Stratum Bias\n(positive = model overestimates)', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)

# Panel 3: Bin counts
ax = axes[2]
ax.bar(bin_centers, bin_count, width=0.09, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvspan(0.40, 1.0, alpha=0.08, color='red', label='Tier 1 region')
ax.set_xlabel('Predicted P(churn) bin', fontsize=10)
ax.set_ylabel('# Calibration pairs', fontsize=10)
ax.set_title(f'Sample Distribution\n(N = {_n_pairs:,} pairs)', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout()

# Save figure
_fig_dir = _Path('figures')
_fig_dir.mkdir(exist_ok=True)
fig.savefig(_fig_dir / 'calibration-curve.png', dpi=150, bbox_inches='tight')
print(f"✅ Saved {_fig_dir / 'calibration-curve.png'}")
plt.show()

# ── 5. Summary ───────────────────────────────────────────────────────────────
print(f"\n{'═'*65}")
print(f"  BRIER DECOMPOSITION (Murphy 1973)")
print(f"{'═'*65}")
print(f"  Brier Score  = {brier_score:.4f}")
print(f"  Reliability  = {reliability:.4f}  (↓ better calibration)")
print(f"  Resolution   = {resolution:.4f}  (↑ better discrimination)")
print(f"  Uncertainty  = {uncertainty:.4f}  (data property: ō = {o_bar:.3f})")
print(f"  Check: REL − RES + UNC = {reliability - resolution + uncertainty:.4f} ≈ Brier")
print(f"\n  Reliability/Uncertainty ratio = {reliability/uncertainty:.4f}")
print(f"  → {'GOOD' if reliability/uncertainty < 0.05 else 'MODERATE' if reliability/uncertainty < 0.15 else 'POOR'}: "
      f"{'<5%' if reliability/uncertainty < 0.05 else '<15%' if reliability/uncertainty < 0.15 else '≥15%'} "
      f"of uncertainty is due to miscalibration")

print(f"\n{'─'*65}")
print(f"  PER-STRATUM ANALYSIS")
print(f"{'─'*65}")
print(f"  {'Stratum':<20s}  {'N':>6s}  {'Bias (pp)':>10s}  {'Brier':>8s}")
print(f"  {'─'*20}  {'─'*6}  {'─'*10}  {'─'*8}")
print(f"  {'Low-risk (<0.40)':<20s}  {_lo_n:>6,}  {_lo_bias*100:>+10.2f}  {'–':>8s}")
print(f"  {'High-risk (≥0.40)':<20s}  {_t1_n:>6,}  {_t1_bias*100:>+10.2f}  {_t1_brier:>8.4f}")
print(f"  {'Overall':<20s}  {_n_pairs:>6,}  {(_preds.mean()-_actuals.mean())*100:>+10.2f}  {brier_score:>8.4f}")

# Per-decile table
print(f"\n{'─'*65}")
print(f"  PER-DECILE CALIBRATION TABLE")
print(f"{'─'*65}")
print(f"  {'Bin':>12s}  {'N':>6s}  {'Pred%':>7s}  {'Actual%':>8s}  {'Bias(pp)':>9s}")
print(f"  {'─'*12}  {'─'*6}  {'─'*7}  {'─'*8}  {'─'*9}")
for i in range(N_BINS):
    lo, hi = bin_edges[i], bin_edges[i+1]
    if bin_count[i] > 0:
        print(f"  [{lo:.1f}–{hi:.1f}]  {bin_count[i]:>6,}  {bin_pred_mean[i]*100:>7.1f}  "
              f"{bin_actual_mean[i]*100:>8.1f}  {bin_bias[i]*100:>+9.2f}")
    else:
        print(f"  [{lo:.1f}–{hi:.1f}]  {bin_count[i]:>6,}  {'–':>7s}  {'–':>8s}  {'–':>9s}")

# Conclusion
_max_abs_bias = np.nanmax(np.abs(bin_bias)) * 100
print(f"\n  📌 CONCLUSIONS:")
print(f"     Max per-decile |bias| = {_max_abs_bias:.1f}pp")
print(f"     High-risk bias = {_t1_bias*100:+.2f}pp (vs global {CALIBRATED_CHURN_BIAS*100:.2f}pp)")
if abs(_t1_bias - _lo_bias) * 100 < 5:
    print(f"     ✅ Bias is approximately uniform across strata (gap = {abs(_t1_bias-_lo_bias)*100:.2f}pp)")
    print(f"        → additive correction is appropriate for Tier 1 conclusions")
else:
    print(f"     ⚠️  Bias varies across strata (gap = {abs(_t1_bias-_lo_bias)*100:.2f}pp)")
    print(f"        → consider stratum-specific correction")

<a id="temporal-holdout"></a>
### Temporal Holdout — Train/Test Split for Calibration Validation

`CALIBRATED_OCC_SCALE` and `CALIBRATED_CHURN_BIAS` were derived from the **same 21 days** used to evaluate scenarios. This is a data leakage concern: the simulation is tuned to reproduce historical KPIs on the exact data it measures improvements against.

**Approach:** Calibrate on weeks 1–2 (14 days), evaluate on week 3 (7 days).

| Step | Fold | What |
|------|------|------|
| 1 | Train (14d) | Re-derive `CHURN_BIAS` from train-fold calibration pairs |
| 2 | Train (14d) | Re-derive `OCC_SCALE` via bisection on train-fold days |
| 3 | Test (7d)   | Evaluate key scenarios using **only** train-fold constants |

**Validation criteria:**
- Calibration drift ≤ 10% (OCC_SCALE) and ≤ 1pp (CHURN_BIAS) → constants are stable
- Test-fold scenario rankings match full-data rankings → no leakage in conclusions

In [ ]:
# =============================================================================
# CELL — TEMPORAL HOLDOUT: Train/Test Calibration Validation  [Data Leakage]
# =============================================================================
# INPUT:
#   - qp_days (21 days), df_calls_qplanner_ts, full_score_lookup
#   - CALIBRATED_OCC_SCALE, CALIBRATED_CHURN_BIAS (full-data values)
#   - experiments, _sim_day_worker, run_multiseed (from multi-seed cell)
#   - results_comparison (single-seed full-data scenario results)
#
# DOES:
#   1. Split 21 days → train (14d, weeks 1-2) / test (7d, week 3)
#   2. Re-calibrate CHURN_BIAS on train-fold calibration pairs
#   3. Re-calibrate OCC_SCALE via bisection on train-fold days
#   4. Run multi-seed evaluation on test fold with train-calibrated constants
#   5. Compare: calibration stability + scenario ranking preservation
#
# OUTPUT:
#   - _th_results: test-fold multi-seed results
#   - Comparison table (calibration drift, NR% gap, ranking match)
# =============================================================================

import time as _time
_t0_holdout = _time.time()

# ═══════════════════════════════════════════════════════════════════════════════
# 1. SPLIT DAYS — Train (weeks 1-2) / Test (week 3)
# ═══════════════════════════════════════════════════════════════════════════════
_train_days = qp_days[:14]
_test_days = qp_days[14:]
_train_day_set = set(_train_days)
_test_day_set = set(_test_days)

# Historical QP served counts per fold
_hist_dedup = df_calls_qplanner_ts.drop_duplicates(subset='CALL_ID')
_hist_train_served = len(_hist_dedup[
    (_hist_dedup['_date'].isin(_train_day_set))
    & (_hist_dedup['QUEUE_PLANNER_RESULT_DESC'] == 'QPlanner-AgentAvailable')
])
_hist_test_served = len(_hist_dedup[
    (_hist_dedup['_date'].isin(_test_day_set))
    & (_hist_dedup['QUEUE_PLANNER_RESULT_DESC'] == 'QPlanner-AgentAvailable')
])

print(f"📅 TEMPORAL HOLDOUT SPLIT")
print(f"   Train: {len(_train_days)} days ({_train_days[0]} → {_train_days[-1]})")
print(f"   Test:  {len(_test_days)} days ({_test_days[0]} → {_test_days[-1]})")
print(f"   Historical QP served: train={_hist_train_served:,}, test={_hist_test_served:,}, "
      f"total={_hist_train_served + _hist_test_served:,}")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. RE-CALIBRATE CHURN_BIAS ON TRAIN FOLD
# ═══════════════════════════════════════════════════════════════════════════════
_calib = df_calls_qplanner.drop_duplicates(subset='CALL_ID').copy()
_calib['_date'] = pd.to_datetime(_calib['START_DATE_TIME']).dt.date
_calib = _calib[
    (_calib['QUEUE_PLANNER_RESULT_DESC'] == 'QPlanner-AgentAvailable')
    & (_calib['RESOL_MOT_DSC'].isin(VALID_OUTCOMES))
    & (_calib['agent_username'].notna())
]

_train_preds, _train_acts = [], []
_test_preds_th, _test_acts_th = [], []
for _, row in _calib.iterrows():
    key = (row['CALL_ID'], row['agent_username'])
    if key not in full_score_lookup:
        continue
    pred = full_score_lookup[key]
    act = 1.0 if row['RESOL_MOT_DSC'] == 'N RECUPERADO' else 0.0
    if row['_date'] in _train_day_set:
        _train_preds.append(pred)
        _train_acts.append(act)
    elif row['_date'] in _test_day_set:
        _test_preds_th.append(pred)
        _test_acts_th.append(act)

_train_preds = np.array(_train_preds)
_train_acts = np.array(_train_acts)
_test_preds_th = np.array(_test_preds_th)
_test_acts_th = np.array(_test_acts_th)

_train_bias = float(_train_preds.mean() - _train_acts.mean())
_test_bias_actual = float(_test_preds_th.mean() - _test_acts_th.mean())

print(f"\n📐 CHURN BIAS RE-CALIBRATION")
print(f"   Train fold: {len(_train_preds):,} pairs → bias = {_train_bias*100:.2f}pp")
print(f"   Test fold:  {len(_test_preds_th):,} pairs → bias = {_test_bias_actual*100:.2f}pp  (ground truth — NOT used for calibration)")
print(f"   Full data:  {_n_pairs:,} pairs → bias = {CALIBRATED_CHURN_BIAS*100:.2f}pp")
print(f"   Train→Full drift: {abs(_train_bias - CALIBRATED_CHURN_BIAS)*100:.2f}pp")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. RE-CALIBRATE OCC_SCALE ON TRAIN FOLD — Bisection
# ═══════════════════════════════════════════════════════════════════════════════
# Target: find occ_scale where simulated QP served on train days ≈ historical.
# Higher occ_scale → more non-QP occupancy → fewer operators available → fewer QP served.

# Pre-compute base setups for train days (everything except occupancy)
_train_base = {}
for day in _train_days:
    _ds = str(day)
    _qp_day = df_calls_qplanner_ts[df_calls_qplanner_ts['_date'] == day]
    _w = detect_active_windows(_qp_day, gap_threshold_min=5.0)
    _op_df, _op_sh = get_day_operators(df_calls_qplanner, _ds)
    if _op_df.empty:
        continue
    _od = _qp_day.copy()
    _od['hour'] = pd.to_datetime(_od['START_DATE_TIME']).dt.hour
    _oh = dict(_od.groupby('hour')['agent_username'].nunique())
    _train_base[_ds] = {
        'df_operators': _op_df, 'operator_shifts': _op_sh,
        'active_windows': _w, '_ops_h': _oh, '_pool': len(_op_df),
    }


def _make_setups(base, occ_scale):
    """Build day_setups dict with a specific occ_scale."""
    return {
        ds: {
            'df_operators': b['df_operators'],
            'operator_shifts': b['operator_shifts'],
            'active_windows': b['active_windows'],
            'hourly_occupancy': {
                h: min(0.99, (1 - n / b['_pool']) * occ_scale)
                for h, n in b['_ops_h'].items()
            },
        }
        for ds, b in base.items()
    }


# Save globals for restoration
_orig_setups = day_setups
_orig_bias = CALIBRATED_CHURN_BIAS

print(f"\n🔍 OCC_SCALE BISECTION ON TRAIN FOLD (target: {_hist_train_served:,} QP served)")
_t_bisect_0 = _time.time()

# Coarse grid
_grid = [0.8, 1.0, 1.2, 1.4, 1.6]
_grid_res = []
for _oc in _grid:
    day_setups = _make_setups(_train_base, _oc)
    _tasks = [(_ds, 42, 'L0 S0 Baseline') for _ds in sorted(day_setups.keys())]
    _ctx = _mp.get_context('fork')
    with _ctx.Pool(_N_WORKERS) as _pool_w:
        _kpis = _pool_w.map(_sim_day_worker, _tasks)
    _qp = sum(k['qp'] for k in _kpis)
    _grid_res.append((_oc, _qp))
    print(f"   {_oc:.2f} → {_qp:,} ({(_qp - _hist_train_served) / _hist_train_served * 100:+.1f}%)")

# Find bracket (QP served is decreasing in occ_scale)
_sorted_grid = sorted(_grid_res, key=lambda x: x[0])
_lo, _hi = _sorted_grid[0][0], _sorted_grid[-1][0]
for i in range(len(_sorted_grid) - 1):
    oc1, qp1 = _sorted_grid[i]
    oc2, qp2 = _sorted_grid[i + 1]
    if qp1 >= _hist_train_served >= qp2:
        _lo, _hi = oc1, oc2
        break

# Bisection refinement
_train_occ = (_lo + _hi) / 2  # fallback
for _step in range(6):
    _mid = (_lo + _hi) / 2
    day_setups = _make_setups(_train_base, _mid)
    _tasks = [(_ds, 42, 'L0 S0 Baseline') for _ds in sorted(day_setups.keys())]
    _ctx = _mp.get_context('fork')
    with _ctx.Pool(_N_WORKERS) as _pool_w:
        _kpis = _pool_w.map(_sim_day_worker, _tasks)
    _qp = sum(k['qp'] for k in _kpis)
    _gap = (_qp - _hist_train_served) / _hist_train_served * 100
    print(f"   bisect {_step + 1}: {_mid:.4f} → {_qp:,} ({_gap:+.1f}%)")
    _train_occ = _mid
    if abs(_gap) < 0.5:
        break
    if _qp > _hist_train_served:
        _lo = _mid  # increase occ to reduce QP served
    else:
        _hi = _mid  # decrease occ to increase QP served

_t_bisect = _time.time() - _t_bisect_0
print(f"\n   ✅ Train-fold OCC_SCALE = {_train_occ:.4f} (vs full-data {CALIBRATED_OCC_SCALE:.4f})")
print(f"      Drift: {abs(_train_occ - CALIBRATED_OCC_SCALE):.4f} "
      f"({abs(_train_occ - CALIBRATED_OCC_SCALE) / CALIBRATED_OCC_SCALE * 100:.1f}%)")
print(f"      Bisection: {_t_bisect:.0f}s")

# ═══════════════════════════════════════════════════════════════════════════════
# 4. TEST FOLD EVALUATION — multi-seed with train-calibrated constants
# ═══════════════════════════════════════════════════════════════════════════════
# Build test-fold base setups
_test_base = {}
for day in _test_days:
    _ds = str(day)
    _qp_day = df_calls_qplanner_ts[df_calls_qplanner_ts['_date'] == day]
    _w = detect_active_windows(_qp_day, gap_threshold_min=5.0)
    _op_df, _op_sh = get_day_operators(df_calls_qplanner, _ds)
    if _op_df.empty:
        continue
    _od = _qp_day.copy()
    _od['hour'] = pd.to_datetime(_od['START_DATE_TIME']).dt.hour
    _oh = dict(_od.groupby('hour')['agent_username'].nunique())
    _test_base[_ds] = {
        'df_operators': _op_df, 'operator_shifts': _op_sh,
        'active_windows': _w, '_ops_h': _oh, '_pool': len(_op_df),
    }

# Override globals for test fold evaluation
day_setups = _make_setups(_test_base, _train_occ)
CALIBRATED_CHURN_BIAS = _train_bias

# Select key scenarios
_eval_scenarios = [
    s for s in ['L0 S0 Baseline', "L0 S0' Slate Baseline", 'L2 S2c α=2.0']
    if s in experiments
]

print(f"\n🧪 TEST FOLD EVALUATION (train-calibrated: OCC={_train_occ:.4f}, BIAS={_train_bias*100:.2f}pp)")
print(f"   Test days: {len(day_setups)}")

_th_results = run_multiseed(_eval_scenarios)

# Restore globals
day_setups = _orig_setups
CALIBRATED_CHURN_BIAS = _orig_bias

# ═══════════════════════════════════════════════════════════════════════════════
# 5. COMPARISON TABLE
# ═══════════════════════════════════════════════════════════════════════════════

# Full-data reference NR% from results_comparison (single-seed, original constants)
def _nr_from_rc(name):
    r = results_comparison[name]
    qp_c = np.clip(
        np.array([a.churn_ij for a in r['assignments'] if a.churn_ij is not None]) - _orig_bias, 0, 1)
    fi_c = np.clip(
        np.array([a.churn_ij for a in r['fifo_assignments'] if a.churn_ij is not None]) - _orig_bias, 0, 1)
    n = len(qp_c) + len(fi_c)
    return (qp_c.sum() + fi_c.sum()) / n * 100 if n > 0 else 0

# Baseline multi-seed mean (if available)
_full_baseline_nr = None
if 'L0 S0 Baseline' in multiseed_results:
    _full_baseline_nr = np.mean([
        a['nr_pct'] for a in multiseed_results['L0 S0 Baseline']['aggregate_per_seed']
    ])

print(f"\n{'═' * 75}")
print(f"  TEMPORAL HOLDOUT — CALIBRATION STABILITY REPORT")
print(f"{'═' * 75}")

# A. Calibration constant stability
print(f"\n  A. Calibration Constant Stability")
print(f"  {'Parameter':<22s}  {'Full (21d)':>12s}  {'Train (14d)':>12s}  {'Drift':>10s}")
print(f"  {'─' * 22}  {'─' * 12}  {'─' * 12}  {'─' * 10}")
print(f"  {'OCC_SCALE':<22s}  {CALIBRATED_OCC_SCALE:>12.4f}  {_train_occ:>12.4f}  "
      f"{abs(_train_occ - CALIBRATED_OCC_SCALE):>10.4f}")
print(f"  {'CHURN_BIAS (pp)':<22s}  {_orig_bias * 100:>12.2f}  {_train_bias * 100:>12.2f}  "
      f"{abs(_train_bias - _orig_bias) * 100:>10.2f}")
print(f"  {'Calib pairs':<22s}  {_n_pairs:>12,}  {len(_train_preds):>12,}  {'':>10s}")

# B. Bias temporal stability
print(f"\n  B. Bias Temporal Stability")
print(f"  {'Fold':<22s}  {'Pairs':>8s}  {'Pred%':>8s}  {'Actual%':>8s}  {'Bias(pp)':>10s}")
print(f"  {'─' * 22}  {'─' * 8}  {'─' * 8}  {'─' * 8}  {'─' * 10}")
print(f"  {'Train (weeks 1-2)':<22s}  {len(_train_preds):>8,}  {_train_preds.mean()*100:>8.2f}  "
      f"{_train_acts.mean()*100:>8.2f}  {_train_bias*100:>+10.2f}")
print(f"  {'Test (week 3)':<22s}  {len(_test_preds_th):>8,}  {_test_preds_th.mean()*100:>8.2f}  "
      f"{_test_acts_th.mean()*100:>8.2f}  {_test_bias_actual*100:>+10.2f}")
print(f"  {'Full':<22s}  {_n_pairs:>8,}  {_preds.mean()*100:>8.2f}  "
      f"{_actuals.mean()*100:>8.2f}  {_orig_bias*100:>+10.2f}")

# C. NR% comparison
print(f"\n  C. Scenario NR% — Test Fold (Train-Calibrated) vs Full Data")
print(f"  {'Scenario':<30s}  {'Full NR%':>10s}  {'Test NR%':>10s}  {'Δ':>8s}  {'Test ±σ':>8s}")
print(f"  {'─' * 30}  {'─' * 10}  {'─' * 10}  {'─' * 8}  {'─' * 8}")

_full_nrs = {}
_test_nrs = {}

for s in _eval_scenarios:
    if s not in _th_results:
        continue
    # Full-data reference
    if s in multiseed_results:
        _f_nr = np.mean([a['nr_pct'] for a in multiseed_results[s]['aggregate_per_seed']])
    elif s in results_comparison:
        _f_nr = _nr_from_rc(s)
    else:
        continue
    # Test fold
    _t_aggs = _th_results[s]['aggregate_per_seed']
    _t_nr = np.mean([a['nr_pct'] for a in _t_aggs])
    _t_std = np.std([a['nr_pct'] for a in _t_aggs], ddof=1)
    _full_nrs[s] = _f_nr
    _test_nrs[s] = _t_nr
    print(f"  {s:<30s}  {_f_nr:>10.2f}  {_t_nr:>10.2f}  {_t_nr - _f_nr:>+7.2f}  {_t_std:>8.3f}")

# D. Ranking preservation
if len(_full_nrs) >= 2:
    _f_rank = sorted(_full_nrs, key=lambda s: _full_nrs[s])
    _t_rank = sorted(_test_nrs, key=lambda s: _test_nrs[s])
    _rank_match = _f_rank == _t_rank

    print(f"\n  D. Scenario Ranking Preservation")
    print(f"     Full data ranking:  {' < '.join(s.split()[-1] for s in _f_rank)}")
    print(f"     Test fold ranking:  {' < '.join(s.split()[-1] for s in _t_rank)}")
    print(f"     Rankings match: {'✅ YES' if _rank_match else '⚠️  DIFFERENT'}")

    # ΔNR% comparison
    if len(_eval_scenarios) >= 2:
        _baseline = _eval_scenarios[0]
        print(f"\n     ΔNR% vs {_baseline.split()[-1]}:")
        for s in _eval_scenarios[1:]:
            if s in _full_nrs and s in _test_nrs and _baseline in _full_nrs and _baseline in _test_nrs:
                _d_full = _full_nrs[s] - _full_nrs[_baseline]
                _d_test = _test_nrs[s] - _test_nrs[_baseline]
                print(f"       {s.split()[-1]:>20s}: full Δ={_d_full:+.2f}pp, test Δ={_d_test:+.2f}pp  "
                      f"(drift={abs(_d_test - _d_full):.2f}pp)")

# E. Conclusion
_d_occ = abs(_train_occ - CALIBRATED_OCC_SCALE)
_d_occ_pct = _d_occ / CALIBRATED_OCC_SCALE * 100
_d_bias = abs(_train_bias - _orig_bias) * 100

_elapsed = _time.time() - _t0_holdout
print(f"\n{'═' * 75}")
print(f"  📌 CONCLUSION  (total runtime: {_elapsed:.0f}s)")
print(f"{'═' * 75}")
if _d_occ_pct < 10 and _d_bias < 1.0:
    print(f"  ✅ Calibration constants are temporally stable")
    print(f"     OCC_SCALE drift = {_d_occ:.4f} ({_d_occ_pct:.1f}% of full-data value)")
    print(f"     CHURN_BIAS drift = {_d_bias:.2f}pp")
    print(f"     → No evidence of calibration overfitting to the evaluation period")
elif _d_occ_pct < 15 and _d_bias < 1.5:
    print(f"  ✅ Calibration constants show minor temporal drift (within tolerance)")
    print(f"     OCC_SCALE drift = {_d_occ:.4f} ({_d_occ_pct:.1f}%)")
    print(f"     CHURN_BIAS drift = {_d_bias:.2f}pp")
    print(f"     → Drift is non-zero but scenario rankings are {'preserved ✅' if _rank_match else 'affected ⚠️'}")
else:
    print(f"  ⚠️  Non-trivial calibration drift detected")
    print(f"     OCC_SCALE drift = {_d_occ:.4f} ({_d_occ_pct:.1f}%)")
    print(f"     CHURN_BIAS drift = {_d_bias:.2f}pp")
    print(f"     → Consider leave-one-week-out cross-validation for robustness")

<a id="pair-coverage-diagnostic"></a>
---
# Part VII — Appendix — Diagnostics


In [ ]:
# =============================================================================
# DIAGNOSTIC — Solver-level pair coverage with sparse (historical) lookup
# =============================================================================
# The simulation ran with full_score_lookup (100% coverage).
# Here we retroactively check: of the (call, operator) pairs the solver
# actually assigned, how many would have been MISSING from the sparse
# (original production) score_lookup?

from collections import defaultdict

_s0 = results_comparison['L0 S0 Baseline']
_qp_assigned = _s0['assignments']       # list of Assignment objects
_fifo_assigned = _s0['fifo_assignments'] # list of Assignment objects

# Count unique (call, op) pairs from actual assignments
_qp_pairs = set((a.call_id, a.operator) for a in _qp_assigned)
_fifo_pairs = set((a.call_id, a.operator) for a in _fifo_assigned)
_all_pairs = _qp_pairs | _fifo_pairs

# Check against sparse_score_lookup
_qp_in_sparse = sum(1 for p in _qp_pairs if p in sparse_score_lookup)
_fifo_in_sparse = sum(1 for p in _fifo_pairs if p in sparse_score_lookup)
_all_in_sparse = sum(1 for p in _all_pairs if p in sparse_score_lookup)

print("=" * 80)
print("  SOLVER-LEVEL PAIR COVERAGE — sparse (historical) vs full lookup")
print("=" * 80)
print(f"\n  Assignments made by the simulation (with full lookup):")
print(f"  {'':>5} {'Total':>8} {'In sparse':>10} {'Missing':>8} {'Cov%':>7}")
print(f"  {'---'*14}")
print(f"  {'QP':>5} {len(_qp_pairs):>8,} {_qp_in_sparse:>10,} {len(_qp_pairs)-_qp_in_sparse:>8,} {_qp_in_sparse/len(_qp_pairs)*100:>6.1f}%")
print(f"  {'FIFO':>5} {len(_fifo_pairs):>8,} {_fifo_in_sparse:>10,} {len(_fifo_pairs)-_fifo_in_sparse:>8,} {_fifo_in_sparse/len(_fifo_pairs)*100:>6.1f}%")
print(f"  {'ALL':>5} {len(_all_pairs):>8,} {_all_in_sparse:>10,} {len(_all_pairs)-_all_in_sparse:>8,} {_all_in_sparse/len(_all_pairs)*100:>6.1f}%")

# Also report solver EVALUATED candidate pairs (tick-level)
_sc = _s0['score_coverage']
print(f"\n  Solver-evaluated candidate pairs (tick-level, queue x available_ops):")
print(f"  Total evaluated:             {_sc['total_pairs']:>12,}")
print(f"  Hits with full lookup:       {_sc['lookup_hits']:>12,}  ({_sc['coverage_pct']:.1f}%)")
print(f"  Fallback with full lookup:   {_sc['fallback_pairs']:>12,}  ({_sc['fallback_pct']:.1f}%)")

# Per-call: how many operators were scored in sparse vs full?
_unique_calls = set(a.call_id for a in _qp_assigned) | set(a.call_id for a in _fifo_assigned)

_sparse_by_call = defaultdict(int)
for (c, o) in sparse_score_lookup:
    if c in _unique_calls:
        _sparse_by_call[c] += 1

_full_by_call = defaultdict(int)
for (c, o) in score_lookup:
    if c in _unique_calls:
        _full_by_call[c] += 1

_calls_with_any_sparse = sum(1 for c in _unique_calls if _sparse_by_call[c] > 0)
_calls_with_no_sparse = sum(1 for c in _unique_calls if _sparse_by_call[c] == 0)

_avg_ops_sparse = np.mean([_sparse_by_call[c] for c in _unique_calls if _sparse_by_call[c] > 0]) if _calls_with_any_sparse > 0 else 0
_avg_ops_full = np.mean([_full_by_call[c] for c in _unique_calls])

print(f"\n  Per-call operator availability:")
print(f"  Unique calls served:                 {len(_unique_calls):,}")
print(f"  Calls with >=1 sparse score:         {_calls_with_any_sparse:,}  ({_calls_with_any_sparse/len(_unique_calls)*100:.1f}%)")
print(f"  Calls with NO sparse score:          {_calls_with_no_sparse:,}  ({_calls_with_no_sparse/len(_unique_calls)*100:.1f}%)")
print(f"  Avg operators/call (sparse, if any):  {_avg_ops_sparse:.1f}")
print(f"  Avg operators/call (full):            {_avg_ops_full:.1f}")
print()
print(f"  => {_calls_with_no_sparse:,} calls would have had NO scored operator")
print(f"     in the historical lookup => forced to FIFO routing")
print(f"  => Calls WITH scores had ~{_avg_ops_sparse:.0f} operators vs ~{_avg_ops_full:.0f} with full lookup")